# Poseidon 2 — 1D Mechanical Earth Model
## Increment 2.1: LAS Ingestion and Per-File Curve-Contract Resolution (corrected)

**Author:** Mikael Elgo
**Project classification:** Tier C — Screening-Level / Uncalibrated Educational 1D MEM
**Foundation:** Increment 1.1 (project skeleton + unit-control system, `p2mem/units.py`) has passed independent technical review and is treated as LOCKED — it is not rewritten in this notebook unless an actual blocking defect is demonstrated (none was).

**Increment scope:** build a reliable, auditable LAS-ingestion layer for the four approved wells — Poseidon 2 (primary), Boreas 1, Poseidon North 1, Proteus 1ST2 (offset wells) — with an explicit, per-file curve-identity contract for every curve in every file. This increment ends at verified LAS ingestion, deterministic curve-identity resolution, and factual inventory generation.

**Explicitly NOT implemented in this increment:** deviation-survey calculations, MD-to-TVD/TVDSS transformation, checkshot processing, formation-top correction, petrophysical interpretation, shale-volume calculation, density estimation, NCT fitting, pore-pressure prediction, elastic properties, rock strength, stresses, wellbore stability, or any plot intended as geological interpretation.

### Why this increment exists: the empty-mnemonic problem

All four approved LAS files declare an EMPTY mnemonic field for every curve except the depth curve. Instead, the curve section carries only a unit and a free-text description such as `: 1 DCAV` — an embedded ordinal number plus the intended curve name. A naive parser would either invent placeholder names (`UNKNOWN`, `UNKNOWN:1`) or trust column position alone — both risk silently mislabeling a curve. Worse, **column order is not consistent across the four files**: three of them place their neutron-porosity curve last, but Proteus 1ST2 places it fifth, shifting three other curves one position earlier. A single shared, order-based assumption would silently mislabel Proteus 1ST2 end-to-end.

This increment's answer is an explicit, per-file, human-reviewable **curve contract** (`config/las_curve_contracts.yml`) built by directly inspecting each file's actual header text — never guessed, never copied from another well's contract — and a resolver that requires several independent identifiers (ordinal position, unit, description-embedded ordinal, description-embedded name) to agree before trusting a curve's identity. A contract failure stops ingestion for that file; it never causes a silent fallback to a different column.

### What changed in the Increment 2.1 correction

An independent technical audit of the first Increment 2 delivery found the parsing/resolution architecture above to be sound (it loaded all four real wells correctly and passed 131 tests), but identified real gaps, corrected here without touching that working architecture:

1. **Explicit, unit-suffixed naming.** Increment 2 stored converted compressional/shear velocity under the keys `"DTCO"`/`"DTSM"` — mnemonics that conventionally mean *slowness* (µs/ft), not velocity (m/s). Every canonical array is now named explicitly with its unit (`VP_m_s`, `VS_m_s`, `RHOB_kg_m3`, `GR_api`, `DCAV_in`, ...), and the raw, pre-conversion identity is separately documented (`DTCO_us_per_ft`, `RHOB_g_per_cm3`, ...). `GR_api`, `ECGR_api`, and `GRD_api` remain three distinct canonical names — never merged into one geological interpretation.
2. **Explicit measured-depth role.** The depth curve is now located via a contract field (`semantic_role: measured_depth`), validated to be present exactly once, never inferred from a canonical name spelled `"DEPT"` — its canonical name is now `"MD_m"`.
3. **Blocking file-identity checks.** An independent audit demonstrated the Increment 2 resolver *passed* a file with a missing `WELL`, a missing `NULL`, a changed `VERS`, a changed `WRAP`, or even a contract for a different file than the one being loaded. All of these are now blocking `ERROR`s, checked before any curve is resolved. A `NULL` mismatch is now an `ERROR`, not a `WARNING`.
4. **Unambiguous curve-coverage statistics.** `las_curve_coverage.csv` now reports an explicit RAW/CANONICAL pair for every curve, each tagged with its own unit, instead of a single min/max pair labeled only with a canonical (implicitly-converted) name.
5. **A controlled conversion-signature registry.** A contract can no longer pair an incompatible unit with a conversion function (e.g. applying a density conversion to an API gamma-ray curve) — this is validated at contract-load time.
6. **Typed batch failures.** `load_wells` now isolates four distinct, expected per-file failure categories (file-not-found, structural-parsing, contract, conversion) as typed `IngestionFailure` records, rather than a single bare caught exception.

See `INCREMENT_02_v2.1_MANIFEST.md` for the full audit and every corrected file's SHA-256.

### Step 1 — Mount Google Drive

**Technical objective:** re-attach the persistent project folder created in Increment 1.1, so this notebook builds on the same files rather than a fresh, empty VM disk.

**Inputs/outputs:** no project inputs are read here; this only establishes the `/content/drive` mount point Google Colab uses to expose the user's Drive filesystem.

**Failure behavior:** if the user declines the authorization prompt, all subsequent cells that read/write under `/content/drive/MyDrive/...` will fail with a clear "no such file or directory" error — there is no silent fallback to local VM storage.

**Expected result:** `/content/drive/MyDrive` becomes readable/writable from this notebook.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### Step 2 — Verify the Increment 1.1 foundation is present and locked

**Technical objective:** confirm that `p2mem/units.py`, `pyproject.toml`, and `tests/test_units.py` from Increment 1.1 already exist in this Drive project folder, and that they still import and pass, BEFORE this notebook adds anything on top. Increment 2 is explicitly instructed to treat Increment 1.1 as a locked foundation, not to rewrite `p2mem/units.py` unless an actual blocking defect is demonstrated — none was found or is claimed here.

**Why this matters:** silently proceeding on a missing or broken foundation would make every later step's provenance claims meaningless. This is a hard gate: if the Increment 1.1 files are not found, this cell stops with an explicit, actionable message rather than attempting to reconstruct them from memory (which would risk silently diverging from the reviewed, locked version).

**Inputs:** the existing `/content/drive/MyDrive/Poseidon_1D_MEM/` folder from Increment 1.1.
**Outputs:** a pass/fail confirmation printed to the cell output; no files are written by this cell.

**Failure behavior:** raises `RuntimeError` naming exactly which expected file is missing, and instructs the user to run the Increment 1.1 notebook first.

**Expected result:** `p2mem/units.py` and `pyproject.toml` are found, and (if a prior venv/kernel already has `p2mem` importable) `p2mem.__version__` reports the Increment 1.1 version.

In [ ]:
import os
from pathlib import Path

PROJECT_ROOT = "/content/drive/MyDrive/Poseidon_1D_MEM"

_required_increment_1_1_files = [
    "pyproject.toml",
    os.path.join("p2mem", "__init__.py"),
    os.path.join("p2mem", "units.py"),
    os.path.join("tests", "test_units.py"),
]
_missing = [f for f in _required_increment_1_1_files if not os.path.exists(os.path.join(PROJECT_ROOT, f))]
if _missing:
    raise RuntimeError(
        "Increment 1.1 foundation not found in this Drive project folder. "
        f"Missing: {_missing}. Run the Increment 1.1 notebook "
        "(01_Project_Foundation_and_Units.ipynb) before this one. Stopping cleanly."
    )

print("Increment 1.1 foundation files found:")
for f in _required_increment_1_1_files:
    print(" -", f)
print("\np2mem/units.py will NOT be rewritten by this notebook (locked foundation, unchanged since Increment 1.1).")

### Step 3 — Create the Increment 2 directory additions

**Technical objective:** extend the project layout with the directories Increment 2 needs, without disturbing anything Increment 1.1 already created.

**Directory contract for this increment:**
```
/content/drive/MyDrive/Poseidon_1D_MEM/
├── data/
│   └── raw/
│       └── logs/          <- the four raw LAS files go here (not created by this notebook)
├── config/                <- already existed; will hold las_curve_contracts.yml
├── p2mem/                 <- already existed; gains models.py and io/
├── tests/                 <- already existed; gains fixtures/ and test_las.py
├── notebooks/             <- already existed
└── outputs/
    └── 02_las_inventory/  <- generated inventory CSV/JSON land here
```

**Inputs/outputs:** no files are written in this cell, only directories (created with `exist_ok=True`, so re-running this notebook is safe and does not disturb existing content).

**Validation logic:** the cell prints the resulting tree so the structure can be visually confirmed.

**Expected result:** `data/raw/logs/`, `outputs/02_las_inventory/`, and `tests/fixtures/` exist (the last two package directories, `p2mem/io/` and its contents, are created implicitly when their files are written in Step 6).

In [ ]:
for d in ["data/raw/logs", "config", "p2mem/io", "tests/fixtures", "notebooks", "outputs/02_las_inventory"]:
    os.makedirs(os.path.join(PROJECT_ROOT, d), exist_ok=True)

os.chdir(PROJECT_ROOT)
print(f"Project root: {PROJECT_ROOT}\n")
for root, dirs, files in os.walk(PROJECT_ROOT):
    dirs[:] = [d for d in dirs if d not in (".git", "__pycache__", ".pytest_cache", "p2mem.egg-info")]
    depth = root.replace(PROJECT_ROOT, "").count(os.sep)
    indent = "  " * depth
    print(f"{indent}{os.path.basename(root) or '.'}/")
    for f in sorted(files):
        print(f"{indent}  {f}")

### Step 4 — Verify the four raw LAS filenames are present (exact names only)

**Technical objective:** check for the four exact, approved filenames under `data/raw/logs/` — nothing more permissive. Per the increment specification, this notebook does NOT search other Drive folders automatically, does NOT accept renamed files, and does NOT substitute a similarly-named file if the exact name is absent.

**Why exact-name matching, not a folder scan:** these four files are the only approved inputs for this project (Poseidon 2, Boreas 1, Poseidon North 1, Proteus 1ST2). A folder scan that picked up "close enough" filenames could silently ingest an unapproved or stale file — exactly the kind of silent substitution this project's integrity rules forbid.

**Inputs:** `data/raw/logs/` on Drive.
**Outputs:** none (read-only check).

**Failure behavior:** if any of the four exact filenames is missing, this cell prints every missing filename explicitly and raises `RuntimeError` — it stops cleanly rather than attempting to continue with a partial well set.

**Expected result:** all four filenames are found. (The four raw files themselves are treated as immutable inputs throughout this notebook: never overwritten, never rewritten, never included in any deliverable ZIP — only their SHA-256 checksums are recorded.)

In [ ]:
LOGS_DIR = os.path.join(PROJECT_ROOT, "data", "raw", "logs")

REQUIRED_LAS_FILES = [
    "Poseidon_2_logs.las",
    "Boreas_1_logs.las",
    "Poseidon_North_1_logs.las",
    "Proteus_1ST2_logs.las",
]

_missing_las = [f for f in REQUIRED_LAS_FILES if not os.path.exists(os.path.join(LOGS_DIR, f))]
if _missing_las:
    print("MISSING required LAS file(s):")
    for f in _missing_las:
        print(" -", f)
    raise RuntimeError(
        f"{len(_missing_las)} of {len(REQUIRED_LAS_FILES)} required LAS file(s) missing from {LOGS_DIR}. "
        "Upload the exact files listed above (exact filenames, no renaming) and re-run this cell. "
        "Stopping cleanly - no other Drive folder was searched."
    )

print("All four required LAS files found in:", LOGS_DIR)
for f in REQUIRED_LAS_FILES:
    size = os.path.getsize(os.path.join(LOGS_DIR, f))
    print(f" - {f} ({size:,} bytes)")

### Step 5 — Install dependencies

**Technical objective:** install the runtime and test dependencies this increment adds. `numpy` and `pytest` were already required by Increment 1.1; `pyyaml` is new in Increment 2, added for exactly one purpose — parsing the human-authored, human-reviewable `config/las_curve_contracts.yml` file. `pandas` is installed too, but ONLY for convenient tabular display of inventory results later in this notebook (Step 15) — it is never imported by the `p2mem` package itself and is not a package dependency.

**Failure behavior:** a failed install here will surface as an `ImportError` in later cells; no silent fallback is attempted.

**Expected result:** all packages install without error (Colab ships `numpy` and `pandas` pre-installed; `pyyaml` and `pytest` install quickly).

In [ ]:
!pip install -q numpy pyyaml pytest pandas

### Step 6 — Write the corrected Increment 2.1 package files

**Technical objective:** write every new, updated, or corrected file to the project folder using `%%writefile`, with contents generated programmatically from the exact source that was tested before this notebook was built (see `INCREMENT_02_v2.1_MANIFEST.md` for the SHA-256 of every file, matching what this notebook writes byte-for-byte).

**Files updated in this step:**
- `pyproject.toml` — version bumped to `0.2.1`; dependencies unchanged (`numpy>=1.24`, `pyyaml>=6.0`).
- `p2mem/__init__.py` — docstring extended to describe the Increment 2.1 correction; `__version__` bumped to `0.2.1`.
- `README.md` — status, directory structure, and limitations updated for the Increment 2.1 correction (see Step 16).
- `p2mem/models.py`, `p2mem/io/las.py`, `p2mem/io/inventory.py` — corrected per the six points in the phase header above (explicit unit-suffixed naming, explicit measured-depth role, blocking file-identity checks, unambiguous coverage statistics, a conversion-signature registry, typed batch failures). The multi-signal curve-resolution architecture itself (ordinal + unit + description-embedded ordinal/name agreement) is UNCHANGED — it worked correctly in Increment 2 and is preserved as-is.

**Files NOT touched in this step:** `p2mem/units.py` remains exactly as delivered and reviewed in Increment 1.1 — it is not rewritten here.

**Validation logic:** none in this step beyond the file having been written; correctness is established by the test suite (Step 9) and the real four-well integration run (Steps 10-15).

**Expected result:** every listed file exists at its expected path with the expected content.

In [ ]:
%%writefile pyproject.toml
[build-system]
requires = ["setuptools>=68.0"]
build-backend = "setuptools.build_meta"

[project]
name = "p2mem"
version = "0.2.1"
description = "Screening-level 1D Mechanical Earth Model workflow for Poseidon 2 (Tier C, uncalibrated / educational)."
readme = "README.md"
requires-python = ">=3.9"
license = { text = "All Rights Reserved. Copyright (c) 2026 Mikael Elgo. This is a personal portfolio project; no license is granted for reuse, redistribution, or commercial use without the author's explicit written permission." }
authors = [
    { name = "Mikael Elgo" }
]
keywords = ["geomechanics", "mechanical-earth-model", "pore-pressure", "wellbore-stability", "portfolio-project"]
classifiers = [
    "Development Status :: 3 - Alpha",
    "Programming Language :: Python :: 3",
    "Intended Audience :: Science/Research",
    "Topic :: Scientific/Engineering",
    "License :: Other/Proprietary License",
]

# Runtime dependencies are deliberately minimal. No unit-handling libraries
# (e.g. Pint) are used: unit conversions are implemented explicitly in
# p2mem.units so that every conversion factor is visible, documented, and
# testable rather than delegated to a third-party unit registry. PyYAML is
# added in Increment 2 for exactly one purpose: parsing the human-authored,
# human-reviewable per-file LAS curve contracts in
# config/las_curve_contracts.yml - a plain-text, diffable format was judged
# preferable to a hand-rolled config parser or a hard-coded Python dict.
dependencies = [
    "numpy>=1.24",
    "pyyaml>=6.0",
]

[project.optional-dependencies]
dev = [
    "pytest>=7.4",
]

[tool.setuptools.packages.find]
include = ["p2mem*"]

[tool.pytest.ini_options]
testpaths = ["tests"]
python_files = ["test_*.py"]


In [ ]:
%%writefile p2mem/__init__.py
"""
p2mem - Poseidon 2 1D Mechanical Earth Model workflow package.

Project classification: Tier C - Screening-Level / Uncalibrated Educational
1D Mechanical Earth Model (see project design review, Rev 1). Nothing in
this package should be presented as a calibrated, operational, or
field-validated result unless an explicit independent calibration record
is attached to that specific output.

This package is under incremental, gated construction.

* Increment 1 / 1.1 delivered the project skeleton and the unit-control
  system (``p2mem.units``).
* Increment 2 added an auditable LAS-ingestion layer with explicit
  per-file curve contracts (``p2mem.io.las``, ``p2mem.io.inventory``,
  ``p2mem.models``) for the four approved wells (Poseidon 2, Boreas 1,
  Poseidon North 1, Proteus 1ST2). It performs LAS parsing, curve-identity
  resolution, NULL-sentinel handling, and factual inventory generation
  ONLY - no deviation-survey processing, MD-to-TVD/TVDSS transformation,
  checkshot processing, formation-top correction, petrophysical
  interpretation, or any later-phase geomechanical calculation.
* Increment 2.1 is a corrective patch to Increment 2, applied after an
  independent technical audit, WITHOUT changing this scope or the
  underlying LAS-parsing/curve-resolution architecture (which the audit
  found sound). It corrects: canonical array naming (every array is now
  explicitly unit-suffixed, e.g. ``VP_m_s`` rather than ``DTCO``, so a
  name can never be mistaken for the wrong physical quantity or unit);
  the measured-depth curve is now located via an explicit contract role
  rather than by matching a canonical name spelled "DEPT"; several
  file-identity checks (filename, SHA-256, WELL, VERS, WRAP, NULL) that
  were not previously blocking now are; curve-coverage statistics now
  report raw AND canonical values with explicit units; and per-well
  batch failures are now typed (``p2mem.models.IngestionFailure``)
  instead of bare caught exceptions. See ``INCREMENT_02_v2.1_MANIFEST.md``
  for the full audit and corrected-file checksums.

Subsequent increments (deviation-survey processing, depth harmonization,
petrophysics, pore pressure, elastic properties, strength, stress, and
wellbore-stability screening) are added one validated phase at a time and
are intentionally absent from this version - importing them will fail
until they exist.
"""

__version__ = "0.2.1"

# Fixed project-wide assurance tier. Referenced by later modules (reporting,
# plotting) so that every generated output can stamp its own classification
# without each module re-declaring the string. This value must not be
# changed without a documented calibration event (e.g. a verified RFT/MDT,
# LOT/XLOT, or core-calibrated log tie) recorded in the method-and-citation
# register.
ASSURANCE_TIER = "Tier C - Screening-Level / Uncalibrated Educational"

__all__ = ["__version__", "ASSURANCE_TIER"]


In [ ]:
%%writefile README.md
# Poseidon 2 — 1D Mechanical Earth Model

**Author:** Mikael Elgo

**Project classification:** Tier C — Screening-Level / Uncalibrated Educational 1D Mechanical Earth Model (MEM)

> **This project is screening-level, uncalibrated, and educational in nature. It is NOT validated against independent field measurements (no confirmed RFT/MDT pressure points, LOT/XLOT tests, or core-calibrated log ties are currently incorporated), and it must NOT be used for operational drilling, well-design, or any real-world decision-making. It exists to demonstrate a technically defensible, transparent, modular geomechanics workflow — not to produce field-ready predictions.**

---

## Purpose and technical scope

This repository implements a modular, reproducible 1D Mechanical Earth Model workflow for the Poseidon 2 well, built from well logs, deviation surveys, checkshot data, formation tops, and Vp/Vs data supplied for the project. The intended end-to-end scope (delivered incrementally, one validated phase at a time) covers:

- data quality control and depth alignment across LAS logs, deviation surveys, and checkshot data
- pore-pressure prediction (Eaton-family methods, contingent on a defensible normal compaction trend)
- elastic properties (dynamic Vp/Vs-derived Poisson's ratio, and density-dependent moduli where density coverage permits)
- rock-strength estimation
- vertical-stress (overburden) modelling
- horizontal-stress and wellbore-stability screening (Kirsch elastic wall-stress equations with Mohr–Coulomb/Mogi–Coulomb failure criteria)
- uncertainty treatment via deterministic low/base/high scenarios and one-at-a-time sensitivity (tornado) analysis, rather than unsupported probabilistic distributions

Every empirical or correlation-based relationship used anywhere in this project (Eaton, Bowers, Gardner, Castagna, etc.) is required to have a recorded source, stated units, applicability range, and calibration status in the project's method-and-citation register *before* it is implemented in code. Nothing is fabricated or assumed silently: missing measurements, missing calibration points, and unavailable data are always reported as unavailable rather than filled in.

This is a personal portfolio project intended to demonstrate scientific rigor, reproducibility, and honest handling of data limitations — not a commercial or operational deliverable.

## Current implementation status

**Increment 2.1 (this release, v0.2.1):** a corrective patch to Increment 2 (LAS ingestion and per-file curve-contract resolution), applied after an independent technical audit, on top of the Increment 1.1 foundation. Increment 2's parsing/curve-resolution architecture was found sound (it loaded all four real wells correctly and passed 131 tests) and is **unchanged**; this release corrects naming, reporting, and contract-validation gaps only. See `INCREMENT_02_v2.1_MANIFEST.md` for the full audit and every corrected file's SHA-256.

- `p2mem/units.py` — an explicit, NumPy-based unit-conversion layer (no external unit-registry dependency such as Pint) implementing 21 public conversion functions between oilfield and SI-internal units. Unchanged from Increment 1.1 (locked; not modified in Increment 2 or 2.1). See the module docstring and `tests/test_units.py`.
- `p2mem/models.py` — typed, documented dataclasses for every LAS-ingestion result object (header info, curve contracts, resolutions, depth diagnostics, curve statistics, typed batch failures) rather than undocumented tuples. Curve identity now carries an explicit, unit-suffixed `raw_canonical_name`/`canonical_name` pair (e.g. `DTCO_us_per_ft` raw, `VP_m_s` canonical) and a `semantic_role` (`measured_depth` for exactly one curve per file, `measurement` for every other) — corrected in 2.1 so a canonical array's name can never be mistaken for a different physical quantity or unit, and so the depth curve is never located by matching a canonical name spelled "DEPT".
- `p2mem/io/las.py` — an auditable LAS 2.0 parser and per-file curve-contract resolver for the four approved wells (Poseidon 2, Boreas 1, Poseidon North 1, Proteus 1ST2). Every curve's identity is resolved by requiring agreement between its ordinal position, unit, and description-embedded ordinal/name (never ordinal position alone) against an explicit contract — never guessed, never filled in with a generic `UNKNOWN` label. Corrected in 2.1 to additionally treat a mismatched filename, SHA-256, WELL, VERS, WRAP, or NULL value as a blocking `ERROR` (an independent audit found Increment 2's resolver did not check these at all); to validate a contract's `conversion_function` against a controlled unit-signature registry at load time; and to isolate per-well batch failures as typed `IngestionFailure` records. See the module docstring for the full design rationale.
- `p2mem/io/inventory.py` — deterministic, metadata-only inventory-table builders (file inventory, curve catalog, curve coverage, ingestion issues, JSON manifest) — never raw LAS sample data. `las_curve_coverage.csv` is corrected in 2.1 to report raw AND canonical statistics side by side, each tagged with its own explicit unit.
- `config/las_curve_contracts.yml` — the human-authored, human-reviewable curve contract for each of the four wells, built by directly inspecting each file's actual LAS header (see `INCREMENT_02_v2.1_MANIFEST.md` for the inspection method and full per-curve rationale, including the Proteus 1ST2 column-order anomaly and the Boreas 1 ECGR scale anomaly). Each well entry now also declares `expected_sha256`, `expected_las_version`, `expected_wrap`, and `expected_data_layout`.
- Still not implemented: deviation-survey processing, MD-to-TVD/TVDSS transformation, checkshot processing, formation-top correction, petrophysical interpretation (shale volume, lithology), density estimation, NCT fitting, pore-pressure prediction, elastic properties, rock strength, stresses, or wellbore-stability screening. Those are explicitly out of scope for this increment and are added one gated increment at a time in later releases.

## Installation

Requires Python 3.9 or later.

```bash
# from the project root (the directory containing pyproject.toml)
pip install -e .
```

This installs the `p2mem` package in editable mode along with its runtime dependencies: NumPy (`numpy>=1.24`) and, as of Increment 2, PyYAML (`pyyaml>=6.0`) — added for exactly one purpose, parsing the human-authored curve contracts in `config/las_curve_contracts.yml`. To also install the test dependency:

```bash
pip install -e ".[dev]"
```

## Running the tests

```bash
pytest -v
```

The suite in `tests/test_units.py` validates `p2mem/units.py` (unchanged since Increment 1.1) against analytical reference values, round-trip consistency, scalar/array inputs, NaN preservation, and rejection of invalid/nonphysical/ambiguous inputs. The suite in `tests/test_las.py` validates `p2mem/io/las.py` against small synthetic LAS fixtures under `tests/fixtures/` — it does NOT require the four private/raw project LAS files, so it runs the same way for anyone who clones this repository. Run the command above and read the reported pass/fail count directly — this document does not assert a fixed expected count, since that must always be read from the actual `pytest` output for the code currently on disk. Real four-well integration validation (which DOES require the raw LAS files, not included in this repository) is a separate notebook run — see `02_LAS_Ingestion_and_Curve_Contracts.ipynb`.

## Directory structure

```
Poseidon_1D_MEM/
├── README.md
├── pyproject.toml
├── p2mem/
│   ├── __init__.py
│   ├── models.py
│   ├── units.py
│   └── io/
│       ├── __init__.py
│       ├── las.py
│       └── inventory.py
├── tests/
│   ├── test_units.py
│   ├── test_las.py
│   └── fixtures/         (small synthetic LAS files; no project raw data)
├── config/
│   └── las_curve_contracts.yml
├── data/
│   └── raw/
│       └── logs/         (the four raw LAS files - NOT included in this repository; immutable inputs)
├── notebooks/  (reserved for later increments)
└── outputs/
    └── 02_las_inventory/  (generated by the Increment 2 notebook's real four-well run: CSV/JSON metadata only, no raw log samples)
```

`notebooks/` is created empty by the project-setup notebook cell and is not yet populated in-repo (the increment notebooks themselves are delivered as top-level files, e.g. `02_LAS_Ingestion_and_Curve_Contracts.ipynb`, and are meant to be run from Google Drive per their own directory-setup cells).

## Scientific limitations

These limitations are specific to the Poseidon 2 dataset and this project's current increment, and are carried forward here so they are visible outside the conversation in which they were identified:

- **RHOB (bulk density) coverage in Poseidon 2 ends at approximately 5,296.85 m MD.** Sonic and other curves continue deeper, so Vp/Vs and dynamic Poisson's ratio remain computable below that depth, but density-dependent properties (Young's modulus, shear modulus, bulk modulus, acoustic impedance, shear impedance) are unavailable below it unless density is explicitly estimated and flagged as such — never silently substituted.
- **No reliable shale-based normal compaction trend (NCT) exists from Poseidon 2 alone.** A provisional, transferred candidate NCT identified in offset well Poseidon North 1 is a *candidate*, not a validated trend, and must not be presented as calibrated.
- **Independently measured Vp/Vs quality flags:** approximately 3.38% of Poseidon 2 Vp/Vs values fall below 1.5, and approximately 0.53% fall below the physical validity cutoff of √2 (≈1.4142) required for a non-negative dynamic Poisson's ratio.
- **No independent calibration data (RFT/MDT pressure points, LOT/XLOT tests, or core data) has been supplied or incorporated.** Any pore-pressure or stress output in later increments must be presented as a bounded or theoretical estimate, not a validated field prediction.
- **Empirical/correlation equations are not implemented until their governing equation, units, applicability range, and calibration status are recorded in the project's method-and-citation register.** Several candidate methods remain in "pending" status and are intentionally absent from the codebase for that reason, not because they were overlooked.
- Additional open items (offset-well GR/ECGR scale adjudication, missing formation tops for one offset well) are tracked in the project's design-review documentation and gate specific later phases (lithology and pore-pressure), not this increment.
- **Increment 2 update:** LAS ingestion independently reconfirms (does not newly discover, and does not act on) two previously-flagged anomalies from the Rev 1 design review: Boreas 1's ECGR curve (canonical name `ECGR_api`) ranges from approximately −0.0001 to 519.18 API (vs. roughly 5–205 API for the other three wells' GR-family curves) with 96.98% valid coverage; and Proteus 1ST2's LAS log file places its neutron-porosity curve (canonical name `NPHI_pct`) at column position 5 rather than the last position (8) used by the other three wells. Both are reported as ingestion facts (see `outputs/02_las_inventory/`); neither is rescaled, reinterpreted, or otherwise acted on by this increment.
- **Increment 2.1 update:** a corrective patch addressing an independent audit's naming, reporting, and contract-validation findings — see "Current implementation status" above and `INCREMENT_02_v2.1_MANIFEST.md`. No new scientific finding was made in 2.1; the two anomalies above are unaffected and remain open items for a later, explicitly-scoped increment.

## Screening-level statement

**This 1D Mechanical Earth Model is a screening-level, uncalibrated, educational work product.** It has not been validated against independent field measurements and does not carry the assurance level required for drilling engineering, well design, casing/mud-weight selection, or any other operational decision. Any numerical result produced by this codebase should be read as illustrative of a defensible methodology applied to the available data, not as a certified or field-ready prediction.


#### `p2mem/models.py`

**LAS theory / why typed result objects:** a LAS file's structure (Version/Well/Curve/Parameter/Ascii-data sections) naturally decomposes into several related-but-distinct kinds of fact: what the header literally declares, what a contract expects, whether they agreed, and what the resulting data looks like. Returning an undocumented tuple for all of this would force every caller to remember positional meaning; `p2mem/models.py` gives each kind of fact its own frozen, documented dataclass instead.

In [ ]:
%%writefile p2mem/models.py
"""
p2mem.models - Minimal typed result/schema objects for the LAS-ingestion
layer (Increment 2 / 2.1).

Design rationale
-----------------
The Increment 2 specification explicitly requires "typed, documented
result objects rather than an undocumented tuple." This module collects
every such object in one place so that `p2mem/io/las.py` can focus on
parsing/resolution logic and every caller (tests, the notebook, the
inventory builder) imports structure from a single, auditable source.

All objects here are plain, frozen `dataclasses` - no third-party schema
library (e.g. pydantic) is introduced, consistent with the project's
minimal-dependency policy. Frozen dataclasses are used wherever an object
represents an immutable fact about a specific ingestion run (a parsed
header, a resolved curve, a computed statistic); this makes it impossible
to accidentally mutate a diagnostic result after it has been computed and
reported.

Nothing in this module performs I/O, unit conversion, or numerical
computation - it is data structure only.

Increment 2.1 correction
-------------------------
An independent audit of Increment 2 found that canonical curve names were
not unit-suffixed (e.g. a *velocity* array, in m/s, was stored under the
key "DTCO" - the mnemonic that conventionally means sonic *slowness* in
us/ft). This risks a downstream reader assuming the wrong physical
quantity purely from the key name. Increment 2.1 requires every raw and
canonical array to be identified by an explicit, unit-suffixed name (e.g.
"DTCO_us_per_ft" for the raw slowness, "VP_m_s" for the converted
velocity), and requires a curve contract to declare which single curve
plays the "measured depth" role explicitly, rather than that role being
inferred by searching for a canonical name spelled "DEPT". The fields
below reflect that correction; see `CurveContractEntry.semantic_role`,
`CurveContractEntry.source_curve_name`, `CurveContractEntry.raw_canonical_name`,
and the redesigned `CurveStats`.
"""

from __future__ import annotations

from dataclasses import dataclass, field
from typing import Dict, Optional, Tuple

import numpy as np

# Role a contract-declared curve plays. "measured_depth" is the single
# curve used as the depth index for diagnostics (see
# `las.py:_MEASURED_DEPTH_ROLE`); every other curve is a "measurement".
SEMANTIC_ROLE_MEASURED_DEPTH = "measured_depth"
SEMANTIC_ROLE_MEASUREMENT = "measurement"


# ---------------------------------------------------------------------------
# Raw LAS header structures (what was literally present in the file)
# ---------------------------------------------------------------------------
@dataclass(frozen=True)
class DefinitionLine:
    """
    One parsed LAS "definition line" from the Version, Well, Curve, or
    Parameter sections, in the standard LAS 2.0 four-field form:

        MNEM.UNIT    VALUE/NAME    :DESCRIPTION

    Every field is preserved exactly as read (including empty strings for
    an absent mnemonic or unit) - no field is inferred or filled in.
    """

    mnemonic: str
    unit: str
    value: str
    description: str


@dataclass(frozen=True)
class CurveHeaderEntry:
    """
    One curve as declared in a LAS file's ~Curve Information Section,
    in original file order.

    `ordinal` is the curve's zero-based position within the curve section
    (and, for an unwrapped LAS file, its corresponding zero-based column
    position in the ~Ascii data section). `description_ordinal` and
    `description_name` are parsed OUT of the free-text description when it
    follows the "<index> <NAME>" pattern seen in the Poseidon 2 / Boreas 1
    / Poseidon North 1 / Proteus 1ST2 files (e.g. "1 DCAV") - they are
    independent, file-content-derived signals used to cross-check
    `ordinal`, never a substitute for it.
    """

    ordinal: int
    raw_mnemonic: str
    raw_unit: str
    raw_api_code: str
    raw_description: str
    description_ordinal: Optional[int]
    description_name: Optional[str]


@dataclass(frozen=True)
class LasHeaderInfo:
    """
    Everything read from a LAS file's header sections (Version, Well,
    Curve, Parameter), plus file-identity metadata, WITHOUT reading or
    parsing the ~Ascii data section. This is what notebook Step 11
    ("Inspect actual LAS headers") and Step 12 ("Validate each per-file
    contract") operate on before any numeric data is touched.
    """

    source_path: str
    source_filename: str
    sha256: str
    las_version: Optional[str]
    wrap: Optional[str]
    well_name: Optional[str]
    declared_null: Optional[float]
    declared_strt: Optional[float]
    declared_stop: Optional[float]
    declared_step: Optional[float]
    version_section: Tuple[DefinitionLine, ...]
    well_section: Tuple[DefinitionLine, ...]
    parameter_section: Tuple[DefinitionLine, ...]
    curve_headers: Tuple[CurveHeaderEntry, ...]
    data_section_line_offset: int  # line index (0-based) where ~Ascii data begins


# ---------------------------------------------------------------------------
# Curve contract structures (what the config/las_curve_contracts.yml says
# a given curve in a given file SHOULD be)
# ---------------------------------------------------------------------------
@dataclass(frozen=True)
class CurveContractEntry:
    """
    One curve's expected identity and canonicalization rule, as declared
    in `config/las_curve_contracts.yml` for one specific source file.

    Identity fields (never fabricated - read directly from the source
    header during contract authoring):
        `source_curve_name`   - the physical curve's own short identity
                                 token (e.g. "DTCO", "GR", "DEPT"), as used
                                 in this file's own description text. This
                                 is NOT necessarily the array key - see
                                 below.
        `raw_mnemonic`, `raw_unit`, `raw_description`, `description_ordinal`,
        `description_name` - exactly what Increment 2 already captured.

    Naming fields (Increment 2.1 correction - explicit, unit-suffixed
    array keys so a key name can never be mistaken for the wrong physical
    quantity):
        `raw_canonical_name`  - the key under which this curve's raw
                                 (NULL-sentinel-preserving) values would be
                                 identified, e.g. "DTCO_us_per_ft". This
                                 documents the raw unit in the name; the
                                 numeric matrix itself is still
                                 `LasFileResult.raw_data` (see that
                                 docstring for why the matrix form is
                                 retained).
        `canonical_name`      - the key in `LasFileResult.canonical_data`
                                 holding the NULL-substituted, exactly-
                                 unit-converted (if `conversion_function`
                                 is not "identity") array, e.g. "VP_m_s".
                                 For a curve with no declared conversion,
                                 `canonical_name` still carries an explicit
                                 unit suffix (e.g. "GR_api"), never the
                                 bare source name alone.
        `canonical_unit`      - the unit of the array stored under
                                 `canonical_name`.

    `semantic_role` is `"measured_depth"` for exactly one curve per file
    (the file's depth index) and `"measurement"` for every other curve.
    The depth role is never inferred by matching a canonical name spelled
    "DEPT" - see `las.py:_MEASURED_DEPTH_ROLE` and
    `load_file_contract_config`'s validation.

    `conversion_function` is either the literal string `"identity"` (no
    numeric transform - the canonical value equals the NULL-substituted
    raw value; only the array's *name* changes when depth is renamed
    DEPT -> MD) or the name of an exact, unit-only function in
    `p2mem.units` (e.g. "us_per_ft_to_m_per_s"). It is never `None` in
    Increment 2.1 - every curve entry must say explicitly how its
    canonical array was produced.
    """

    ordinal: int
    semantic_role: str
    source_curve_name: str
    raw_mnemonic: str
    raw_unit: str
    raw_description: str
    description_ordinal: Optional[int]
    description_name: Optional[str]
    raw_canonical_name: str
    canonical_name: str
    canonical_unit: str
    required: bool
    conversion_function: str
    notes: str


@dataclass(frozen=True)
class FileContract:
    """
    The complete expected curve contract for one specific LAS source file.

    Increment 2.1 adds the file-identity fields an independent audit found
    the Increment 2 resolver did not actually check: `expected_sha256`,
    `expected_las_version`, `expected_wrap`, and `expected_data_layout`.
    These, together with `expected_well_identifier`, `expected_null_value`,
    and `expected_curve_count`, are cross-checked against the file's
    actual parsed header by `las.py:resolve_curve_contract` BEFORE any
    curve-level resolution is attempted, and any mismatch (or an absent
    value where one is required) is a blocking ERROR - never a warning,
    and never silently ignored.

    `expected_null_value` is required (not optional) in Increment 2.1: a
    contract that does not know its file's NULL sentinel cannot safely
    validate NULL-substitution, which the Increment 2 audit identified as
    a real gap (the resolver previously "passed" a file with no NULL
    value declared and a contract that also left it unset).
    """

    source_filename: str
    expected_sha256: str
    expected_well_identifier: str
    expected_las_version: str
    expected_wrap: str
    expected_null_value: float
    expected_curve_count: int
    expected_data_layout: str
    curves: Tuple[CurveContractEntry, ...]


# ---------------------------------------------------------------------------
# Resolution / diagnostic results (what happened when the contract was
# checked against the actual header, and what the data looks like)
# ---------------------------------------------------------------------------
@dataclass(frozen=True)
class CurveResolution:
    """
    The outcome of attempting to resolve one contract curve entry against
    the actual parsed curve headers of a specific file.

    `status` is one of:
        "RESOLVED"         - all required identifiers agreed; safe to use.
        "MISSING_OPTIONAL" - the curve was not found, but the contract
                              marks it optional, so this is not an error.
        "FAILED"           - the curve could not be safely resolved (a
                              required curve was missing, or the
                              identifiers that were found disagreed with
                              each other). This never means "a different
                              column was substituted" - resolution either
                              succeeds cleanly or is reported as failed.

    `canonical_name` identifies the contract entry that was evaluated
    (matches `CurveContractEntry.canonical_name`); `source_curve_name` is
    carried alongside it so a resolution report never needs a second
    lookup into the contract to explain what physical curve was involved.
    """

    canonical_name: str
    source_curve_name: str
    status: str
    ordinal: Optional[int]
    matched_raw_mnemonic: Optional[str]
    matched_raw_unit: Optional[str]
    matched_raw_description: Optional[str]
    conversion_applied: Optional[str]
    detail: str


@dataclass(frozen=True)
class IngestionIssue:
    """
    One fact worth reporting about an ingestion attempt: either an ERROR
    (blocks a successful load - see `load_las_file`) or a WARNING
    (recorded and surfaced, but does not by itself stop ingestion, per
    the Increment 2 specification's distinction between curves the loader
    must "detect/report" and conditions the loader must "reject").

    Increment 2.1 note: a NULL-sentinel mismatch between a file's declared
    header value and its contract's `expected_null_value` is now always
    reported as an ERROR (see code "NULL_MISMATCH"), not a WARNING - a
    wrong NULL value changes which samples get substituted to NaN, so it
    is a data-safety issue, not a cosmetic one.
    """

    severity: str  # "ERROR" or "WARNING"
    code: str
    message: str
    context: str


@dataclass(frozen=True)
class DepthDiagnostics:
    """
    Data-derived depth-index diagnostics, and their comparison against the
    file's declared STRT/STOP/STEP header values. Comparisons use a small
    floating-point tolerance (see `las.py:_DEPTH_ENDPOINT_TOLERANCE_M`),
    never exact equality, since declared header values are frequently
    rounded to a "nice" number while the data's actual first/last sample
    is not.
    """

    declared_strt: Optional[float]
    declared_stop: Optional[float]
    declared_step: Optional[float]
    data_start: float
    data_stop: float
    data_min_step: float
    data_max_step: float
    data_median_step: float
    n_samples: int
    n_duplicate_md: int
    n_non_monotonic: int
    strt_matches_declared: Optional[bool]
    stop_matches_declared: Optional[bool]
    step_matches_declared: Optional[bool]


@dataclass(frozen=True)
class CurveStats:
    """
    Descriptive (not QC-judgemental) statistics for one resolved curve,
    reported with EVERY number's unit made explicit - the Increment 2.1
    correction for an audit finding that the Increment 2 coverage output
    reported minimum/maximum values under a canonical (post-conversion-
    implying) name while the numbers themselves were still in the raw,
    pre-conversion unit.

    Two independent sets of statistics are always reported side by side:

        raw_min / raw_max        - computed directly from `raw_unit`
                                    values (the literal parsed column,
                                    e.g. from `LasFileResult.raw_data`),
                                    restricted to samples that are NOT an
                                    exact match of the file's declared
                                    NULL sentinel (a sentinel value like
                                    -999.25 is a metadata marker, not a
                                    physical measurement, so including it
                                    in a "minimum value" would misrepresent
                                    the curve's actual physical range).
        canonical_min / canonical_max - computed from the corresponding
                                    `LasFileResult.canonical_data[canonical_name]`
                                    array (NULL-substituted to NaN, and
                                    exactly unit-converted if
                                    `conversion_function != "identity"`).

    For an identity conversion (`conversion_function == "identity"`),
    `raw_min == canonical_min` and `raw_max == canonical_max` numerically,
    but `raw_unit` and `canonical_unit` are still both reported explicitly
    (they are typically equal strings, e.g. both "API") so a reader never
    has to assume they match.

    `statistics_basis` states in one sentence exactly what filtering was
    applied, so this object is self-describing without cross-referencing
    the code that produced it.
    """

    source_curve_name: str
    raw_mnemonic: str
    raw_description: str
    raw_unit: str
    canonical_name: str
    canonical_unit: str
    conversion_function: str
    n_samples: int
    valid_count: int
    null_count: int
    valid_fraction: float
    raw_min: Optional[float]
    raw_max: Optional[float]
    canonical_min: Optional[float]
    canonical_max: Optional[float]
    statistics_basis: str


@dataclass(frozen=True)
class IngestionFailure:
    """
    A typed, structured record of why one well failed to load during a
    batch (`las.py:load_wells`) - the Increment 2.1 correction for an
    audit finding that Increment 2 stored a bare caught exception per
    failed well, forcing every caller to parse exception text or
    `isinstance` checks to know what kind of failure occurred.

    `error_type` is one of:
        "file_not_found"    - the LAS file path does not exist
                               (`las.py:LasFileNotFoundError`).
        "parsing_failure"   - a structural LAS-syntax defect
                               (`las.py:LasParsingError`): a malformed
                               definition line, a data row with the wrong
                               column count, a non-numeric token, or a
                               non-finite (NaN/Inf) literal token that does
                               not correspond to the declared NULL policy.
        "contract_failure"  - the file's actual header/data could not be
                               safely reconciled with its contract
                               (`las.py:LasContractError`): a file-identity
                               mismatch (filename, SHA-256, WELL, VERS,
                               WRAP, NULL, curve count), an unresolved
                               required curve, or a data-column-count
                               mismatch.
        "conversion_failure"- an exact unit conversion could not be safely
                               applied to a curve's substituted values
                               (`las.py:LasConversionError`): typically a
                               non-null physical value that converts to a
                               non-finite (Inf/-Inf) result.

    `exception` is the original exception object, preserved so a caller
    that wants the full original detail (e.g. `.issues`/`.resolutions` on
    a `LasContractError`) still has it; `message` is `str(exception)` for
    the common case of just wanting to display or log the failure without
    re-importing every exception type.

    This is a *typed* structure, not a substitute for exceptions:
    `load_wells` still lets any exception NOT in this list (a programming
    error, not an expected ingestion-failure mode) propagate uncaught.
    """

    well_key: str
    source_path: str
    error_type: str
    message: str
    exception: BaseException


@dataclass(frozen=True)
class LasFileResult:
    """
    The complete, typed result of successfully loading and contract-
    resolving one LAS file. Returned only when no ERROR-severity issue
    was found (see `las.py:load_las_file`); `issues` may still contain
    WARNING-severity entries (e.g. a duplicate MD sample, a declared/
    data-derived STOP mismatch).

    `raw_data` holds the EXACT parsed numeric matrix, column-ordered
    exactly as the source file's curve section (shape: n_samples x
    n_curves), INCLUDING the original LAS NULL sentinel value wherever it
    appears - no substitution of any kind has been applied to it.

    Increment 2.1 correction: the Increment 2 docstring for this field
    incorrectly stated that `raw_data` had "NULL-substitution ... applied"
    to it. That was never true of the array itself (only of the separate,
    per-curve arrays computed on demand for `canonical_data` and
    `curve_stats`) and has been corrected here. Column `entry.ordinal` of
    this matrix corresponds to `contract.curves[i].raw_canonical_name` for
    the curve whose contract entry has that ordinal.

    `canonical_data` holds one NumPy array per contract-resolved curve,
    keyed by that curve's `canonical_name` (an explicit, unit-suffixed
    name, e.g. "VP_m_s", "GR_api", "MD_m" - never a bare mnemonic that
    could be mistaken for a different physical quantity or unit). Each
    array has had NULL-sentinel-to-NaN substitution applied and, if the
    contract's `conversion_function` for that curve is not `"identity"`,
    the declared exact `p2mem.units` conversion applied. The raw source
    column is never lost - it remains recoverable from `raw_data` at that
    curve's `ordinal`.
    """

    header: LasHeaderInfo
    contract: FileContract
    resolutions: Tuple[CurveResolution, ...]
    depth: DepthDiagnostics
    curve_stats: Tuple[CurveStats, ...]
    raw_data: np.ndarray
    canonical_data: Dict[str, np.ndarray] = field(default_factory=dict)
    issues: Tuple[IngestionIssue, ...] = field(default_factory=tuple)
    contract_status: str = "PASSED"


#### `p2mem/io/__init__.py` and `p2mem/io/las.py`

**LAS theory / file-format basis:** a LAS 2.0 file is organized into `~`-prefixed sections (Version, Well, Curve, Parameter, Other, Ascii data), each holding `MNEM.UNIT  VALUE  :DESCRIPTION`-style definition lines (except the Ascii data section itself, which is whitespace-delimited numeric rows, one per depth step for an unwrapped file — `WRAP.  NO` in all four approved files). The full design rationale — including the empty-mnemonic problem, the Proteus 1ST2 column-order anomaly, and the Boreas 1 ECGR handling policy — is reproduced in this module's own docstring below; it is not duplicated here to avoid the two ever silently drifting apart.

**Why the curve-contract system is necessary:** summarized in the phase header above — no universal column-order assumption is safe across these four files, and an empty mnemonic can never be resolved from ordinal position alone.

**Inputs:** LAS file paths, and `config/las_curve_contracts.yml`.
**Outputs:** typed `LasHeaderInfo` / `LasFileResult` objects (see `p2mem/models.py`); no CSV/JSON is written by this module itself (that is `p2mem/io/inventory.py`'s job).

**Validation logic and failure behavior:** documented exhaustively in the module docstring and per-function docstrings below - in summary, a required curve that cannot be resolved, a duplicate/ambiguous description, a unit or well-identifier mismatch, a data-column-count mismatch, or a malformed numeric row all raise an explicit, actionable exception; duplicate/non-monotonic measured depth are detected and reported as warnings without blocking ingestion.

In [ ]:
%%writefile p2mem/io/__init__.py
"""
p2mem.io - File-ingestion layer for the Poseidon 2 MEM project.

Increment 2 adds the LAS-ingestion module (`p2mem.io.las`) and its
inventory-report builder (`p2mem.io.inventory`). No other file formats
(deviation surveys, checkshots, formation tops) are ingested by this
package yet - those are explicitly out of scope for this increment.
"""

from __future__ import annotations

__all__ = ["las", "inventory"]


In [ ]:
%%writefile p2mem/io/las.py
"""
p2mem.io.las - Auditable LAS 2.0 ingestion with explicit per-file curve
contracts (Increment 2, corrected in Increment 2.1).

Why this module exists
------------------------
The four LAS files approved for this project (Poseidon 2, Boreas 1,
Poseidon North 1, Proteus 1ST2) all have EMPTY mnemonic fields for every
curve except DEPT: the ~Curve Information Section carries only a unit and
a free-text description such as ":  1 DCAV" (an embedded ordinal number
plus the intended curve name). A generic LAS parser has no reliable way
to name these curves - it would either fabricate placeholder names
(``UNKNOWN``, ``UNKNOWN:1``) or guess identity from column position alone,
either of which risks silently mislabeling a curve.

This module never guesses. Every curve in every file is resolved against
an explicit, human-authored contract (`config/las_curve_contracts.yml`)
that was built by directly inspecting each file's actual header text (see
the Increment 2 manifest for the inspection method). Resolution requires
several independent identifiers to agree - ordinal position, the
description's embedded ordinal, the description's embedded name, and unit
- and NEVER relies on ordinal position alone, per the project's explicit
mandate. Any disagreement fails that curve's resolution rather than
silently choosing a different column.

Two further LAS-format nuances specific to these files, discovered while
building the contracts, are handled explicitly rather than assumed away:

* Column order is NOT the same across the four files. Poseidon 2, Boreas 1,
  and Poseidon North 1 all place their neutron-porosity curve last
  (ordinal 8), but Proteus 1ST2 places it at ordinal 5, shifting RD, RHOB,
  and RS one position earlier than in the other three files. A shared,
  cross-well column-order assumption would silently mislabel Proteus's
  curves - hence one contract per file, never one contract for all wells.
* The curve carrying gamma-ray-like information is named differently in
  every file that has one (GR in Poseidon 2 and Proteus 1ST2, ECGR in
  Boreas 1, GRD in Poseidon North 1). Boreas 1's ECGR curve is additionally
  known (from the Rev 1 design review) to carry an unresolved scale
  anomaly. This module reports that anomaly as a fact (via curve
  statistics) but does not rescale it, reinterpret it, or otherwise act on
  it - that adjudication remains outside this increment's scope.

Scope boundary
--------------
This module parses LAS structure and resolves curve identity ONLY. It
performs no deviation-survey processing, no MD-to-TVD conversion, no
petrophysical interpretation, and no empirical/correlation calculations.
The only numeric transformations it ever applies are (a) NULL-sentinel
substitution to NaN and (b) the exact, contract-declared unit conversions
implemented in `p2mem.units` (e.g. us/ft -> m/s, g/cc -> kg/m3). No curve
is clipped, interpolated, despiked, resampled, smoothed, normalized, or
derived from another curve.

Increment 2.1 correction (independent audit of Increment 2)
-------------------------------------------------------------
The parsing architecture above (definition-line parsing, multi-signal
curve resolution, per-file contracts) is UNCHANGED and is preserved as-is
- it worked correctly against all four real wells and is not the subject
of this correction. What changed:

1. Canonical array names are now explicit and unit-suffixed everywhere
   (e.g. "VP_m_s" for converted compressional velocity, "DTCO_us_per_ft"
   as the raw curve's documented identity) - a canonical name can no
   longer be mistaken for a different physical quantity or unit than the
   array it labels. See `p2mem.models.CurveContractEntry`.
2. The "measured depth" curve is now identified by an explicit contract
   field (`semantic_role: measured_depth`), never by searching for a
   canonical name spelled "DEPT" - a contract that renames its depth
   curve's canonical target (as this correction itself does, to "MD_m")
   would otherwise silently break depth-diagnostic computation.
3. File-identity checks an independent audit found the Increment 2
   resolver did NOT perform (a missing WELL, a missing NULL, a changed
   VERS or WRAP, or even a contract for a different file than the one
   actually being loaded, all previously passed silently) are now
   blocking ERRORs, checked before any curve is resolved. A NULL mismatch
   is now an ERROR, not a WARNING, because it changes which samples get
   substituted to NaN.
4. Curve-coverage statistics are now reported as an explicit RAW/
   CANONICAL pair, each with its own unit, rather than a single
   min/max pair mislabeled with a canonical (implicitly converted) name
   while actually holding pre-conversion values.
5. A curve contract is validated, at YAML-load time, against a small
   conversion-signature registry so it cannot declare an internally
   incompatible unit/conversion-function pairing (e.g. applying
   `gcc_to_kgm3` to an API gamma-ray curve).
6. `load_wells` now returns a typed `IngestionFailure` per failed well
   (carrying an `error_type` of "file_not_found", "parsing_failure",
   "contract_failure", or "conversion_failure") instead of a bare caught
   exception, and isolates all four of those expected failure modes per
   well; any other exception is a programming error and still propagates.
"""

from __future__ import annotations

import hashlib
import math
import re
from pathlib import Path
from typing import Callable, Dict, FrozenSet, List, Optional, Tuple

import numpy as np
import yaml

from p2mem import units
from p2mem.models import (
    SEMANTIC_ROLE_MEASURED_DEPTH,
    SEMANTIC_ROLE_MEASUREMENT,
    CurveContractEntry,
    CurveHeaderEntry,
    CurveResolution,
    CurveStats,
    DefinitionLine,
    DepthDiagnostics,
    FileContract,
    IngestionFailure,
    IngestionIssue,
    LasFileResult,
    LasHeaderInfo,
)

__all__ = [
    "LasFileNotFoundError",
    "LasParsingError",
    "LasContractDefinitionError",
    "LasContractError",
    "LasConversionError",
    "parse_las_header",
    "load_file_contract_config",
    "resolve_curve_contract",
    "compute_depth_diagnostics",
    "compute_curve_stats",
    "load_las_file",
    "load_wells",
]


# ---------------------------------------------------------------------------
# Exceptions
# ---------------------------------------------------------------------------
class LasFileNotFoundError(FileNotFoundError):
    """Raised when a required LAS source file cannot be found on disk."""


class LasParsingError(ValueError):
    """
    Raised for a structural/syntactic defect that prevents safely parsing a
    LAS file: a definition line with no '.' separator, a data row whose
    column count does not match the declared curve count, a data token
    that cannot be parsed as a number, or (Increment 2.1) a data token
    that DOES parse but to a non-finite value (a literal "nan"/"inf"/"-inf"
    text token) - this project's NULL policy is a single finite sentinel
    value, so a literal non-finite token is never treated as that
    sentinel and is rejected as a structural defect rather than silently
    substituted, clipped, or passed through. The loader stops immediately
    rather than skipping or guessing at the offending row.
    """


class LasContractDefinitionError(ValueError):
    """
    Raised when `config/las_curve_contracts.yml` itself is malformed or
    internally inconsistent. Increment 2.1 checks (in addition to the
    Increment 2 checks - an unparsable YAML document, a missing required
    field, an unknown `conversion_function`, or two curves mapped to the
    same `canonical_name`): a declared but unimplemented LAS version,
    WRAP value, or data layout; a non-boolean `required`; a non-numeric
    `expected_null_value`; a duplicate or negative curve ordinal; an
    ordinal set with an unjustified gap; a `conversion_function` whose
    declared raw/canonical units are incompatible with its registered
    signature; zero or more than one curve declared with
    `semantic_role: measured_depth`; or a measured-depth curve whose
    identity/canonical name/unit do not match the required convention.
    This is a configuration-authoring error, distinct from a data-
    resolution error.
    """


class LasContractError(RuntimeError):
    """
    Raised by `load_las_file` when the file's actual header cannot be
    safely reconciled with its contract: a file-identity mismatch
    (filename, SHA-256, WELL, VERS, WRAP, NULL, or curve count), a
    required curve failed to resolve, or a data-column-count mismatch was
    found.

    Ingestion stops - no alternate column is ever silently substituted.
    The exception carries `.issues` (all `IngestionIssue`s found, both
    ERROR and any WARNING already raised at that point), `.resolutions`
    (every attempted `CurveResolution`), and `.header` (the successfully
    parsed `LasHeaderInfo`, for diagnostic reporting, or `None` if the
    contract does not even define a measured-depth curve) so a caller can
    report exactly what went wrong without re-parsing the file.
    """

    issues: Tuple[IngestionIssue, ...] = ()
    resolutions: Tuple[CurveResolution, ...] = ()
    header: Optional[LasHeaderInfo] = None


class LasConversionError(RuntimeError):
    """
    Raised when applying a contract-declared exact unit conversion to a
    curve's NULL-substituted values cannot be done safely: either the
    conversion function itself raised (e.g. rejected an ambiguous dtype),
    or it produced a non-finite (Inf/-Inf) result from a value that was
    NOT a NULL-sentinel match - i.e. a real, non-null measurement that the
    exact conversion cannot represent (for example, a zero sonic slowness
    converting to an infinite velocity).

    This exception always names the source file, the curve involved
    (by its source curve name and canonical name), the expected canonical
    unit, and - when identifiable - the offending raw value and its row's
    measured depth, so a caller never sees a raw NumPy exception with no
    project context.
    """


# ---------------------------------------------------------------------------
# Constants
# ---------------------------------------------------------------------------
# Declared STRT/STOP are frequently rounded to a "nice" number rather than
# the exact first/last sample (e.g. Poseidon 2 declares STOP=5351.0000 m
# while its last actual sample is 5350.9507 m) - a tolerance-based
# comparison is therefore used rather than exact equality.
_DEPTH_ENDPOINT_TOLERANCE_M = 0.5
_DEPTH_STEP_TOLERANCE_M = 1e-3
_NULL_TOLERANCE = 1e-6

_DESC_ORDINAL_RE = re.compile(r"^\s*(\d+)\s+(\S.*?)\s*$")

# What this parser actually implements. A contract that declares anything
# outside these sets is rejected at YAML-load time (LasContractDefinitionError)
# rather than silently "supported" - see the Increment 2.1 module docstring,
# point 3, and the corrective-patch requirement: "do not pretend to support
# a mode that has not been implemented."
_SUPPORTED_LAS_VERSIONS: FrozenSet[str] = frozenset({"2.0"})
_SUPPORTED_WRAP_VALUES: FrozenSet[str] = frozenset({"NO"})
_SUPPORTED_DATA_LAYOUTS: FrozenSet[str] = frozenset({"unwrapped_whitespace_delimited"})

# Conversion functions this module is permitted to apply, keyed by the
# name a contract entry may write in `conversion_function`. Every entry
# here is an EXACT, unit-only conversion from p2mem.units (Increment 1.1);
# no empirical/correlation function is ever listed here. "identity" is
# handled separately (see `_KNOWN_CONVERSION_NAMES` / `_apply_conversion`)
# since it is not a p2mem.units function - it is a declared no-op.
_ALLOWED_CONVERSION_FUNCTIONS: Dict[str, Callable] = {
    "us_per_ft_to_m_per_s": units.us_per_ft_to_m_per_s,
    "m_per_s_to_us_per_ft": units.m_per_s_to_us_per_ft,
    "gcc_to_kgm3": units.gcc_to_kgm3,
    "kgm3_to_gcc": units.kgm3_to_gcc,
    "feet_to_meters": units.feet_to_meters,
    "meters_to_feet": units.meters_to_feet,
}

_IDENTITY_CONVERSION = "identity"
_KNOWN_CONVERSION_NAMES = {_IDENTITY_CONVERSION} | set(_ALLOWED_CONVERSION_FUNCTIONS)

# Increment 2.1: a controlled conversion-signature registry. Each non-
# identity entry maps a conversion-function name to the set of raw units
# (lower-cased) it may be declared FROM and the single canonical unit
# (lower-cased) it produces. `load_file_contract_config` uses this to
# reject a contract that pairs a conversion function with an incompatible
# unit (e.g. `gcc_to_kgm3` on a curve whose raw_unit is "API") - a purely
# contract-authoring-time check, independent of what any actual LAS file
# declares.
_CONVERSION_UNIT_SIGNATURES: Dict[str, Tuple[FrozenSet[str], str]] = {
    "us_per_ft_to_m_per_s": (frozenset({"us/ft"}), "m/s"),
    "m_per_s_to_us_per_ft": (frozenset({"m/s"}), "us/ft"),
    "gcc_to_kgm3": (frozenset({"g/cc"}), "kg/m3"),
    "kgm3_to_gcc": (frozenset({"kg/m3"}), "g/cc"),
    "feet_to_meters": (frozenset({"ft"}), "m"),
    "meters_to_feet": (frozenset({"m"}), "ft"),
}

_REQUIRED_FILE_FIELDS = (
    "expected_sha256",
    "expected_well_identifier",
    "expected_las_version",
    "expected_wrap",
    "expected_null_value",
    "expected_curve_count",
    "expected_data_layout",
    "curves",
)
_REQUIRED_CURVE_FIELDS = (
    "ordinal",
    "source_curve_name",
    "raw_unit",
    "raw_description",
    "raw_canonical_name",
    "canonical_name",
    "canonical_unit",
    "required",
    "conversion_function",
)


# ---------------------------------------------------------------------------
# Definition-line parsing (LAS "MNEM.UNIT  VALUE  :DESCRIPTION" convention)
# ---------------------------------------------------------------------------
def _parse_definition_line(line: str) -> DefinitionLine:
    """
    Parse one LAS definition line into (mnemonic, unit, value, description).

    The LAS 2.0 convention is ``MNEM.UNIT<ws>VALUE/NAME<ws>:DESCRIPTION``,
    where UNIT occupies the characters immediately after the '.' up to the
    first whitespace - critically, if the '.' is immediately followed by
    whitespace, the unit field is EMPTY and everything up to the colon
    (which may itself contain internal spaces, e.g. a well name) is the
    value, not a unit-plus-value pair. Getting this rule wrong is exactly
    the kind of silent misparse this module is designed to avoid, since
    several of these files declare well/company names with no unit field.
    """
    if "." not in line:
        raise LasParsingError(
            f"Malformed LAS definition line (no '.' separating mnemonic from unit): {line!r}"
        )
    mnem_part, rest = line.split(".", 1)
    mnemonic = mnem_part.strip()

    if ":" in rest:
        pre_colon, desc_part = rest.split(":", 1)
        description = desc_part.strip()
    else:
        pre_colon = rest
        description = ""

    if pre_colon == "":
        unit, value = "", ""
    elif pre_colon[0].isspace():
        # '.' immediately followed by whitespace => empty unit field;
        # everything before the colon (stripped) is the value.
        unit = ""
        value = pre_colon.strip()
    else:
        parts = pre_colon.split(None, 1)
        unit = parts[0]
        value = parts[1].strip() if len(parts) > 1 else ""

    return DefinitionLine(mnemonic=mnemonic, unit=unit, value=value, description=description)


def _parse_description_ordinal_name(description: str) -> Tuple[Optional[int], Optional[str]]:
    """
    Extract an embedded "<ordinal> <NAME>" pattern from a curve description
    (e.g. "1 DCAV" -> (1, "DCAV")), if present. Returns (None, None) when
    the description does not follow this pattern - callers must not treat
    the absence of this pattern as an error by itself, only as one fewer
    corroborating signal available for resolution.
    """
    m = _DESC_ORDINAL_RE.match(description)
    if not m:
        return None, None
    return int(m.group(1)), m.group(2).strip()


def _find_by_mnemonic(lines: Tuple[DefinitionLine, ...], mnemonic: str) -> Optional[DefinitionLine]:
    for d in lines:
        if d.mnemonic.strip().upper() == mnemonic.upper():
            return d
    return None


def _to_float_or_none(d: Optional[DefinitionLine]) -> Optional[float]:
    if d is None or d.value.strip() == "":
        return None
    try:
        return float(d.value)
    except ValueError:
        return None


# ---------------------------------------------------------------------------
# Header parsing (Version / Well / Curve / Parameter sections only)
# ---------------------------------------------------------------------------
def parse_las_header(path: str) -> LasHeaderInfo:
    """
    Parse a LAS file's Version, Well, Curve, and Parameter sections and
    compute its SHA-256, WITHOUT reading the ~Ascii data section.

    Section boundaries are detected from lines beginning with '~' (after
    stripping leading whitespace); the section kind is the first letter
    following '~', uppercased (V/W/C/P/O/A). Parsing stops as soon as the
    'A' (Ascii data) section marker is reached - original header order is
    preserved exactly as encountered, and nothing after the first comment
    or section marker is reordered or deduplicated.

    Raises
    ------
    LasFileNotFoundError
        If `path` does not exist.
    LasParsingError
        If a definition line cannot be parsed, or no data-section marker
        is found at all.
    """
    p = Path(path)
    if not p.exists():
        raise LasFileNotFoundError(f"LAS file not found: {path}")

    raw_bytes = p.read_bytes()
    sha256 = hashlib.sha256(raw_bytes).hexdigest()
    text = raw_bytes.decode("utf-8", errors="strict")
    lines = text.splitlines()

    version_lines: List[DefinitionLine] = []
    well_lines: List[DefinitionLine] = []
    param_lines: List[DefinitionLine] = []
    curve_def_lines: List[DefinitionLine] = []

    section: Optional[str] = None
    data_start_idx: Optional[int] = None

    for i, line in enumerate(lines):
        stripped = line.strip()
        if stripped.startswith("~"):
            key = stripped[1:2].upper()
            section = key
            if key == "A":
                data_start_idx = i + 1
                break
            continue
        if stripped == "" or stripped.startswith("#"):
            continue
        if section == "V":
            version_lines.append(_parse_definition_line(line))
        elif section == "W":
            well_lines.append(_parse_definition_line(line))
        elif section == "C":
            curve_def_lines.append(_parse_definition_line(line))
        elif section == "P":
            param_lines.append(_parse_definition_line(line))
        # section == "O" (Other) or None: intentionally not modeled;
        # this project's files carry no content there.

    if data_start_idx is None:
        raise LasParsingError(f"{p.name}: no '~A' (Ascii data) section marker found.")

    curve_headers: List[CurveHeaderEntry] = []
    for ordinal, d in enumerate(curve_def_lines):
        desc_ordinal, desc_name = _parse_description_ordinal_name(d.description)
        curve_headers.append(
            CurveHeaderEntry(
                ordinal=ordinal,
                raw_mnemonic=d.mnemonic,
                raw_unit=d.unit,
                raw_api_code=d.value,
                raw_description=d.description,
                description_ordinal=desc_ordinal,
                description_name=desc_name,
            )
        )

    vers_line = _find_by_mnemonic(tuple(version_lines), "VERS")
    wrap_line = _find_by_mnemonic(tuple(version_lines), "WRAP")
    well_line = _find_by_mnemonic(tuple(well_lines), "WELL")
    null_line = _find_by_mnemonic(tuple(well_lines), "NULL")
    strt_line = _find_by_mnemonic(tuple(well_lines), "STRT")
    stop_line = _find_by_mnemonic(tuple(well_lines), "STOP")
    step_line = _find_by_mnemonic(tuple(well_lines), "STEP")

    return LasHeaderInfo(
        source_path=str(p),
        source_filename=p.name,
        sha256=sha256,
        las_version=vers_line.value if vers_line and vers_line.value else None,
        wrap=wrap_line.value if wrap_line and wrap_line.value else None,
        well_name=well_line.value if well_line and well_line.value else None,
        declared_null=_to_float_or_none(null_line),
        declared_strt=_to_float_or_none(strt_line),
        declared_stop=_to_float_or_none(stop_line),
        declared_step=_to_float_or_none(step_line),
        version_section=tuple(version_lines),
        well_section=tuple(well_lines),
        parameter_section=tuple(param_lines),
        curve_headers=tuple(curve_headers),
        data_section_line_offset=data_start_idx,
    )


# ---------------------------------------------------------------------------
# Ascii data-section parsing
# ---------------------------------------------------------------------------
def _read_ascii_data(path: str, header: LasHeaderInfo) -> np.ndarray:
    """
    Parse the ~Ascii data section into a float64 matrix of shape
    (n_samples, n_curves), exactly as written in the file (no NULL
    substitution here - see `_apply_null_sentinel`).

    Raises
    ------
    LasParsingError
        If any data row's column count does not match the number of
        declared curves, if any token cannot be parsed as a number, if
        any token DOES parse but to a non-finite value (a literal "nan"/
        "inf"/"-inf" text - Increment 2.1: this project's NULL policy is
        a single finite sentinel, so a non-finite literal is a structural
        defect, never silently treated as that sentinel), or if no data
        rows are found at all. The error message names the exact line
        number, column, and offending content.
    """
    p = Path(path)
    text = p.read_text(encoding="utf-8")
    lines = text.splitlines()
    n_curves = len(header.curve_headers)

    rows: List[List[float]] = []
    for lineno, line in enumerate(lines[header.data_section_line_offset :], start=header.data_section_line_offset + 1):
        if line.strip() == "":
            continue
        tokens = line.split()
        if len(tokens) != n_curves:
            raise LasParsingError(
                f"{header.source_filename}: line {lineno} has {len(tokens)} data column(s), "
                f"but {n_curves} curve(s) are declared in the ~Curve section "
                f"(curve/data-column width mismatch). Line content: {line!r}"
            )
        row: List[float] = []
        for col_idx, token in enumerate(tokens):
            try:
                value = float(token)
            except ValueError as exc:
                raise LasParsingError(
                    f"{header.source_filename}: malformed numeric value at line {lineno}, "
                    f"column {col_idx}: {token!r} ({exc})"
                ) from exc
            if not math.isfinite(value):
                raise LasParsingError(
                    f"{header.source_filename}: line {lineno}, column {col_idx} contains a literal "
                    f"non-finite numeric token {token!r} (parses to {value!r}). This project's NULL "
                    f"policy is a single finite sentinel value declared in the file's ~Well section - "
                    f"a literal NaN/Inf token is never treated as that sentinel and is rejected here "
                    f"as a structural defect rather than silently substituted or clipped."
                )
            row.append(value)
        rows.append(row)

    if not rows:
        raise LasParsingError(f"{header.source_filename}: no data rows found in the ~Ascii data section.")

    return np.array(rows, dtype=np.float64)


def _apply_null_sentinel(raw_column: np.ndarray, declared_null: Optional[float]) -> np.ndarray:
    """
    Return a copy of `raw_column` with values matching `declared_null`
    (within `_NULL_TOLERANCE`) replaced by NaN. No other value is ever
    replaced, regardless of how extreme it looks - an out-of-range but
    non-null value is a fact about the data to report via `CurveStats`,
    never a value to silently discard.
    """
    out = raw_column.astype(np.float64, copy=True)
    if declared_null is None:
        return out
    mask = np.abs(out - declared_null) <= _NULL_TOLERANCE
    out[mask] = np.nan
    return out


# ---------------------------------------------------------------------------
# Contract-authoring validation helpers (Increment 2.1)
# ---------------------------------------------------------------------------
def _validate_conversion_unit_compatibility(
    conversion_function: str, raw_unit: str, canonical_unit: str, context: str
) -> None:
    """
    Reject, at contract-load time, a `conversion_function` whose declared
    raw/canonical units are incompatible with the controlled conversion-
    signature registry (`_CONVERSION_UNIT_SIGNATURES`). For example, a
    contract must not be able to pair `gcc_to_kgm3` with a raw_unit of
    "API" - the function's registered signature is g/cc -> kg/m3 only.
    """
    if conversion_function == _IDENTITY_CONVERSION:
        if raw_unit.strip().lower() != canonical_unit.strip().lower():
            raise LasContractDefinitionError(
                f"{context}: conversion_function 'identity' requires raw_unit and canonical_unit to "
                f"be the same physical unit; contract declares raw_unit={raw_unit!r}, "
                f"canonical_unit={canonical_unit!r}."
            )
        return

    signature = _CONVERSION_UNIT_SIGNATURES.get(conversion_function)
    if signature is None:
        # Unreachable in practice (caller already validated membership in
        # _KNOWN_CONVERSION_NAMES), but fail loudly rather than silently
        # skipping validation if the registries ever drift apart.
        raise LasContractDefinitionError(
            f"{context}: conversion_function {conversion_function!r} has no registered unit signature."
        )
    expected_raw_units, expected_canonical_unit = signature
    if raw_unit.strip().lower() not in expected_raw_units:
        raise LasContractDefinitionError(
            f"{context}: conversion_function {conversion_function!r} expects raw_unit in "
            f"{sorted(expected_raw_units)}; contract declares raw_unit={raw_unit!r}. A contract must "
            f"not pair a conversion function with an incompatible source unit."
        )
    if canonical_unit.strip().lower() != expected_canonical_unit:
        raise LasContractDefinitionError(
            f"{context}: conversion_function {conversion_function!r} produces canonical_unit "
            f"{expected_canonical_unit!r}; contract declares canonical_unit={canonical_unit!r}."
        )


# ---------------------------------------------------------------------------
# Curve-contract configuration (config/las_curve_contracts.yml)
# ---------------------------------------------------------------------------
def load_file_contract_config(yaml_path: str) -> Dict[str, FileContract]:
    """
    Load and validate `config/las_curve_contracts.yml`, returning a
    mapping of source filename -> `FileContract`.

    Validates, at load time (before any LAS file is touched):
    * the YAML parses and has a top-level `files` mapping;
    * every required file-level and curve-level field is present
      (Increment 2.1 requires several new fields - see
      `_REQUIRED_FILE_FIELDS` / `_REQUIRED_CURVE_FIELDS`);
    * `expected_las_version`, `expected_wrap`, and `expected_data_layout`
      each name something this parser actually implements - never a mode
      it would silently mishandle;
    * `expected_null_value` is numeric and `expected_curve_count` is an
      integer matching the number of curve entries actually listed;
    * every curve ordinal is a non-negative integer, unique within the
      file, and the full ordinal set is contiguous from 0 unless the file
      explicitly sets `ordinal_gaps_justified: true` with a non-empty
      `ordinal_gap_notes`;
    * no two curve entries within the same file map to the same
      `canonical_name` (a "duplicate canonical target" contract-authoring
      bug);
    * `required` is a genuine boolean (not a truthy string or number);
    * every non-identity `conversion_function` name is one of the exact,
      unit-only conversions this module recognizes, AND is compatible
      with the entry's declared raw_unit/canonical_unit per the
      conversion-signature registry;
    * exactly one curve per file declares `semantic_role: measured_depth`,
      and that curve's `source_curve_name` is "DEPT"/"Depth", its
      `canonical_name` is "MD_m", its `canonical_unit` is "m", and it is
      `required`.

    Raises
    ------
    LasContractDefinitionError
        On any of the above validation failures, or if the file is
        missing / not valid YAML.
    """
    p = Path(yaml_path)
    if not p.exists():
        raise LasContractDefinitionError(f"Curve contract file not found: {yaml_path}")

    try:
        with p.open("r", encoding="utf-8") as f:
            raw = yaml.safe_load(f)
    except yaml.YAMLError as exc:
        raise LasContractDefinitionError(f"{yaml_path}: not valid YAML ({exc})") from exc

    if not raw or "files" not in raw or not isinstance(raw["files"], dict):
        raise LasContractDefinitionError(f"{yaml_path}: expected a top-level 'files' mapping.")

    contracts: Dict[str, FileContract] = {}
    for filename, file_def in raw["files"].items():
        if not isinstance(file_def, dict):
            raise LasContractDefinitionError(f"{yaml_path}: entry for {filename!r} must be a mapping.")

        missing_file_fields = [field for field in _REQUIRED_FILE_FIELDS if field not in file_def]
        if missing_file_fields:
            raise LasContractDefinitionError(
                f"{yaml_path}: file {filename!r} is missing required field(s): {missing_file_fields}."
            )

        expected_las_version = str(file_def["expected_las_version"])
        if expected_las_version not in _SUPPORTED_LAS_VERSIONS:
            raise LasContractDefinitionError(
                f"{yaml_path}: file {filename!r} declares expected_las_version={expected_las_version!r}, "
                f"which this parser does not implement (supported: {sorted(_SUPPORTED_LAS_VERSIONS)}). "
                f"Refusing to pretend support for an unimplemented LAS version."
            )
        expected_wrap = str(file_def["expected_wrap"])
        if expected_wrap not in _SUPPORTED_WRAP_VALUES:
            raise LasContractDefinitionError(
                f"{yaml_path}: file {filename!r} declares expected_wrap={expected_wrap!r}, which this "
                f"parser does not implement (supported: {sorted(_SUPPORTED_WRAP_VALUES)} - unwrapped "
                f"LAS only). Refusing to pretend support for wrapped LAS parsing."
            )
        expected_data_layout = str(file_def["expected_data_layout"])
        if expected_data_layout not in _SUPPORTED_DATA_LAYOUTS:
            raise LasContractDefinitionError(
                f"{yaml_path}: file {filename!r} declares expected_data_layout={expected_data_layout!r}, "
                f"which this parser does not implement (supported: {sorted(_SUPPORTED_DATA_LAYOUTS)})."
            )

        expected_null_value = file_def["expected_null_value"]
        if isinstance(expected_null_value, bool) or not isinstance(expected_null_value, (int, float)):
            raise LasContractDefinitionError(
                f"{yaml_path}: file {filename!r} has a non-numeric expected_null_value="
                f"{expected_null_value!r}."
            )

        expected_curve_count = file_def["expected_curve_count"]
        if isinstance(expected_curve_count, bool) or not isinstance(expected_curve_count, int):
            raise LasContractDefinitionError(
                f"{yaml_path}: file {filename!r} has a non-integer expected_curve_count="
                f"{expected_curve_count!r}."
            )

        curve_defs = file_def.get("curves")
        if not isinstance(curve_defs, list) or not curve_defs:
            raise LasContractDefinitionError(
                f"{yaml_path}: file {filename!r} has an empty or missing 'curves' list."
            )
        if len(curve_defs) != expected_curve_count:
            raise LasContractDefinitionError(
                f"{yaml_path}: file {filename!r} declares expected_curve_count={expected_curve_count} "
                f"but lists {len(curve_defs)} curve entries - the contract is internally inconsistent."
            )

        curves: List[CurveContractEntry] = []
        seen_canonical: set = set()
        seen_ordinals: set = set()
        measured_depth_count = 0

        for c in curve_defs:
            if not isinstance(c, dict):
                raise LasContractDefinitionError(
                    f"{yaml_path}: file {filename!r} has a curve entry that is not a mapping."
                )

            missing = [field for field in _REQUIRED_CURVE_FIELDS if field not in c]
            if missing:
                raise LasContractDefinitionError(
                    f"{yaml_path}: file {filename!r} has a curve entry missing required field(s): {missing}."
                )

            ordinal = c["ordinal"]
            if isinstance(ordinal, bool) or not isinstance(ordinal, int):
                raise LasContractDefinitionError(
                    f"{yaml_path}: file {filename!r} has a non-integer ordinal {ordinal!r}."
                )
            if ordinal < 0:
                raise LasContractDefinitionError(
                    f"{yaml_path}: file {filename!r} has a negative ordinal {ordinal!r}, which is not "
                    f"a valid column position."
                )
            if ordinal in seen_ordinals:
                raise LasContractDefinitionError(
                    f"{yaml_path}: file {filename!r} declares ordinal {ordinal} more than once "
                    f"(duplicate ordinal)."
                )
            seen_ordinals.add(ordinal)

            canonical_name = c["canonical_name"]
            if canonical_name in seen_canonical:
                raise LasContractDefinitionError(
                    f"{yaml_path}: file {filename!r} maps more than one curve to canonical name "
                    f"{canonical_name!r} (duplicate canonical target)."
                )
            seen_canonical.add(canonical_name)

            required_flag = c["required"]
            if not isinstance(required_flag, bool):
                raise LasContractDefinitionError(
                    f"{yaml_path}: file {filename!r} curve {canonical_name!r} has a non-boolean "
                    f"'required' value {required_flag!r}."
                )

            conv_name = c["conversion_function"]
            if conv_name not in _KNOWN_CONVERSION_NAMES:
                raise LasContractDefinitionError(
                    f"{yaml_path}: file {filename!r} curve {canonical_name!r} names unknown "
                    f"conversion_function {conv_name!r}. Allowed: {sorted(_KNOWN_CONVERSION_NAMES)}."
                )

            raw_unit = str(c["raw_unit"])
            canonical_unit = str(c["canonical_unit"])
            _validate_conversion_unit_compatibility(
                conv_name, raw_unit, canonical_unit, context=f"{yaml_path}: {filename}:{canonical_name}"
            )

            semantic_role = c.get("semantic_role", SEMANTIC_ROLE_MEASUREMENT)
            if semantic_role not in (SEMANTIC_ROLE_MEASURED_DEPTH, SEMANTIC_ROLE_MEASUREMENT):
                raise LasContractDefinitionError(
                    f"{yaml_path}: file {filename!r} curve {canonical_name!r} has unknown "
                    f"semantic_role {semantic_role!r}."
                )
            if semantic_role == SEMANTIC_ROLE_MEASURED_DEPTH:
                measured_depth_count += 1

            curves.append(
                CurveContractEntry(
                    ordinal=ordinal,
                    semantic_role=semantic_role,
                    source_curve_name=str(c["source_curve_name"]),
                    raw_mnemonic=str(c.get("raw_mnemonic", "")),
                    raw_unit=raw_unit,
                    raw_description=str(c["raw_description"]),
                    description_ordinal=c.get("description_ordinal"),
                    description_name=c.get("description_name"),
                    raw_canonical_name=str(c["raw_canonical_name"]),
                    canonical_name=canonical_name,
                    canonical_unit=canonical_unit,
                    required=required_flag,
                    conversion_function=conv_name,
                    notes=str(c.get("notes", "")),
                )
            )

        # Ordinal contiguity: unless explicitly justified, ordinals must be
        # exactly {0, 1, ..., expected_curve_count - 1}. A gap most often
        # means a contract-authoring mistake (a curve entry skipped or
        # mis-numbered), not a deliberate design, so it is rejected unless
        # the file explicitly says otherwise and explains why.
        expected_ordinal_set = set(range(expected_curve_count))
        if seen_ordinals != expected_ordinal_set:
            if not bool(file_def.get("ordinal_gaps_justified", False)):
                raise LasContractDefinitionError(
                    f"{yaml_path}: file {filename!r} has ordinals {sorted(seen_ordinals)}, which are "
                    f"not contiguous from 0 (expected {sorted(expected_ordinal_set)}). If this is "
                    f"deliberate, set 'ordinal_gaps_justified: true' and explain why in "
                    f"'ordinal_gap_notes'."
                )
            if not str(file_def.get("ordinal_gap_notes", "")).strip():
                raise LasContractDefinitionError(
                    f"{yaml_path}: file {filename!r} sets ordinal_gaps_justified but has no "
                    f"'ordinal_gap_notes' explaining the gap."
                )

        if measured_depth_count == 0:
            raise LasContractDefinitionError(
                f"{yaml_path}: file {filename!r} declares no curve with "
                f"semantic_role='{SEMANTIC_ROLE_MEASURED_DEPTH}'. Exactly one curve must be declared "
                f"as the measured-depth role; it is never inferred from a canonical name."
            )
        if measured_depth_count > 1:
            raise LasContractDefinitionError(
                f"{yaml_path}: file {filename!r} declares {measured_depth_count} curves with "
                f"semantic_role='{SEMANTIC_ROLE_MEASURED_DEPTH}'; exactly one is required."
            )

        depth_entry = next(c for c in curves if c.semantic_role == SEMANTIC_ROLE_MEASURED_DEPTH)
        if depth_entry.source_curve_name.strip().upper() not in ("DEPT", "DEPTH"):
            raise LasContractDefinitionError(
                f"{yaml_path}: file {filename!r}'s measured-depth curve has source_curve_name="
                f"{depth_entry.source_curve_name!r}; expected 'DEPT' or 'Depth'."
            )
        if depth_entry.canonical_name != "MD_m":
            raise LasContractDefinitionError(
                f"{yaml_path}: file {filename!r}'s measured-depth curve must have canonical_name "
                f"'MD_m'; found {depth_entry.canonical_name!r}."
            )
        if depth_entry.canonical_unit.strip().lower() != "m":
            raise LasContractDefinitionError(
                f"{yaml_path}: file {filename!r}'s measured-depth curve must have canonical_unit "
                f"'m' (metres); found {depth_entry.canonical_unit!r}."
            )
        if not depth_entry.required:
            raise LasContractDefinitionError(
                f"{yaml_path}: file {filename!r}'s measured-depth curve must be required=true."
            )

        contracts[filename] = FileContract(
            source_filename=filename,
            expected_sha256=str(file_def["expected_sha256"]),
            expected_well_identifier=str(file_def["expected_well_identifier"]),
            expected_las_version=expected_las_version,
            expected_wrap=expected_wrap,
            expected_null_value=float(expected_null_value),
            expected_curve_count=int(expected_curve_count),
            expected_data_layout=expected_data_layout,
            curves=tuple(curves),
        )

    return contracts


# ---------------------------------------------------------------------------
# Contract resolution
# ---------------------------------------------------------------------------
def resolve_curve_contract(
    header: LasHeaderInfo, contract: FileContract
) -> Tuple[Tuple[CurveResolution, ...], Tuple[IngestionIssue, ...], str]:
    """
    Attempt to resolve every curve in `contract` against the curves
    actually present in `header`, WITHOUT reading any numeric data.

    Increment 2.1 adds a block of FILE-IDENTITY checks, run before any
    curve is resolved, that an independent audit found Increment 2 did
    not perform (the resolver previously "passed" a file with a missing
    WELL, a missing NULL, a changed VERS/WRAP, or a contract for an
    entirely different file than the one being loaded): the loaded file's
    basename must match `contract.source_filename`; its SHA-256 must
    match `contract.expected_sha256`; it must declare a WELL value
    matching `contract.expected_well_identifier`; it must declare a
    supported, contract-matching VERS and WRAP; and it must declare a
    NULL value matching `contract.expected_null_value` (a NULL mismatch
    is an ERROR, not a warning, because it changes which samples are
    substituted to NaN). Every one of these is blocking.

    Curve-level resolution (unchanged from Increment 2): a contract entry
    resolves only when the curve found at its declared `ordinal` has both
    the expected `raw_unit` AND (when the header curve carries an
    embedded description ordinal/name) matching `description_ordinal`/
    `description_name`, AND, when the contract declares a non-empty
    `raw_mnemonic`, a matching mnemonic. Ordinal position is never, by
    itself, treated as sufficient.

    Also performs file-level checks independent of any single curve:
    duplicate raw curve descriptions (which would make description-based
    disambiguation unsafe) and the file's declared curve count against
    the contract's `expected_curve_count`.

    Returns
    -------
    (resolutions, issues, status) where `status` is "PASSED" if no
    ERROR-severity issue or FAILED resolution was found, else "FAILED".
    This function never raises for a data-content problem (only
    `load_las_file` raises, after also considering the numeric data) -
    it is used standalone by the "validate contract before loading"
    notebook step.
    """
    issues: List[IngestionIssue] = []
    resolutions: List[CurveResolution] = []

    # --- File-identity checks (Increment 2.1 correction) ---------------
    if header.source_filename != contract.source_filename:
        issues.append(
            IngestionIssue(
                severity="ERROR",
                code="FILENAME_MISMATCH",
                message=(
                    f"File being loaded is named {header.source_filename!r}, but this contract is "
                    f"declared for {contract.source_filename!r}. Refusing to apply a contract "
                    f"authored for a different file."
                ),
                context=header.source_filename,
            )
        )

    if header.sha256 != contract.expected_sha256:
        issues.append(
            IngestionIssue(
                severity="ERROR",
                code="SHA256_MISMATCH",
                message=(
                    f"File SHA-256 is {header.sha256!r}; contract expects "
                    f"{contract.expected_sha256!r}. The file's content does not match what this "
                    f"contract was authored against."
                ),
                context=header.source_filename,
            )
        )

    if header.well_name is None:
        issues.append(
            IngestionIssue(
                severity="ERROR",
                code="WELL_MISSING",
                message="File declares no WELL value in its ~Well section; cannot verify well identity.",
                context=header.source_filename,
            )
        )
    elif contract.expected_well_identifier.strip() != header.well_name.strip():
        issues.append(
            IngestionIssue(
                severity="ERROR",
                code="WELL_IDENTIFIER_MISMATCH",
                message=(
                    f"Contract expects WELL={contract.expected_well_identifier!r}; file declares "
                    f"WELL={header.well_name!r}."
                ),
                context=header.source_filename,
            )
        )

    if header.las_version is None:
        issues.append(
            IngestionIssue(
                severity="ERROR",
                code="VERS_MISSING",
                message="File declares no VERS value in its ~Version section; cannot verify LAS version.",
                context=header.source_filename,
            )
        )
    elif header.las_version not in _SUPPORTED_LAS_VERSIONS:
        issues.append(
            IngestionIssue(
                severity="ERROR",
                code="VERS_UNSUPPORTED",
                message=(
                    f"File declares VERS={header.las_version!r}, which this parser does not "
                    f"implement (supported: {sorted(_SUPPORTED_LAS_VERSIONS)})."
                ),
                context=header.source_filename,
            )
        )
    elif header.las_version != contract.expected_las_version:
        issues.append(
            IngestionIssue(
                severity="ERROR",
                code="VERS_MISMATCH",
                message=(
                    f"Contract expects VERS={contract.expected_las_version!r}; file declares "
                    f"VERS={header.las_version!r}."
                ),
                context=header.source_filename,
            )
        )

    if header.wrap is None:
        issues.append(
            IngestionIssue(
                severity="ERROR",
                code="WRAP_MISSING",
                message="File declares no WRAP value in its ~Version section; cannot verify data layout.",
                context=header.source_filename,
            )
        )
    elif header.wrap not in _SUPPORTED_WRAP_VALUES:
        issues.append(
            IngestionIssue(
                severity="ERROR",
                code="WRAP_UNSUPPORTED",
                message=(
                    f"File declares WRAP={header.wrap!r}. This parser implements unwrapped "
                    f"(WRAP=NO), whitespace-delimited LAS data only; wrapped or other layouts are "
                    f"explicitly rejected rather than silently misread."
                ),
                context=header.source_filename,
            )
        )
    elif header.wrap != contract.expected_wrap:
        issues.append(
            IngestionIssue(
                severity="ERROR",
                code="WRAP_MISMATCH",
                message=(
                    f"Contract expects WRAP={contract.expected_wrap!r}; file declares "
                    f"WRAP={header.wrap!r}."
                ),
                context=header.source_filename,
            )
        )

    if header.declared_null is None:
        issues.append(
            IngestionIssue(
                severity="ERROR",
                code="NULL_MISSING",
                message="File declares no NULL value in its ~Well section; NULL-sentinel substitution cannot be safely performed.",
                context=header.source_filename,
            )
        )
    elif abs(contract.expected_null_value - header.declared_null) > _NULL_TOLERANCE:
        issues.append(
            IngestionIssue(
                severity="ERROR",
                code="NULL_MISMATCH",
                message=(
                    f"Contract expects NULL={contract.expected_null_value}; file declares "
                    f"NULL={header.declared_null}. A NULL-value mismatch changes which samples are "
                    f"substituted to NaN and is treated as blocking, not advisory."
                ),
                context=header.source_filename,
            )
        )

    # --- File-level structural checks (unchanged from Increment 2) -----
    desc_positions: Dict[str, List[int]] = {}
    for ch in header.curve_headers:
        key = ch.raw_description.strip()
        if key == "":
            continue
        desc_positions.setdefault(key, []).append(ch.ordinal)
    for desc, ordinals in desc_positions.items():
        if len(ordinals) > 1:
            issues.append(
                IngestionIssue(
                    severity="ERROR",
                    code="DUPLICATE_RAW_DESCRIPTION",
                    message=(
                        f"Curve description {desc!r} appears at ordinals {ordinals}; "
                        f"curve identity cannot be safely disambiguated by description."
                    ),
                    context=header.source_filename,
                )
            )

    if contract.expected_curve_count != len(header.curve_headers):
        issues.append(
            IngestionIssue(
                severity="ERROR",
                code="CURVE_COUNT_MISMATCH",
                message=(
                    f"Contract expects {contract.expected_curve_count} curves; the file's "
                    f"~Curve section declares {len(header.curve_headers)}."
                ),
                context=header.source_filename,
            )
        )

    # --- Curve-level resolution (unchanged logic; names updated) -------
    by_ordinal = {ch.ordinal: ch for ch in header.curve_headers}

    for entry in contract.curves:
        header_entry = by_ordinal.get(entry.ordinal)

        if header_entry is None:
            if entry.required:
                issues.append(
                    IngestionIssue(
                        severity="ERROR",
                        code="CURVE_NOT_FOUND",
                        message=(
                            f"No curve found at ordinal {entry.ordinal} for required curve "
                            f"{entry.source_curve_name!r} (canonical {entry.canonical_name!r})."
                        ),
                        context=header.source_filename,
                    )
                )
                status = "FAILED"
            else:
                status = "MISSING_OPTIONAL"
            resolutions.append(
                CurveResolution(
                    canonical_name=entry.canonical_name,
                    source_curve_name=entry.source_curve_name,
                    status=status,
                    ordinal=entry.ordinal,
                    matched_raw_mnemonic=None,
                    matched_raw_unit=None,
                    matched_raw_description=None,
                    conversion_applied=None,
                    detail="No header entry found at the contract's declared ordinal.",
                )
            )
            continue

        mismatches: List[str] = []

        if entry.raw_mnemonic.strip() != "" and header_entry.raw_mnemonic.strip() != entry.raw_mnemonic.strip():
            mismatches.append(
                f"mnemonic: contract expects {entry.raw_mnemonic!r}, file has {header_entry.raw_mnemonic!r}"
            )

        if header_entry.raw_unit.strip() != entry.raw_unit.strip():
            mismatches.append(f"unit: contract expects {entry.raw_unit!r}, file has {header_entry.raw_unit!r}")

        if entry.description_ordinal is not None and header_entry.description_ordinal != entry.description_ordinal:
            mismatches.append(
                f"description ordinal: contract expects {entry.description_ordinal!r}, "
                f"file has {header_entry.description_ordinal!r}"
            )

        if entry.description_name is not None:
            found_name = (header_entry.description_name or "").upper()
            if found_name != entry.description_name.upper():
                mismatches.append(
                    f"description name: contract expects {entry.description_name!r}, "
                    f"file has {header_entry.description_name!r}"
                )

        # File-internal self-consistency: an empty-mnemonic curve's own
        # embedded description ordinal (when present) must match its
        # physical column position. This check exists so that ordinal
        # position is never the ONLY thing standing behind a resolution -
        # the file's own description must independently agree with where
        # it physically sits.
        if header_entry.description_ordinal is not None and header_entry.description_ordinal != header_entry.ordinal:
            mismatches.append(
                f"file-internal inconsistency: description ordinal {header_entry.description_ordinal} "
                f"does not match this curve's physical column position {header_entry.ordinal}"
            )

        if mismatches:
            issues.append(
                IngestionIssue(
                    severity="ERROR",
                    code="CURVE_IDENTITY_MISMATCH",
                    message=(
                        f"{entry.source_curve_name} / {entry.canonical_name} (ordinal {entry.ordinal}): "
                        + "; ".join(mismatches)
                    ),
                    context=header.source_filename,
                )
            )
            resolutions.append(
                CurveResolution(
                    canonical_name=entry.canonical_name,
                    source_curve_name=entry.source_curve_name,
                    status="FAILED",
                    ordinal=entry.ordinal,
                    matched_raw_mnemonic=None,
                    matched_raw_unit=None,
                    matched_raw_description=None,
                    conversion_applied=None,
                    detail="; ".join(mismatches),
                )
            )
        else:
            agreed = ["ordinal", "unit"]
            if entry.description_ordinal is not None:
                agreed.append("description ordinal")
            if entry.description_name is not None:
                agreed.append("description name")
            if entry.raw_mnemonic.strip():
                agreed.append("mnemonic")
            resolutions.append(
                CurveResolution(
                    canonical_name=entry.canonical_name,
                    source_curve_name=entry.source_curve_name,
                    status="RESOLVED",
                    ordinal=entry.ordinal,
                    matched_raw_mnemonic=header_entry.raw_mnemonic,
                    matched_raw_unit=header_entry.raw_unit,
                    matched_raw_description=header_entry.raw_description,
                    conversion_applied=entry.conversion_function,
                    detail="Agreement confirmed on: " + ", ".join(agreed) + ".",
                )
            )

    has_error = any(i.severity == "ERROR" for i in issues) or any(r.status == "FAILED" for r in resolutions)
    status = "FAILED" if has_error else "PASSED"
    return tuple(resolutions), tuple(issues), status


# ---------------------------------------------------------------------------
# Depth and curve diagnostics
# ---------------------------------------------------------------------------
def compute_depth_diagnostics(
    md: np.ndarray,
    declared_strt: Optional[float],
    declared_stop: Optional[float],
    declared_step: Optional[float],
) -> DepthDiagnostics:
    """
    Compute data-derived depth-index diagnostics and compare them against
    the file's declared STRT/STOP/STEP header values.

    `md` must be the raw (NOT NULL-substituted) measured-depth column -
    depth is an index, never subject to NULL-sentinel treatment, and (see
    `load_las_file`) is required to be entirely finite before this
    function is called. Duplicate and non-monotonic samples are DETECTED
    and reported here; per the Increment 2 specification this function
    never raises for them (they are not on the "reject" list) - only
    `resolve_curve_contract` / `load_las_file`'s ERROR-severity issues
    stop ingestion.
    """
    md = np.asarray(md, dtype=np.float64)
    diffs = np.diff(md)
    n_duplicate = int(np.sum(diffs == 0.0))
    n_non_monotonic = int(np.sum(diffs < 0.0))

    data_start = float(md[0]) if md.size else float("nan")
    data_stop = float(md[-1]) if md.size else float("nan")
    data_min_step = float(np.min(diffs)) if diffs.size else float("nan")
    data_max_step = float(np.max(diffs)) if diffs.size else float("nan")
    data_median_step = float(np.median(diffs)) if diffs.size else float("nan")

    def _close(declared: Optional[float], derived: float, tol: float) -> Optional[bool]:
        if declared is None:
            return None
        return bool(abs(declared - derived) <= tol)

    return DepthDiagnostics(
        declared_strt=declared_strt,
        declared_stop=declared_stop,
        declared_step=declared_step,
        data_start=data_start,
        data_stop=data_stop,
        data_min_step=data_min_step,
        data_max_step=data_max_step,
        data_median_step=data_median_step,
        n_samples=int(md.size),
        n_duplicate_md=n_duplicate,
        n_non_monotonic=n_non_monotonic,
        strt_matches_declared=_close(declared_strt, data_start, _DEPTH_ENDPOINT_TOLERANCE_M),
        stop_matches_declared=_close(declared_stop, data_stop, _DEPTH_ENDPOINT_TOLERANCE_M),
        step_matches_declared=_close(declared_step, data_median_step, _DEPTH_STEP_TOLERANCE_M),
    )


def compute_curve_stats(
    entry: CurveContractEntry,
    raw_column: np.ndarray,
    canonical_column: np.ndarray,
    declared_null: Optional[float],
) -> CurveStats:
    """
    Compute descriptive (non-judgemental), unit-explicit statistics for
    one resolved curve.

    `raw_column` is the literal parsed column (from `LasFileResult.raw_data`
    at `entry.ordinal`), still containing the NULL sentinel wherever it
    occurs. `canonical_column` is the corresponding
    `LasFileResult.canonical_data[entry.canonical_name]` array (NULL-
    substituted to NaN, and exactly unit-converted if
    `entry.conversion_function != "identity"`).

    Increment 2.1 correction: Increment 2 computed a single min/max pair
    from the NULL-substituted-but-NOT-converted array and reported it
    under the canonical (post-conversion-implying) name with no unit
    field at all - so, for example, DTCO's coverage row showed numbers
    that were still in us/ft while being labeled with a name that implied
    m/s. This function now reports RAW and CANONICAL statistics
    separately, each tagged with its own explicit unit, so no number can
    be misread as being in the wrong unit.
    """
    raw_column = np.asarray(raw_column, dtype=np.float64)
    canonical_column = np.asarray(canonical_column, dtype=np.float64)
    n = int(raw_column.size)

    if declared_null is not None:
        null_mask = np.abs(raw_column - declared_null) <= _NULL_TOLERANCE
    else:
        null_mask = np.zeros(n, dtype=bool)
    valid_mask = ~null_mask
    valid_count = int(np.sum(valid_mask))
    null_count = n - valid_count
    valid_fraction = (valid_count / n) if n > 0 else 0.0

    if valid_count > 0:
        raw_min: Optional[float] = float(np.min(raw_column[valid_mask]))
        raw_max: Optional[float] = float(np.max(raw_column[valid_mask]))
        canonical_min: Optional[float] = float(np.nanmin(canonical_column))
        canonical_max: Optional[float] = float(np.nanmax(canonical_column))
    else:
        raw_min = raw_max = canonical_min = canonical_max = None

    display_mnemonic = entry.raw_mnemonic.strip() or f"(empty mnemonic, ordinal {entry.ordinal})"

    return CurveStats(
        source_curve_name=entry.source_curve_name,
        raw_mnemonic=display_mnemonic,
        raw_description=entry.raw_description,
        raw_unit=entry.raw_unit,
        canonical_name=entry.canonical_name,
        canonical_unit=entry.canonical_unit,
        conversion_function=entry.conversion_function,
        n_samples=n,
        valid_count=valid_count,
        null_count=null_count,
        valid_fraction=valid_fraction,
        raw_min=raw_min,
        raw_max=raw_max,
        canonical_min=canonical_min,
        canonical_max=canonical_max,
        statistics_basis=(
            f"valid_count/null_count/valid_fraction and min/max are computed over samples whose raw "
            f"value does NOT exactly match the file's declared NULL sentinel (tolerance "
            f"{_NULL_TOLERANCE:g}). raw_min/raw_max are in raw_unit, computed directly from the "
            f"unconverted raw column. canonical_min/canonical_max are in canonical_unit, computed "
            f"from the NULL-substituted array after any declared exact conversion "
            f"(conversion_function={entry.conversion_function!r})."
        ),
    )


def _apply_conversion(
    entry: CurveContractEntry,
    substituted: np.ndarray,
    source_filename: str,
    md_raw: Optional[np.ndarray],
) -> np.ndarray:
    """
    Apply `entry.conversion_function` to a NULL-substituted column,
    raising `LasConversionError` with full project context rather than
    letting a raw NumPy/`p2mem.units` exception escape, and rather than
    silently accepting a non-finite result for a non-null input value.
    """
    if entry.conversion_function == _IDENTITY_CONVERSION:
        return substituted

    conv_fn = _ALLOWED_CONVERSION_FUNCTIONS[entry.conversion_function]
    try:
        canonical = conv_fn(substituted)
    except Exception as exc:
        raise LasConversionError(
            f"{source_filename}: applying conversion_function {entry.conversion_function!r} to curve "
            f"{entry.source_curve_name!r} (canonical {entry.canonical_name!r}, expected canonical "
            f"unit {entry.canonical_unit!r}) raised {type(exc).__name__}: {exc}"
        ) from exc

    canonical = np.asarray(canonical, dtype=np.float64)
    # A non-null physical value must never silently convert to a
    # non-finite result (e.g. a zero sonic slowness -> infinite velocity).
    bad_mask = np.isinf(canonical) & ~np.isnan(substituted)
    if np.any(bad_mask):
        bad_idx = int(np.argmax(bad_mask))
        offending_raw = float(substituted[bad_idx])
        depth_note = ""
        if md_raw is not None and bad_idx < md_raw.size:
            depth_note = f" at row {bad_idx} (MD={float(md_raw[bad_idx])!r})"
        raise LasConversionError(
            f"{source_filename}: converting curve {entry.source_curve_name!r} (canonical "
            f"{entry.canonical_name!r}) via {entry.conversion_function!r} produced a non-finite "
            f"result{depth_note} from raw value {offending_raw!r} (expected canonical unit "
            f"{entry.canonical_unit!r}). This is not a NULL-sentinel row - the raw value is a real, "
            f"non-null measurement that the exact conversion cannot represent safely."
        )
    return canonical


# ---------------------------------------------------------------------------
# Top-level loader
# ---------------------------------------------------------------------------
def load_las_file(path: str, contract: FileContract) -> LasFileResult:
    """
    Fully load and contract-resolve one LAS file.

    Sequence: parse header -> resolve contract against header, including
    the Increment 2.1 file-identity checks (no data read yet) -> parse
    ~Ascii data -> cross-check data column count -> if any ERROR-severity
    issue exists at this point, raise `LasContractError` (ingestion stops;
    no data is returned) -> locate the contract's declared measured-depth
    curve (by `semantic_role`, never by searching for a canonical name
    "DEPT") and require its measured-depth column to be entirely finite
    -> compute depth diagnostics -> for every RESOLVED curve, apply NULL-
    sentinel substitution and, if the contract declares one, an exact
    `p2mem.units` conversion (raising `LasConversionError` with full
    context if that conversion cannot be applied safely), producing
    `canonical_data` - the raw column is separately preserved unchanged
    in `raw_data`.

    Raises
    ------
    LasFileNotFoundError, LasParsingError
        For file-access or structural parsing problems (see
        `parse_las_header` / `_read_ascii_data`).
    LasContractError
        If the file's header cannot be safely reconciled with `contract`
        (a file-identity mismatch, a required curve failed to resolve, a
        duplicate/ambiguous description was found, or the data section's
        column count does not match the contract).
    LasConversionError
        If a contract-declared exact unit conversion cannot be safely
        applied to a resolved curve's values.
    """
    header = parse_las_header(path)
    resolutions, issues, status = resolve_curve_contract(header, contract)

    raw = _read_ascii_data(path, header)

    issues_list = list(issues)
    if raw.shape[1] != contract.expected_curve_count:
        issues_list.append(
            IngestionIssue(
                severity="ERROR",
                code="DATA_COLUMN_COUNT_MISMATCH",
                message=(
                    f"~Ascii data section has {raw.shape[1]} column(s); contract expects "
                    f"{contract.expected_curve_count}."
                ),
                context=header.source_filename,
            )
        )
        status = "FAILED"

    error_issues = [i for i in issues_list if i.severity == "ERROR"]
    if error_issues:
        msg = "; ".join(f"[{i.code}] {i.message}" for i in error_issues)
        exc = LasContractError(f"{header.source_filename}: contract resolution FAILED - {msg}")
        exc.issues = tuple(issues_list)
        exc.resolutions = tuple(resolutions)
        exc.header = header
        raise exc

    # The measured-depth curve is found by its declared semantic role,
    # never by matching a canonical name spelled "DEPT" (Increment 2.1
    # correction). `load_file_contract_config` guarantees exactly one
    # such entry exists in any contract it returns; the check below is
    # defensive, for a contract constructed directly in Python (e.g. in a
    # test) rather than loaded from YAML.
    depth_entry = next(
        (c for c in contract.curves if c.semantic_role == SEMANTIC_ROLE_MEASURED_DEPTH), None
    )
    if depth_entry is None:
        exc = LasContractError(f"{header.source_filename}: contract defines no measured-depth curve.")
        exc.issues = tuple(issues_list)
        exc.resolutions = tuple(resolutions)
        exc.header = header
        raise exc

    depth_resolution = next(r for r in resolutions if r.canonical_name == depth_entry.canonical_name)
    if depth_resolution.status != "RESOLVED":
        exc = LasContractError(
            f"{header.source_filename}: the contract's measured-depth curve "
            f"({depth_entry.source_curve_name!r}) did not resolve."
        )
        exc.issues = tuple(issues_list)
        exc.resolutions = tuple(resolutions)
        exc.header = header
        raise exc

    md_raw = raw[:, depth_entry.ordinal]
    if not np.all(np.isfinite(md_raw)):
        exc = LasContractError(
            f"{header.source_filename}: measured-depth column ({depth_entry.source_curve_name!r}) "
            f"contains non-finite value(s); depth must always be finite."
        )
        exc.issues = tuple(issues_list)
        exc.resolutions = tuple(resolutions)
        exc.header = header
        raise exc

    depth = compute_depth_diagnostics(md_raw, header.declared_strt, header.declared_stop, header.declared_step)

    if depth.n_duplicate_md > 0:
        issues_list.append(
            IngestionIssue(
                "WARNING", "DUPLICATE_MD",
                f"{depth.n_duplicate_md} duplicate measured-depth sample(s) detected.",
                header.source_filename,
            )
        )
    if depth.n_non_monotonic > 0:
        issues_list.append(
            IngestionIssue(
                "WARNING", "NON_MONOTONIC_MD",
                f"{depth.n_non_monotonic} non-monotonic measured-depth step(s) detected.",
                header.source_filename,
            )
        )
    if depth.strt_matches_declared is False:
        issues_list.append(
            IngestionIssue(
                "WARNING", "STRT_MISMATCH",
                f"Declared STRT={depth.declared_strt} vs data-derived start={depth.data_start}.",
                header.source_filename,
            )
        )
    if depth.stop_matches_declared is False:
        issues_list.append(
            IngestionIssue(
                "WARNING", "STOP_MISMATCH",
                f"Declared STOP={depth.declared_stop} vs data-derived stop={depth.data_stop}.",
                header.source_filename,
            )
        )
    if depth.step_matches_declared is False:
        issues_list.append(
            IngestionIssue(
                "WARNING", "STEP_MISMATCH",
                f"Declared STEP={depth.declared_step} vs data-derived median step={depth.data_median_step}.",
                header.source_filename,
            )
        )

    resolutions_by_name = {r.canonical_name: r for r in resolutions}
    canonical_data: Dict[str, np.ndarray] = {}
    curve_stats: List[CurveStats] = []

    for entry in contract.curves:
        res = resolutions_by_name[entry.canonical_name]
        if res.status != "RESOLVED":
            continue
        col = raw[:, entry.ordinal]
        substituted = _apply_null_sentinel(col, header.declared_null)
        canonical_values = _apply_conversion(entry, substituted, header.source_filename, md_raw)
        canonical_data[entry.canonical_name] = canonical_values
        curve_stats.append(compute_curve_stats(entry, col, canonical_values, header.declared_null))

    return LasFileResult(
        header=header,
        contract=contract,
        resolutions=resolutions,
        depth=depth,
        curve_stats=tuple(curve_stats),
        raw_data=raw,
        canonical_data=canonical_data,
        issues=tuple(issues_list),
        contract_status="PASSED",
    )


def load_wells(
    file_paths: Dict[str, str], contracts: Dict[str, FileContract]
) -> Tuple[Dict[str, LasFileResult], Dict[str, IngestionFailure]]:
    """
    Load several LAS files against their respective contracts.

    `file_paths` maps an arbitrary caller-chosen key (e.g. a canonical
    well name) to a filesystem path; the matching contract is looked up
    by the file's basename. One well's ingestion failure does not stop
    the others from loading - each key ends up in exactly one of the two
    returned dicts, never both.

    Increment 2.1 correction: a failed well is now recorded as a typed
    `IngestionFailure` (with `error_type` one of "file_not_found",
    "parsing_failure", "contract_failure", "conversion_failure") rather
    than a bare caught exception, so a caller can branch on the kind of
    failure without importing every exception class or parsing message
    text. All four of those are EXPECTED per-file ingestion-failure
    modes and are isolated here; any other exception (a programming
    error, not an expected data-quality problem) is NOT caught and still
    propagates out of this function.

    Raises
    ------
    LasContractDefinitionError
        If a path's basename has no matching entry in `contracts` - this
        is a configuration problem (contracts and file set out of sync),
        not a per-well data problem, so it is not caught per-well.
    """
    results: Dict[str, LasFileResult] = {}
    errors: Dict[str, IngestionFailure] = {}
    for key, path in file_paths.items():
        filename = Path(path).name
        contract = contracts.get(filename)
        if contract is None:
            raise LasContractDefinitionError(
                f"No curve contract found for {filename!r} (key {key!r}). "
                f"Contracts are defined for: {sorted(contracts)}."
            )
        try:
            results[key] = load_las_file(path, contract)
        except LasFileNotFoundError as exc:
            errors[key] = IngestionFailure(key, path, "file_not_found", str(exc), exc)
        except LasParsingError as exc:
            errors[key] = IngestionFailure(key, path, "parsing_failure", str(exc), exc)
        except LasContractError as exc:
            errors[key] = IngestionFailure(key, path, "contract_failure", str(exc), exc)
        except LasConversionError as exc:
            errors[key] = IngestionFailure(key, path, "conversion_failure", str(exc), exc)
    return results, errors


#### `p2mem/io/inventory.py`

**Technical objective:** turn the typed `LasFileResult`/`IngestionFailure` objects from `p2mem.io.las` into the flat, deterministic-order CSV/JSON tables this increment must produce under `outputs/02_las_inventory/` (Step 13) — metadata and descriptive statistics only, never raw LAS sample data. The curve-coverage table (`las_curve_coverage.csv`) now reports raw and canonical statistics side by side, each with its own explicit unit (the Increment 2.1 correction to this module).

In [ ]:
%%writefile p2mem/io/inventory.py
"""
p2mem.io.inventory - Deterministic inventory-report builders for LAS
ingestion results (Increment 2, corrected in Increment 2.1).

This module turns the typed results from `p2mem.io.las` into the flat,
tabular rows the Increment 2 specification requires under
`outputs/02_las_inventory/`:

    las_file_inventory.csv     - one row per source file
    las_curve_catalog.csv      - one row per contract-declared curve
    las_curve_coverage.csv     - one row per successfully resolved curve
    las_ingestion_issues.csv   - one row per WARNING/ERROR issue found
    las_ingestion_manifest.json - one JSON document tying it all together

No raw LAS sample data is ever written by this module - every function
here emits metadata, provenance, and descriptive statistics only. Row
order is always the deterministic order of the inputs (file iteration
order as given by the caller, then contract-declaration order within a
file), so re-running ingestion against unchanged inputs reproduces
byte-identical output ordering.

Increment 2.1 correction
-------------------------
* `las_curve_coverage.csv` is redesigned (see `build_curve_coverage_rows`)
  to report an explicit RAW/CANONICAL pair of statistics, each tagged
  with its own unit, instead of a single min/max pair labeled only with a
  canonical name and no unit.
* `las.py:load_wells` now returns `IngestionFailure` objects (not bare
  exceptions) for failed wells; every function here that previously took
  `Dict[str, LasContractError]` now takes `Dict[str, IngestionFailure]`
  and reports `error_type`/`error_message` explicitly for every failure
  category (file-not-found and structural-parsing failures included,
  which Increment 2 did not surface in these tables at all since
  `load_wells` let them propagate as uncaught exceptions in most cases).
"""

from __future__ import annotations

from pathlib import Path
from typing import Dict, List, Optional

from p2mem.models import IngestionFailure, LasFileResult


def _failed_well_header_and_issues(failure: IngestionFailure):
    """
    For a "contract_failure", the wrapped `LasContractError` carries a
    successfully-parsed `.header` and the `.issues` collected before
    ingestion stopped - return those. For every other failure category
    ("file_not_found", "parsing_failure", "conversion_failure"), no such
    header/issue list exists (the failure happened before or outside
    per-curve issue tracking), so both are reported as absent rather than
    guessed at.
    """
    if failure.error_type == "contract_failure":
        exc = failure.exception
        return getattr(exc, "header", None), getattr(exc, "issues", ())
    return None, ()


def build_file_inventory_rows(
    results: Dict[str, LasFileResult], errors: Optional[Dict[str, IngestionFailure]] = None
) -> List[dict]:
    """One row per source file: identity, provenance, and header-level facts."""
    rows: List[dict] = []
    for key, result in results.items():
        h = result.header
        rows.append(
            {
                "well_key": key,
                "source_filename": h.source_filename,
                "sha256": h.sha256,
                "well_name": h.well_name,
                "las_version": h.las_version,
                "wrap": h.wrap,
                "declared_null": h.declared_null,
                "declared_strt_m": h.declared_strt,
                "declared_stop_m": h.declared_stop,
                "declared_step_m": h.declared_step,
                "declared_curve_count": len(h.curve_headers),
                "n_samples": result.depth.n_samples,
                "data_start_m": result.depth.data_start,
                "data_stop_m": result.depth.data_stop,
                "data_min_step_m": result.depth.data_min_step,
                "data_max_step_m": result.depth.data_max_step,
                "data_median_step_m": result.depth.data_median_step,
                "n_duplicate_md": result.depth.n_duplicate_md,
                "n_non_monotonic_md": result.depth.n_non_monotonic,
                "strt_matches_declared": result.depth.strt_matches_declared,
                "stop_matches_declared": result.depth.stop_matches_declared,
                "step_matches_declared": result.depth.step_matches_declared,
                "contract_status": result.contract_status,
                "n_warnings": sum(1 for i in result.issues if i.severity == "WARNING"),
                "n_errors": sum(1 for i in result.issues if i.severity == "ERROR"),
                "error_type": None,
                "error_message": None,
            }
        )
    if errors:
        for key, failure in errors.items():
            h, issues = _failed_well_header_and_issues(failure)
            rows.append(
                {
                    "well_key": key,
                    "source_filename": h.source_filename if h else Path(failure.source_path).name,
                    "sha256": h.sha256 if h else None,
                    "well_name": h.well_name if h else None,
                    "las_version": h.las_version if h else None,
                    "wrap": h.wrap if h else None,
                    "declared_null": h.declared_null if h else None,
                    "declared_strt_m": h.declared_strt if h else None,
                    "declared_stop_m": h.declared_stop if h else None,
                    "declared_step_m": h.declared_step if h else None,
                    "declared_curve_count": len(h.curve_headers) if h else None,
                    "n_samples": None,
                    "data_start_m": None,
                    "data_stop_m": None,
                    "data_min_step_m": None,
                    "data_max_step_m": None,
                    "data_median_step_m": None,
                    "n_duplicate_md": None,
                    "n_non_monotonic_md": None,
                    "strt_matches_declared": None,
                    "stop_matches_declared": None,
                    "step_matches_declared": None,
                    "contract_status": "FAILED",
                    "n_warnings": sum(1 for i in issues if i.severity == "WARNING"),
                    "n_errors": sum(1 for i in issues if i.severity == "ERROR") or 1,
                    "error_type": failure.error_type,
                    "error_message": failure.message,
                }
            )
    return rows


def build_curve_catalog_rows(results: Dict[str, LasFileResult]) -> List[dict]:
    """One row per contract-declared curve: raw identity, canonical target, resolution status."""
    rows: List[dict] = []
    for key, result in results.items():
        resolutions_by_name = {r.canonical_name: r for r in result.resolutions}
        for entry in result.contract.curves:
            res = resolutions_by_name[entry.canonical_name]
            rows.append(
                {
                    "well_key": key,
                    "source_filename": result.header.source_filename,
                    "ordinal": entry.ordinal,
                    "semantic_role": entry.semantic_role,
                    "source_curve_name": entry.source_curve_name,
                    "raw_mnemonic": entry.raw_mnemonic,
                    "raw_unit": entry.raw_unit,
                    "raw_description": entry.raw_description,
                    "raw_canonical_name": entry.raw_canonical_name,
                    "canonical_name": entry.canonical_name,
                    "canonical_unit": entry.canonical_unit,
                    "required": entry.required,
                    "conversion_function": entry.conversion_function,
                    "resolution_status": res.status,
                    "resolution_detail": res.detail,
                    "notes": entry.notes.strip(),
                }
            )
    return rows


def build_curve_coverage_rows(results: Dict[str, LasFileResult]) -> List[dict]:
    """
    One row per successfully resolved curve, with EVERY number tagged by
    an explicit unit (Increment 2.1 correction - see the module and
    `CurveStats` docstrings for why the Increment 2 version of this table
    was ambiguous).
    """
    rows: List[dict] = []
    for key, result in results.items():
        for stat in result.curve_stats:
            rows.append(
                {
                    "well_key": key,
                    "source_filename": result.header.source_filename,
                    "source_curve_name": stat.source_curve_name,
                    "raw_mnemonic": stat.raw_mnemonic,
                    "raw_description": stat.raw_description,
                    "raw_unit": stat.raw_unit,
                    "canonical_name": stat.canonical_name,
                    "canonical_unit": stat.canonical_unit,
                    "conversion_function": stat.conversion_function,
                    "n_samples": stat.n_samples,
                    "valid_count": stat.valid_count,
                    "null_count": stat.null_count,
                    "valid_fraction": stat.valid_fraction,
                    "raw_min": stat.raw_min,
                    "raw_max": stat.raw_max,
                    "canonical_min": stat.canonical_min,
                    "canonical_max": stat.canonical_max,
                    "statistics_basis": stat.statistics_basis,
                }
            )
    return rows


def build_issues_rows(
    results: Dict[str, LasFileResult], errors: Optional[Dict[str, IngestionFailure]] = None
) -> List[dict]:
    """One row per WARNING/ERROR issue found across all wells (successes and failures)."""
    rows: List[dict] = []
    for key, result in results.items():
        for issue in result.issues:
            rows.append(
                {
                    "well_key": key,
                    "source_filename": result.header.source_filename,
                    "severity": issue.severity,
                    "code": issue.code,
                    "message": issue.message,
                    "context": issue.context,
                }
            )
    if errors:
        for key, failure in errors.items():
            _, issues = _failed_well_header_and_issues(failure)
            if issues:
                for issue in issues:
                    rows.append(
                        {
                            "well_key": key,
                            "source_filename": Path(failure.source_path).name,
                            "severity": issue.severity,
                            "code": issue.code,
                            "message": issue.message,
                            "context": issue.context,
                        }
                    )
            else:
                # file_not_found / parsing_failure / conversion_failure carry
                # no structured IngestionIssue list (the failure happened
                # before or outside per-curve issue tracking) - the failure
                # itself is still recorded as a single ERROR row so it is
                # never silently absent from this table.
                rows.append(
                    {
                        "well_key": key,
                        "source_filename": Path(failure.source_path).name,
                        "severity": "ERROR",
                        "code": failure.error_type.upper(),
                        "message": failure.message,
                        "context": failure.source_path,
                    }
                )
    return rows


def build_ingestion_manifest(
    results: Dict[str, LasFileResult], errors: Optional[Dict[str, IngestionFailure]] = None
) -> dict:
    """
    Build the single JSON manifest document tying every well's provenance,
    contract-resolution outcome, and summary statistics together.
    """
    errors = errors or {}
    wells = {}
    for key, result in results.items():
        h = result.header
        wells[key] = {
            "source_filename": h.source_filename,
            "sha256": h.sha256,
            "well_name": h.well_name,
            "las_version": h.las_version,
            "wrap": h.wrap,
            "declared_null": h.declared_null,
            "declared_strt_m": h.declared_strt,
            "declared_stop_m": h.declared_stop,
            "declared_step_m": h.declared_step,
            "n_samples": result.depth.n_samples,
            "data_start_m": result.depth.data_start,
            "data_stop_m": result.depth.data_stop,
            "contract_status": result.contract_status,
            "resolved_curves": [r.canonical_name for r in result.resolutions if r.status == "RESOLVED"],
            "missing_optional_curves": [
                r.canonical_name for r in result.resolutions if r.status == "MISSING_OPTIONAL"
            ],
            "n_warnings": sum(1 for i in result.issues if i.severity == "WARNING"),
            "n_errors": sum(1 for i in result.issues if i.severity == "ERROR"),
        }
    for key, failure in errors.items():
        h, issues = _failed_well_header_and_issues(failure)
        wells[key] = {
            "source_filename": h.source_filename if h else Path(failure.source_path).name,
            "sha256": h.sha256 if h else None,
            "well_name": h.well_name if h else None,
            "las_version": h.las_version if h else None,
            "wrap": h.wrap if h else None,
            "declared_null": h.declared_null if h else None,
            "declared_strt_m": h.declared_strt if h else None,
            "declared_stop_m": h.declared_stop if h else None,
            "declared_step_m": h.declared_step if h else None,
            "n_samples": None,
            "data_start_m": None,
            "data_stop_m": None,
            "contract_status": "FAILED",
            "resolved_curves": [],
            "missing_optional_curves": [],
            "n_warnings": sum(1 for i in issues if i.severity == "WARNING"),
            "n_errors": sum(1 for i in issues if i.severity == "ERROR") or 1,
            "error_type": failure.error_type,
            "error_message": failure.message,
        }

    total_rows = sum(r.depth.n_samples for r in results.values())
    return {
        "increment": "2.1",
        "assurance_tier": "Tier C - Screening-Level / Uncalibrated Educational",
        "wells": wells,
        "summary": {
            "n_wells_loaded": len(results),
            "n_wells_failed": len(errors),
            "total_log_rows_all_wells": total_rows,
        },
    }


### Step 7 — Write the per-file curve contracts (`config/las_curve_contracts.yml`)

**Technical objective:** write the human-authored curve contract for each of the four approved wells. Every field in this file was read directly from that file's actual `~Curve Information Section` (see `INCREMENT_02_v2.1_MANIFEST.md`, "How these contracts were built") — no curve name, unit, description, or ordinal was invented.

**Why one contract per file, not one shared contract (repeated from the phase header because it is the single most important design decision in this increment):** column order is NOT consistent across the four files. Poseidon 2, Boreas 1, and Poseidon North 1 all place their neutron-porosity curve last (ordinal 8); Proteus 1ST2 places it at ordinal 5, shifting RD/RHOB/RS one position earlier. A shared contract keyed only by ordinal would silently mislabel Proteus 1ST2.

**Increment 2.1 additions to this file:** each well entry now also declares `expected_sha256`, `expected_las_version`, `expected_wrap`, and `expected_data_layout` (cross-checked, blocking, against the actual file before any curve is resolved); each curve entry now declares `source_curve_name`, `raw_canonical_name` (the raw column's unit-suffixed identity, e.g. `DTCO_us_per_ft`), and an explicit `semantic_role` (`measured_depth` for exactly one curve per file, `measurement` for every other). Canonical names are now unit-suffixed throughout (`VP_m_s`, `VS_m_s`, `RHOB_kg_m3`, `GR_api`, ...) — never a bare mnemonic that could be mistaken for a different physical quantity.

**Inputs:** none (this is the authored contract itself).
**Outputs:** `config/las_curve_contracts.yml`, loaded and validated by `p2mem.io.las.load_file_contract_config` in Step 11.

**Validation logic:** `load_file_contract_config` validates this file at load time — a duplicate `canonical_name` or ordinal, a missing required field, an unrecognized or unit-incompatible `conversion_function`, or a missing/duplicated `measured_depth` role all raise `LasContractDefinitionError` immediately, before any LAS file is touched.

**Expected result:** four well entries (`Poseidon_2_logs.las`, `Boreas_1_logs.las`, `Poseidon_North_1_logs.las`, `Proteus_1ST2_logs.las`), 9 curves each, loading without error.

In [ ]:
%%writefile config/las_curve_contracts.yml
# config/las_curve_contracts.yml
#
# Per-file LAS curve contracts for the four approved Poseidon 2 MEM wells
# (Increment 2, corrected in Increment 2.1). Every field below was read
# directly from each file's actual ~Curve Information Section (see
# INCREMENT_02_v2.1_MANIFEST.md, "How these contracts were built") -
# nothing here was inferred, guessed, or copied from another well's
# contract.
#
# WHY ONE CONTRACT PER FILE, NOT ONE SHARED CONTRACT:
# The curve mnemonic field is empty for every curve except DEPT in all
# four files; only the free-text description carries usable identity
# (e.g. ": 1 DCAV" = ordinal 1, curve name DCAV). Column order is NOT
# consistent across wells: Poseidon 2, Boreas 1, and Poseidon North 1 all
# place their neutron-porosity curve last (ordinal 8), but Proteus 1ST2
# places it at ordinal 5 - shifting RD, RHOB, and RS one position earlier
# than in the other three files. A single shared contract keyed only by
# ordinal would silently mislabel Proteus 1ST2's curves. Each file
# therefore gets its own independent, fully-specified contract.
#
# RESOLUTION RULE (enforced in code, not just here): a curve is resolved
# only when ITS ORDINAL, RAW UNIT, DESCRIPTION-EMBEDDED ORDINAL, and
# DESCRIPTION-EMBEDDED NAME all agree with this contract (plus RAW
# MNEMONIC, for the one curve - the measured-depth curve - that has one).
# Ordinal position is never sufficient by itself. See
# p2mem/io/las.py:resolve_curve_contract.
#
# INCREMENT 2.1 CORRECTIONS APPLIED TO THIS FILE (see INCREMENT_02_v2.1_MANIFEST.md
# for the full audit and rationale):
#
# 1. NAMING: every curve now carries an explicit, unit-suffixed
#    `raw_canonical_name` (the raw column's documented identity, e.g.
#    "DTCO_us_per_ft") and `canonical_name` (the array key in
#    `canonical_data`, e.g. "VP_m_s" for converted compressional
#    velocity). A canonical name can no longer be mistaken for a
#    different physical quantity or unit than the array it labels -
#    "DTCO"/"DTSM" (slowness mnemonics) are never used to label a
#    velocity array. GR, ECGR, and GRD remain three distinct canonical
#    names (GR_api / ECGR_api / GRD_api) - never merged into one
#    geological interpretation.
# 2. DEPTH ROLE: exactly one curve per file now declares
#    `semantic_role: measured_depth` explicitly. It is never located by
#    searching for a canonical name spelled "DEPT" - this file renames
#    that curve's canonical target to "MD_m", and code that inferred the
#    role from the name "DEPT" would otherwise silently break.
# 3. FILE IDENTITY: `expected_sha256`, `expected_las_version`,
#    `expected_wrap`, and `expected_data_layout` are now declared and
#    checked (blocking) against the actual file before any curve is
#    resolved. `expected_null_value` is required (was optional and only
#    a WARNING on mismatch in Increment 2; it is now required and a
#    blocking ERROR on mismatch).
#
# CANONICAL NAMING POLICY: each well's own description-derived curve name
# supplies the identity token used in its canonical name (e.g. Poseidon
# 2's neutron porosity curve is canonicalized as "TNP_pct", Boreas 1's and
# Poseidon North 1's as "TNPH_pct", per their own files) rather than
# unifying physically similar curves across wells under one shared name.
# Cross-well semantic unification is a petrophysical-interpretation
# decision explicitly out of scope for LAS ingestion (Increment 2/2.1) and
# is deferred to a later, explicitly-scoped increment.
#
# UNIT-CONVERSION POLICY: p2mem.units (Increment 1.1, unchanged/locked) is
# used ONLY where it offers an exact, unit-only conversion function for
# the quantity in question: us/ft -> m/s (DTCO, DTSM) and g/cc -> kg/m3
# (RHOB). Measured depth is already in metres (identity - only its name
# changes, DEPT -> MD). No conversion function exists in p2mem.units for
# inches-to-metres, API gamma-ray units, ohm.m resistivity, or percent
# neutron porosity as of Increment 1.1 - those curves are therefore left
# in their original (raw = canonical) unit rather than inventing an ad hoc
# conversion outside the tested, documented units module. Every
# conversion_function declared below is validated at load time against a
# controlled unit-signature registry (p2mem/io/las.py:_CONVERSION_UNIT_SIGNATURES)
# so it cannot be paired with an incompatible unit.

files:

  Poseidon_2_logs.las:
    expected_sha256: "4684864b2d4b37be1f132cc264d4672055865aa283b4fc7d7ef702f5d9a86828"
    expected_well_identifier: "Poseidon 2"
    expected_las_version: "2.0"
    expected_wrap: "NO"
    expected_null_value: -999.25
    expected_curve_count: 9
    expected_data_layout: "unwrapped_whitespace_delimited"
    curves:
      - ordinal: 0
        semantic_role: measured_depth
        source_curve_name: "DEPT"
        raw_mnemonic: "DEPT"
        raw_unit: "M"
        raw_description: "0 Depth"
        description_ordinal: 0
        description_name: "Depth"
        raw_canonical_name: "DEPT_m"
        canonical_name: "MD_m"
        canonical_unit: "m"
        required: true
        conversion_function: "identity"
        notes: >
          Measured depth index. Mnemonic is present (unlike every other
          curve in this file) and unit is already SI metres; canonical
          name renamed DEPT -> MD (identity conversion - no numeric
          transform, only the array's name changes) per the Increment 2.1
          explicit-measured-depth-role correction. Primary well.

      - ordinal: 1
        semantic_role: measurement
        source_curve_name: "DCAV"
        raw_mnemonic: ""
        raw_unit: "in"
        raw_description: "1 DCAV"
        description_ordinal: 1
        description_name: "DCAV"
        raw_canonical_name: "DCAV_in"
        canonical_name: "DCAV_in"
        canonical_unit: "in"
        required: true
        conversion_function: "identity"
        notes: >
          Differential caliper. Raw mnemonic field is empty in source LAS;
          identity resolved via ordinal + unit ("in") + description-ordinal
          + description-name agreement, never via ordinal alone. No
          inches-to-metres conversion function exists in p2mem.units as of
          Increment 1.1 (only feet<->metres is implemented); retained in
          original unit (identity). This curve is unique to Poseidon 2
          among the four approved wells - the other three use HDAR
          instead.

      - ordinal: 2
        semantic_role: measurement
        source_curve_name: "DTCO"
        raw_mnemonic: ""
        raw_unit: "us/ft"
        raw_description: "2 DTCO"
        description_ordinal: 2
        description_name: "DTCO"
        raw_canonical_name: "DTCO_us_per_ft"
        canonical_name: "VP_m_s"
        canonical_unit: "m/s"
        required: true
        conversion_function: "us_per_ft_to_m_per_s"
        notes: >
          Compressional sonic transit time (slowness, us/ft), converted to
          compressional velocity (VP, m/s) via the exact, definitional
          p2mem.units.us_per_ft_to_m_per_s function (Increment 1.1). The
          canonical array is named "VP_m_s", never "DTCO", because DTCO
          conventionally means slowness in us/ft - using it to label a
          velocity array would misidentify the physical quantity
          (Increment 2.1 correction). The raw column remains separately
          identifiable as DTCO_us_per_ft.

      - ordinal: 3
        semantic_role: measurement
        source_curve_name: "DTSM"
        raw_mnemonic: ""
        raw_unit: "us/ft"
        raw_description: "3 DTSM"
        description_ordinal: 3
        description_name: "DTSM"
        raw_canonical_name: "DTSM_us_per_ft"
        canonical_name: "VS_m_s"
        canonical_unit: "m/s"
        required: true
        conversion_function: "us_per_ft_to_m_per_s"
        notes: >
          Shear sonic transit time (slowness, us/ft), converted to shear
          velocity (VS, m/s) via the same exact conversion function as
          DTCO/VP. Canonical name "VS_m_s", not "DTSM", for the same
          reason as VP above.

      - ordinal: 4
        semantic_role: measurement
        source_curve_name: "GR"
        raw_mnemonic: ""
        raw_unit: "API"
        raw_description: "4 GR"
        description_ordinal: 4
        description_name: "GR"
        raw_canonical_name: "GR_api"
        canonical_name: "GR_api"
        canonical_unit: "API"
        required: true
        conversion_function: "identity"
        notes: >
          Gamma ray. API gamma-ray units have no SI equivalent conversion
          function in p2mem.units; retained as-is (identity). This is
          Poseidon 2's primary GR curve, canonicalized "GR_api" - kept
          distinct from Boreas 1's "ECGR_api" and Poseidon North 1's
          "GRD_api" (never merged into one geological interpretation, per
          the Increment 2.1 correction).

      - ordinal: 5
        semantic_role: measurement
        source_curve_name: "RD"
        raw_mnemonic: ""
        raw_unit: "ohmm"
        raw_description: "5 RD"
        description_ordinal: 5
        description_name: "RD"
        raw_canonical_name: "RD_ohm_m"
        canonical_name: "RD_ohm_m"
        canonical_unit: "ohmm"
        required: true
        conversion_function: "identity"
        notes: >
          Deep resistivity. No SI/ohm.m conversion function exists in
          p2mem.units; retained as-is (identity).

      - ordinal: 6
        semantic_role: measurement
        source_curve_name: "RHOB"
        raw_mnemonic: ""
        raw_unit: "g/cc"
        raw_description: "6 RHOB"
        description_ordinal: 6
        description_name: "RHOB"
        raw_canonical_name: "RHOB_g_per_cm3"
        canonical_name: "RHOB_kg_m3"
        canonical_unit: "kg/m3"
        required: true
        conversion_function: "gcc_to_kgm3"
        notes: >
          Bulk density, converted via the exact p2mem.units.gcc_to_kgm3
          function. Raw column (RHOB_g_per_cm3) preserved separately.
          NOTE (carried forward from the Rev 1 design review, not
          re-derived here): RHOB coverage in Poseidon 2 ends near 5296.85
          m MD even though this file's depth extends to ~5350.95 m MD -
          later increments must not assume RHOB is available across the
          full logged interval.

      - ordinal: 7
        semantic_role: measurement
        source_curve_name: "RS"
        raw_mnemonic: ""
        raw_unit: "ohmm"
        raw_description: "7 RS"
        description_ordinal: 7
        description_name: "RS"
        raw_canonical_name: "RS_ohm_m"
        canonical_name: "RS_ohm_m"
        canonical_unit: "ohmm"
        required: true
        conversion_function: "identity"
        notes: >
          Shallow resistivity. No conversion function available; retained
          as-is (identity).

      - ordinal: 8
        semantic_role: measurement
        source_curve_name: "TNP"
        raw_mnemonic: ""
        raw_unit: "%"
        raw_description: "8 TNP"
        description_ordinal: 8
        description_name: "TNP"
        raw_canonical_name: "TNP_pct"
        canonical_name: "TNP_pct"
        canonical_unit: "%"
        required: true
        conversion_function: "identity"
        notes: >
          Neutron porosity, as named in this file ("TNP"). No percent-to-
          fraction conversion function exists in p2mem.units; retained in
          original percent unit (identity). Kept as a distinct canonical
          name from Boreas 1's/Poseidon North 1's "TNPH_pct" per the
          project's cross-well-unification policy (see file header
          comment above).

  Boreas_1_logs.las:
    expected_sha256: "b2fdf60cd9dfeb7843ffa211232d909e7e76b6122cb2fd84f212edc2371dd61f"
    expected_well_identifier: "Boreas 1"
    expected_las_version: "2.0"
    expected_wrap: "NO"
    expected_null_value: -999.25
    expected_curve_count: 9
    expected_data_layout: "unwrapped_whitespace_delimited"
    curves:
      - ordinal: 0
        semantic_role: measured_depth
        source_curve_name: "DEPT"
        raw_mnemonic: "DEPT"
        raw_unit: "M"
        raw_description: "0 Depth"
        description_ordinal: 0
        description_name: "Depth"
        raw_canonical_name: "DEPT_m"
        canonical_name: "MD_m"
        canonical_unit: "m"
        required: true
        conversion_function: "identity"
        notes: "Measured depth index. Offset well."

      - ordinal: 1
        semantic_role: measurement
        source_curve_name: "DTCO"
        raw_mnemonic: ""
        raw_unit: "us/ft"
        raw_description: "1 DTCO"
        description_ordinal: 1
        description_name: "DTCO"
        raw_canonical_name: "DTCO_us_per_ft"
        canonical_name: "VP_m_s"
        canonical_unit: "m/s"
        required: true
        conversion_function: "us_per_ft_to_m_per_s"
        notes: "Compressional sonic transit time, converted to velocity VP_m_s (exact)."

      - ordinal: 2
        semantic_role: measurement
        source_curve_name: "DTSM"
        raw_mnemonic: ""
        raw_unit: "us/ft"
        raw_description: "2 DTSM"
        description_ordinal: 2
        description_name: "DTSM"
        raw_canonical_name: "DTSM_us_per_ft"
        canonical_name: "VS_m_s"
        canonical_unit: "m/s"
        required: true
        conversion_function: "us_per_ft_to_m_per_s"
        notes: "Shear sonic transit time, converted to velocity VS_m_s (exact)."

      - ordinal: 3
        semantic_role: measurement
        source_curve_name: "ECGR"
        raw_mnemonic: ""
        raw_unit: "API"
        raw_description: "3 ECGR"
        description_ordinal: 3
        description_name: "ECGR"
        raw_canonical_name: "ECGR_api"
        canonical_name: "ECGR_api"
        canonical_unit: "API"
        required: true
        conversion_function: "identity"
        notes: >
          KNOWN UNRESOLVED ISSUE (carried forward from the Rev 1 design
          review, not re-derived or re-verified here): this curve has an
          implausible median (~9.3 API) and range (~0 to ~519 API) versus
          the other three wells' GR-family curves (~36-41 API median), and
          samples appear above the declared seabed depth while RD/RS start
          correctly below it. This module reports ECGR's ingested
          statistics as plain facts (see CurveStats in the inventory
          output) but does NOT rescale it, reinterpret its lithology
          meaning, or otherwise act on the anomaly - that adjudication is
          explicitly out of scope for LAS ingestion. Canonicalized
          "ECGR_api", kept distinct from Poseidon 2's "GR_api" and
          Poseidon North 1's "GRD_api" - never merged.

      - ordinal: 4
        semantic_role: measurement
        source_curve_name: "HDAR"
        raw_mnemonic: ""
        raw_unit: "Inches"
        raw_description: "4 HDAR"
        description_ordinal: 4
        description_name: "HDAR"
        raw_canonical_name: "HDAR_in"
        canonical_name: "HDAR_in"
        canonical_unit: "Inches"
        required: true
        conversion_function: "identity"
        notes: >
          Caliper. Raw unit is spelled "Inches" in this file (vs Poseidon
          2's "in" for its DCAV caliper curve) - both denote the same
          physical unit; each is retained exactly as declared in its own
          file rather than normalized to a shared spelling, consistent
          with this project's per-file, non-fabricating contract policy.
          No inches-to-metres conversion function exists in p2mem.units.

      - ordinal: 5
        semantic_role: measurement
        source_curve_name: "RD"
        raw_mnemonic: ""
        raw_unit: "ohmm"
        raw_description: "5 RD"
        description_ordinal: 5
        description_name: "RD"
        raw_canonical_name: "RD_ohm_m"
        canonical_name: "RD_ohm_m"
        canonical_unit: "ohmm"
        required: true
        conversion_function: "identity"
        notes: "Deep resistivity. No conversion function available; retained as-is."

      - ordinal: 6
        semantic_role: measurement
        source_curve_name: "RHOB"
        raw_mnemonic: ""
        raw_unit: "g/cc"
        raw_description: "6 RHOB"
        description_ordinal: 6
        description_name: "RHOB"
        raw_canonical_name: "RHOB_g_per_cm3"
        canonical_name: "RHOB_kg_m3"
        canonical_unit: "kg/m3"
        required: true
        conversion_function: "gcc_to_kgm3"
        notes: "Bulk density, converted via p2mem.units.gcc_to_kgm3 (exact)."

      - ordinal: 7
        semantic_role: measurement
        source_curve_name: "RS"
        raw_mnemonic: ""
        raw_unit: "ohmm"
        raw_description: "7 RS"
        description_ordinal: 7
        description_name: "RS"
        raw_canonical_name: "RS_ohm_m"
        canonical_name: "RS_ohm_m"
        canonical_unit: "ohmm"
        required: true
        conversion_function: "identity"
        notes: "Shallow resistivity. No conversion function available; retained as-is."

      - ordinal: 8
        semantic_role: measurement
        source_curve_name: "TNPH"
        raw_mnemonic: ""
        raw_unit: "%"
        raw_description: "8 TNPH"
        description_ordinal: 8
        description_name: "TNPH"
        raw_canonical_name: "TNPH_pct"
        canonical_name: "TNPH_pct"
        canonical_unit: "%"
        required: true
        conversion_function: "identity"
        notes: >
          Neutron porosity, as named in this file ("TNPH"). Kept distinct
          from Poseidon 2's "TNP_pct" per the cross-well-unification
          policy.

  Poseidon_North_1_logs.las:
    expected_sha256: "ca3fe7b6547a72559127fab8c9540643600f4207efd067b84a08ad219354d78f"
    expected_well_identifier: "Poseidon North 1"
    expected_las_version: "2.0"
    expected_wrap: "NO"
    expected_null_value: -999.25
    expected_curve_count: 9
    expected_data_layout: "unwrapped_whitespace_delimited"
    curves:
      - ordinal: 0
        semantic_role: measured_depth
        source_curve_name: "DEPT"
        raw_mnemonic: "DEPT"
        raw_unit: "M"
        raw_description: "0 Depth"
        description_ordinal: 0
        description_name: "Depth"
        raw_canonical_name: "DEPT_m"
        canonical_name: "MD_m"
        canonical_unit: "m"
        required: true
        conversion_function: "identity"
        notes: >
          Measured depth index. Offset well. Formation tops for this well
          have not been obtained as of this increment (open Rev 1 gating
          item) - unaffected by LAS ingestion itself.

      - ordinal: 1
        semantic_role: measurement
        source_curve_name: "DTCO"
        raw_mnemonic: ""
        raw_unit: "us/ft"
        raw_description: "1 DTCO"
        description_ordinal: 1
        description_name: "DTCO"
        raw_canonical_name: "DTCO_us_per_ft"
        canonical_name: "VP_m_s"
        canonical_unit: "m/s"
        required: true
        conversion_function: "us_per_ft_to_m_per_s"
        notes: >
          Compressional sonic transit time, converted to velocity VP_m_s
          (exact). This well carries the Rev 1 design review's
          provisional, transferred candidate normal-compaction-trend
          interval - NOT validated, not re-assessed by this ingestion
          increment.

      - ordinal: 2
        semantic_role: measurement
        source_curve_name: "DTSM"
        raw_mnemonic: ""
        raw_unit: "us/ft"
        raw_description: "2 DTSM"
        description_ordinal: 2
        description_name: "DTSM"
        raw_canonical_name: "DTSM_us_per_ft"
        canonical_name: "VS_m_s"
        canonical_unit: "m/s"
        required: true
        conversion_function: "us_per_ft_to_m_per_s"
        notes: "Shear sonic transit time, converted to velocity VS_m_s (exact)."

      - ordinal: 3
        semantic_role: measurement
        source_curve_name: "GRD"
        raw_mnemonic: ""
        raw_unit: "API"
        raw_description: "3 GRD"
        description_ordinal: 3
        description_name: "GRD"
        raw_canonical_name: "GRD_api"
        canonical_name: "GRD_api"
        canonical_unit: "API"
        required: true
        conversion_function: "identity"
        notes: >
          Gamma ray, named "GRD" in this file's own description (distinct
          identity from Poseidon 2's "GR" and Boreas 1's "ECGR");
          canonicalized "GRD_api" - kept as its own canonical name, never
          merged with either.

      - ordinal: 4
        semantic_role: measurement
        source_curve_name: "HDAR"
        raw_mnemonic: ""
        raw_unit: "Inches"
        raw_description: "4 HDAR"
        description_ordinal: 4
        description_name: "HDAR"
        raw_canonical_name: "HDAR_in"
        canonical_name: "HDAR_in"
        canonical_unit: "Inches"
        required: true
        conversion_function: "identity"
        notes: "Caliper. No inches-to-metres conversion function available; retained as-is."

      - ordinal: 5
        semantic_role: measurement
        source_curve_name: "RD"
        raw_mnemonic: ""
        raw_unit: "ohmm"
        raw_description: "5 RD"
        description_ordinal: 5
        description_name: "RD"
        raw_canonical_name: "RD_ohm_m"
        canonical_name: "RD_ohm_m"
        canonical_unit: "ohmm"
        required: true
        conversion_function: "identity"
        notes: "Deep resistivity. No conversion function available; retained as-is."

      - ordinal: 6
        semantic_role: measurement
        source_curve_name: "RHOB"
        raw_mnemonic: ""
        raw_unit: "g/cc"
        raw_description: "6 RHOB"
        description_ordinal: 6
        description_name: "RHOB"
        raw_canonical_name: "RHOB_g_per_cm3"
        canonical_name: "RHOB_kg_m3"
        canonical_unit: "kg/m3"
        required: true
        conversion_function: "gcc_to_kgm3"
        notes: "Bulk density, converted via p2mem.units.gcc_to_kgm3 (exact)."

      - ordinal: 7
        semantic_role: measurement
        source_curve_name: "RS"
        raw_mnemonic: ""
        raw_unit: "ohmm"
        raw_description: "7 RS"
        description_ordinal: 7
        description_name: "RS"
        raw_canonical_name: "RS_ohm_m"
        canonical_name: "RS_ohm_m"
        canonical_unit: "ohmm"
        required: true
        conversion_function: "identity"
        notes: "Shallow resistivity. No conversion function available; retained as-is."

      - ordinal: 8
        semantic_role: measurement
        source_curve_name: "TNPH"
        raw_mnemonic: ""
        raw_unit: "%"
        raw_description: "8 TNPH"
        description_ordinal: 8
        description_name: "TNPH"
        raw_canonical_name: "TNPH_pct"
        canonical_name: "TNPH_pct"
        canonical_unit: "%"
        required: true
        conversion_function: "identity"
        notes: "Neutron porosity, as named in this file (\"TNPH\")."

  Proteus_1ST2_logs.las:
    expected_sha256: "ffb100de5fdd5e0639eee71a59a2602aad19a386116d48b8110912e58939e92f"
    expected_well_identifier: "Proteus 1ST2"
    expected_las_version: "2.0"
    expected_wrap: "NO"
    expected_null_value: -999.25
    expected_curve_count: 9
    expected_data_layout: "unwrapped_whitespace_delimited"
    curves:
      - ordinal: 0
        semantic_role: measured_depth
        source_curve_name: "DEPT"
        raw_mnemonic: "DEPT"
        raw_unit: "M"
        raw_description: "0 Depth"
        description_ordinal: 0
        description_name: "Depth"
        raw_canonical_name: "DEPT_m"
        canonical_name: "MD_m"
        canonical_unit: "m"
        required: true
        conversion_function: "identity"
        notes: >
          Measured depth index. Offset well / sidetrack. The Rev 1 design
          review flagged this well's DEVIATION SURVEY as a re-datumed
          sidetrack requiring separate audit - that finding concerns the
          deviation-survey file, not this LAS log file, and is unaffected
          by (and not re-examined by) LAS ingestion.

      - ordinal: 1
        semantic_role: measurement
        source_curve_name: "DTCO"
        raw_mnemonic: ""
        raw_unit: "us/ft"
        raw_description: "1 DTCO"
        description_ordinal: 1
        description_name: "DTCO"
        raw_canonical_name: "DTCO_us_per_ft"
        canonical_name: "VP_m_s"
        canonical_unit: "m/s"
        required: true
        conversion_function: "us_per_ft_to_m_per_s"
        notes: "Compressional sonic transit time, converted to velocity VP_m_s (exact)."

      - ordinal: 2
        semantic_role: measurement
        source_curve_name: "DTSM"
        raw_mnemonic: ""
        raw_unit: "us/ft"
        raw_description: "2 DTSM"
        description_ordinal: 2
        description_name: "DTSM"
        raw_canonical_name: "DTSM_us_per_ft"
        canonical_name: "VS_m_s"
        canonical_unit: "m/s"
        required: true
        conversion_function: "us_per_ft_to_m_per_s"
        notes: "Shear sonic transit time, converted to velocity VS_m_s (exact)."

      - ordinal: 3
        semantic_role: measurement
        source_curve_name: "GR"
        raw_mnemonic: ""
        raw_unit: "API"
        raw_description: "3 GR"
        description_ordinal: 3
        description_name: "GR"
        raw_canonical_name: "GR_api"
        canonical_name: "GR_api"
        canonical_unit: "API"
        required: true
        conversion_function: "identity"
        notes: >
          Gamma ray, named "GR" in this file (same name as Poseidon 2's GR
          curve, but a SEPARATE canonical value per file - this contract
          does not merge or cross-compare the two, even though both share
          the canonical name "GR_api" independently within their own
          file's namespace).

      - ordinal: 4
        semantic_role: measurement
        source_curve_name: "HDAR"
        raw_mnemonic: ""
        raw_unit: "Inches"
        raw_description: "4 HDAR"
        description_ordinal: 4
        description_name: "HDAR"
        raw_canonical_name: "HDAR_in"
        canonical_name: "HDAR_in"
        canonical_unit: "Inches"
        required: true
        conversion_function: "identity"
        notes: "Caliper. No inches-to-metres conversion function available; retained as-is."

      - ordinal: 5
        semantic_role: measurement
        source_curve_name: "NPHI"
        raw_mnemonic: ""
        raw_unit: "%"
        raw_description: "5 NPHI"
        description_ordinal: 5
        description_name: "NPHI"
        raw_canonical_name: "NPHI_pct"
        canonical_name: "NPHI_pct"
        canonical_unit: "%"
        required: true
        conversion_function: "identity"
        notes: >
          Neutron porosity, named "NPHI" in this file. IMPORTANT COLUMN-
          ORDER NOTE: unlike the other three approved wells (which place
          their neutron-porosity curve last, at ordinal 8), Proteus 1ST2
          places NPHI at ordinal 5 - shifting RD and RHOB one position
          earlier than in the other files (and moving RS to the last
          ordinal here). This is exactly the condition that makes a
          single shared, order-based contract unsafe across wells; this
          file's contract reflects its own actual column order only.

      - ordinal: 6
        semantic_role: measurement
        source_curve_name: "RD"
        raw_mnemonic: ""
        raw_unit: "ohmm"
        raw_description: "6 RD"
        description_ordinal: 6
        description_name: "RD"
        raw_canonical_name: "RD_ohm_m"
        canonical_name: "RD_ohm_m"
        canonical_unit: "ohmm"
        required: true
        conversion_function: "identity"
        notes: "Deep resistivity. No conversion function available; retained as-is."

      - ordinal: 7
        semantic_role: measurement
        source_curve_name: "RHOB"
        raw_mnemonic: ""
        raw_unit: "g/cc"
        raw_description: "7 RHOB"
        description_ordinal: 7
        description_name: "RHOB"
        raw_canonical_name: "RHOB_g_per_cm3"
        canonical_name: "RHOB_kg_m3"
        canonical_unit: "kg/m3"
        required: true
        conversion_function: "gcc_to_kgm3"
        notes: "Bulk density, converted via p2mem.units.gcc_to_kgm3 (exact)."

      - ordinal: 8
        semantic_role: measurement
        source_curve_name: "RS"
        raw_mnemonic: ""
        raw_unit: "ohmm"
        raw_description: "8 RS"
        description_ordinal: 8
        description_name: "RS"
        raw_canonical_name: "RS_ohm_m"
        canonical_name: "RS_ohm_m"
        canonical_unit: "ohmm"
        required: true
        conversion_function: "identity"
        notes: "Shallow resistivity. No conversion function available; retained as-is."


### Step 7b — Write the synthetic test fixtures and the LAS test suite

**Technical objective:** write the small synthetic LAS fixtures under `tests/fixtures/` and `tests/test_las.py`. These fixtures are deliberately tiny (a handful of curves, a handful of depth samples) and contain NO real project data - they exist purely so `tests/test_las.py` can run anywhere, without the four private/raw project LAS files, exercising: standard (non-empty) mnemonics, empty mnemonics with usable descriptions, declared-NULL substitution, a duplicate/ambiguous description, non-monotonic and duplicate measured depth, a malformed numeric row, a mismatched data-column width, a missing `WELL`, a missing `NULL`, a `VERS` mismatch, an unsupported `WRAP=YES`, and a literal non-finite ("nan") data token. A few further edge cases (wrong unit, unexpected column order, duplicate canonical mapping, filename/SHA-256 mismatch, missing/duplicated measured-depth role, conversion/unit incompatibility, missing required vs. missing optional curve) are exercised by constructing small contract or YAML variations in Python directly inside `tests/test_las.py`, rather than by multiplying near-duplicate fixture files.

**Expected result:** `tests/fixtures/` gains eleven `.las` files; `tests/test_las.py` is written and ready for Step 9.

In [ ]:
%%writefile tests/fixtures/standard_mnemonics.las
~Version Information Section
VERS.              2.0  : CWLS LOG ASCII STANDARD - VERSION 2.0
WRAP.              NO   : ONE LINE PER DEPTH STEP
#
~Well Information Section
#MNEM.UNIT         VALUE/NAME          DESCRIPTION
STRT.M             100.0000            : START DEPTH
STOP.M             100.4000            : STOP DEPTH
STEP.M             0.1000              : STEP
NULL.              -999.25             : NULL VALUE
WELL.              Synthetic Standard 1 : WELL NAME
#
~Curve Information Section
#MNEM.UNIT         API CODE            DESCRIPTION
DEPT.M                                 : Depth
GR.API                                 : Gamma Ray
RHOB.G/CC                              : Bulk Density
#
~Parameter Information Section
EKB.M              10.0000             : Elevation Kelly Bushing
#
~Ascii Data Section
    100.0000       50.1000        2.4500
    100.1000       51.2000        2.4600
    100.2000     -999.2500        2.4700
    100.3000       53.4000     -999.2500
    100.4000       54.5000        2.4900


In [ ]:
%%writefile tests/fixtures/empty_mnemonic_with_description.las
~Version Information Section
VERS.              2.0  : CWLS LOG ASCII STANDARD - VERSION 2.0
WRAP.              NO   : ONE LINE PER DEPTH STEP
#
~Well Information Section
#MNEM.UNIT         VALUE/NAME          DESCRIPTION
STRT.M             200.0000            : START DEPTH
STOP.M             200.4000            : STOP DEPTH
STEP.M             0.1000              : STEP
NULL.              -999.25             : NULL VALUE
WELL.              Synthetic Empty Mnemonic 1 : WELL NAME
#
~Curve Information Section
#MNEM.UNIT         API CODE            DESCRIPTION
DEPT.M                                 : 0 Depth
.us/ft                                 : 1 DTCO
.API                                   : 2 GR
.g/cc                                  : 3 RHOB
#
~Parameter Information Section
EKB.M              10.0000             : Elevation Kelly Bushing
#
~Ascii Data Section
    200.0000       120.5000       45.1000        2.3500
    200.1000       121.6000       46.2000        2.3600
    200.2000     -999.2500        47.3000        2.3700
    200.3000       123.8000     -999.2500        2.3800
    200.4000       124.9000       49.5000        2.3900


In [ ]:
%%writefile tests/fixtures/duplicate_description.las
~Version Information Section
VERS.              2.0  : CWLS LOG ASCII STANDARD - VERSION 2.0
WRAP.              NO   : ONE LINE PER DEPTH STEP
#
~Well Information Section
#MNEM.UNIT         VALUE/NAME          DESCRIPTION
STRT.M             300.0000            : START DEPTH
STOP.M             300.2000            : STOP DEPTH
STEP.M             0.1000              : STEP
NULL.              -999.25             : NULL VALUE
WELL.              Synthetic Duplicate Description 1 : WELL NAME
#
~Curve Information Section
#MNEM.UNIT         API CODE            DESCRIPTION
DEPT.M                                 : 0 Depth
.API                                   : GR
.API                                   : GR
#
~Parameter Information Section
EKB.M              10.0000             : Elevation Kelly Bushing
#
~Ascii Data Section
    300.0000       10.0000       20.0000
    300.1000       11.0000       21.0000
    300.2000       12.0000       22.0000


In [ ]:
%%writefile tests/fixtures/non_monotonic_and_duplicate_md.las
~Version Information Section
VERS.              2.0  : CWLS LOG ASCII STANDARD - VERSION 2.0
WRAP.              NO   : ONE LINE PER DEPTH STEP
#
~Well Information Section
#MNEM.UNIT         VALUE/NAME          DESCRIPTION
STRT.M             400.0000            : START DEPTH
STOP.M             400.4000            : STOP DEPTH
STEP.M             0.1000              : STEP
NULL.              -999.25             : NULL VALUE
WELL.              Synthetic Non Monotonic 1 : WELL NAME
#
~Curve Information Section
#MNEM.UNIT         API CODE            DESCRIPTION
DEPT.M                                 : 0 Depth
.API                                   : 1 GR
#
~Parameter Information Section
EKB.M              10.0000             : Elevation Kelly Bushing
#
~Ascii Data Section
    400.0000       10.0000
    400.1000       11.0000
    400.1000       11.5000
    400.0500       12.0000
    400.4000       13.0000


In [ ]:
%%writefile tests/fixtures/malformed_numeric_row.las
~Version Information Section
VERS.              2.0  : CWLS LOG ASCII STANDARD - VERSION 2.0
WRAP.              NO   : ONE LINE PER DEPTH STEP
#
~Well Information Section
#MNEM.UNIT         VALUE/NAME          DESCRIPTION
STRT.M             500.0000            : START DEPTH
STOP.M             500.2000            : STOP DEPTH
STEP.M             0.1000              : STEP
NULL.              -999.25             : NULL VALUE
WELL.              Synthetic Malformed Row 1 : WELL NAME
#
~Curve Information Section
#MNEM.UNIT         API CODE            DESCRIPTION
DEPT.M                                 : 0 Depth
.API                                   : 1 GR
#
~Parameter Information Section
EKB.M              10.0000             : Elevation Kelly Bushing
#
~Ascii Data Section
    500.0000       10.0000
    500.1000       N/A
    500.2000       12.0000


In [ ]:
%%writefile tests/fixtures/mismatched_width_row.las
~Version Information Section
VERS.              2.0  : CWLS LOG ASCII STANDARD - VERSION 2.0
WRAP.              NO   : ONE LINE PER DEPTH STEP
#
~Well Information Section
#MNEM.UNIT         VALUE/NAME          DESCRIPTION
STRT.M             600.0000            : START DEPTH
STOP.M             600.2000            : STOP DEPTH
STEP.M             0.1000              : STEP
NULL.              -999.25             : NULL VALUE
WELL.              Synthetic Mismatched Width 1 : WELL NAME
#
~Curve Information Section
#MNEM.UNIT         API CODE            DESCRIPTION
DEPT.M                                 : 0 Depth
.API                                   : 1 GR
.g/cc                                  : 2 RHOB
#
~Parameter Information Section
EKB.M              10.0000             : Elevation Kelly Bushing
#
~Ascii Data Section
    600.0000       10.0000        2.4000
    600.1000       11.0000
    600.2000       12.0000        2.4200


In [ ]:
%%writefile tests/fixtures/missing_well.las
~Version Information Section
VERS.              2.0  : CWLS LOG ASCII STANDARD - VERSION 2.0
WRAP.              NO   : ONE LINE PER DEPTH STEP
#
~Well Information Section
#MNEM.UNIT         VALUE/NAME          DESCRIPTION
STRT.M             700.0000            : START DEPTH
STOP.M             700.2000            : STOP DEPTH
STEP.M             0.1000              : STEP
NULL.              -999.25             : NULL VALUE
#
~Curve Information Section
#MNEM.UNIT         API CODE            DESCRIPTION
DEPT.M                                 : 0 Depth
.API                                   : 1 GR
#
~Parameter Information Section
EKB.M              10.0000             : Elevation Kelly Bushing
#
~Ascii Data Section
    700.0000       10.0000
    700.1000       11.0000
    700.2000       12.0000


In [ ]:
%%writefile tests/fixtures/missing_null.las
~Version Information Section
VERS.              2.0  : CWLS LOG ASCII STANDARD - VERSION 2.0
WRAP.              NO   : ONE LINE PER DEPTH STEP
#
~Well Information Section
#MNEM.UNIT         VALUE/NAME          DESCRIPTION
STRT.M             800.0000            : START DEPTH
STOP.M             800.2000            : STOP DEPTH
STEP.M             0.1000              : STEP
WELL.              Synthetic Missing Null 1 : WELL NAME
#
~Curve Information Section
#MNEM.UNIT         API CODE            DESCRIPTION
DEPT.M                                 : 0 Depth
.API                                   : 1 GR
#
~Parameter Information Section
EKB.M              10.0000             : Elevation Kelly Bushing
#
~Ascii Data Section
    800.0000       10.0000
    800.1000       11.0000
    800.2000       12.0000


In [ ]:
%%writefile tests/fixtures/vers_mismatch.las
~Version Information Section
VERS.              3.0  : CWLS LOG ASCII STANDARD - VERSION 3.0
WRAP.              NO   : ONE LINE PER DEPTH STEP
#
~Well Information Section
#MNEM.UNIT         VALUE/NAME          DESCRIPTION
STRT.M             900.0000            : START DEPTH
STOP.M             900.2000            : STOP DEPTH
STEP.M             0.1000              : STEP
NULL.              -999.25             : NULL VALUE
WELL.              Synthetic Vers Mismatch 1 : WELL NAME
#
~Curve Information Section
#MNEM.UNIT         API CODE            DESCRIPTION
DEPT.M                                 : 0 Depth
.API                                   : 1 GR
#
~Parameter Information Section
EKB.M              10.0000             : Elevation Kelly Bushing
#
~Ascii Data Section
    900.0000       10.0000
    900.1000       11.0000
    900.2000       12.0000


In [ ]:
%%writefile tests/fixtures/wrapped_unsupported.las
~Version Information Section
VERS.              2.0  : CWLS LOG ASCII STANDARD - VERSION 2.0
WRAP.              YES  : MULTIPLE LINES PER DEPTH STEP
#
~Well Information Section
#MNEM.UNIT         VALUE/NAME          DESCRIPTION
STRT.M             1000.0000           : START DEPTH
STOP.M             1000.2000           : STOP DEPTH
STEP.M             0.1000              : STEP
NULL.              -999.25             : NULL VALUE
WELL.              Synthetic Wrapped 1 : WELL NAME
#
~Curve Information Section
#MNEM.UNIT         API CODE            DESCRIPTION
DEPT.M                                 : 0 Depth
.API                                   : 1 GR
#
~Parameter Information Section
EKB.M              10.0000             : Elevation Kelly Bushing
#
~Ascii Data Section
    1000.0000      10.0000
    1000.1000      11.0000
    1000.2000      12.0000


In [ ]:
%%writefile tests/fixtures/non_finite_token.las
~Version Information Section
VERS.              2.0  : CWLS LOG ASCII STANDARD - VERSION 2.0
WRAP.              NO   : ONE LINE PER DEPTH STEP
#
~Well Information Section
#MNEM.UNIT         VALUE/NAME          DESCRIPTION
STRT.M             1100.0000           : START DEPTH
STOP.M             1100.2000           : STOP DEPTH
STEP.M             0.1000              : STEP
NULL.              -999.25             : NULL VALUE
WELL.              Synthetic Non Finite 1 : WELL NAME
#
~Curve Information Section
#MNEM.UNIT         API CODE            DESCRIPTION
DEPT.M                                 : 0 Depth
.API                                   : 1 GR
#
~Parameter Information Section
EKB.M              10.0000             : Elevation Kelly Bushing
#
~Ascii Data Section
    1100.0000      10.0000
    nan            11.0000
    1100.2000      12.0000


In [ ]:
%%writefile tests/test_las.py
"""
tests/test_las.py - Validation suite for p2mem.io.las (Increment 2,
corrected in Increment 2.1).

These are PORTABLE unit tests: they use only the small synthetic LAS
fixtures under tests/fixtures/ (plus a couple of contracts/YAML snippets
built inline) and never require the four private/raw project LAS files.
The real four-well integration run belongs in the Colab notebook
(02_LAS_Ingestion_and_Curve_Contracts.ipynb), not here.

Coverage required by the Increment 2 specification (unchanged, all still
passing under the corrected architecture):
    - valid LAS with standard (non-empty) mnemonics
    - valid LAS with empty mnemonic fields and usable descriptions
    - declared NULL -> NaN conversion
    - duplicate / ambiguous curve description
    - duplicate canonical mapping (contract-authoring error)
    - wrong unit (contract vs file disagreement)
    - unexpected column order
    - mismatched data-column width
    - non-monotonic measured depth
    - duplicate measured depth
    - missing required curve (and a present-but-optional counterpart)
    - malformed numeric row
    - scalar metadata extraction (WELL, STRT, STOP, STEP, NULL, VERS, WRAP)
    - deterministic output ordering

Coverage added by the Increment 2.1 corrective patch (see
INCREMENT_02_v2.1_MANIFEST.md for the audit this responds to):
    - raw DTCO stays "DTCO_us_per_ft"; canonical compressional velocity is
      "VP_m_s" (never "DTCO")
    - raw DTSM stays "DTSM_us_per_ft"; canonical shear velocity is "VS_m_s"
    - RHOB raw/canonical names and units are distinct and both explicit
    - coverage rows carry correct raw AND canonical units/min/max
    - filename mismatch / SHA-256 mismatch / missing WELL / missing NULL /
      VERS mismatch / NULL mismatch are all blocking
    - WRAP=YES is rejected clearly as an unimplemented layout
    - duplicate ordinals are rejected at contract-load time
    - a conversion function incompatible with its declared units is
      rejected at contract-load time
    - zero or multiple measured-depth-role curves are rejected
    - raw data retains the literal sentinel; canonical data replaces only
      the declared sentinel with NaN
    - a non-finite (literal NaN/Inf) token is rejected as a structural
      parsing defect
    - per-file failures (file-not-found, parsing, contract, conversion)
      are isolated by `load_wells` as typed `IngestionFailure` records
    - deterministic output ordering is unchanged under the new schema

Run with:  pytest -v
"""

import hashlib
import math
import tempfile
from pathlib import Path

import numpy as np
import pytest

from p2mem.io.las import (
    LasContractDefinitionError,
    LasContractError,
    LasConversionError,
    LasFileNotFoundError,
    LasParsingError,
    _parse_definition_line,
    _parse_description_ordinal_name,
    compute_depth_diagnostics,
    load_file_contract_config,
    load_las_file,
    load_wells,
    parse_las_header,
    resolve_curve_contract,
)
from p2mem.models import CurveContractEntry, FileContract, IngestionFailure

FIXTURES = Path(__file__).parent / "fixtures"


def _sha256(path: Path) -> str:
    return hashlib.sha256(path.read_bytes()).hexdigest()


def _entry(
    ordinal,
    raw_mnemonic,
    raw_unit,
    raw_description,
    description_ordinal,
    description_name,
    source_curve_name,
    raw_canonical_name,
    canonical_name,
    canonical_unit,
    required=True,
    conversion_function="identity",
    semantic_role="measurement",
    notes="test fixture entry",
):
    return CurveContractEntry(
        ordinal=ordinal,
        semantic_role=semantic_role,
        source_curve_name=source_curve_name,
        raw_mnemonic=raw_mnemonic,
        raw_unit=raw_unit,
        raw_description=raw_description,
        description_ordinal=description_ordinal,
        description_name=description_name,
        raw_canonical_name=raw_canonical_name,
        canonical_name=canonical_name,
        canonical_unit=canonical_unit,
        required=required,
        conversion_function=conversion_function,
        notes=notes,
    )


def _depth_entry(ordinal=0, raw_mnemonic="DEPT", raw_description="Depth", description_ordinal=None, description_name=None):
    return _entry(
        ordinal, raw_mnemonic, "M", raw_description, description_ordinal, description_name,
        source_curve_name="DEPT", raw_canonical_name="DEPT_m", canonical_name="MD_m", canonical_unit="m",
        semantic_role="measured_depth",
    )


def _fixture_contract(filename: str, expected_well_identifier: str, expected_curve_count: int, curves, expected_null_value=-999.25) -> FileContract:
    """Build a FileContract for a fixture file, filling in its real SHA-256 so file-identity checks pass by default."""
    return FileContract(
        source_filename=filename,
        expected_sha256=_sha256(FIXTURES / filename),
        expected_well_identifier=expected_well_identifier,
        expected_las_version="2.0",
        expected_wrap="NO",
        expected_null_value=expected_null_value,
        expected_curve_count=expected_curve_count,
        expected_data_layout="unwrapped_whitespace_delimited",
        curves=tuple(curves),
    )


def _standard_contract() -> FileContract:
    return _fixture_contract(
        "standard_mnemonics.las",
        "Synthetic Standard 1",
        3,
        [
            _depth_entry(0, "DEPT", "Depth"),
            _entry(1, "GR", "API", "Gamma Ray", None, None, "GR", "GR_api", "GR_api", "API"),
            _entry(2, "RHOB", "G/CC", "Bulk Density", None, None, "RHOB", "RHOB_g_per_cm3", "RHOB_kg_m3", "kg/m3", conversion_function="gcc_to_kgm3"),
        ],
    )


def _empty_mnemonic_contract() -> FileContract:
    return _fixture_contract(
        "empty_mnemonic_with_description.las",
        "Synthetic Empty Mnemonic 1",
        4,
        [
            _depth_entry(0, "DEPT", "0 Depth", 0, "Depth"),
            _entry(1, "", "us/ft", "1 DTCO", 1, "DTCO", "DTCO", "DTCO_us_per_ft", "VP_m_s", "m/s", conversion_function="us_per_ft_to_m_per_s"),
            _entry(2, "", "API", "2 GR", 2, "GR", "GR", "GR_api", "GR_api", "API"),
            _entry(3, "", "g/cc", "3 RHOB", 3, "RHOB", "RHOB", "RHOB_g_per_cm3", "RHOB_kg_m3", "kg/m3", conversion_function="gcc_to_kgm3"),
        ],
    )


# ---------------------------------------------------------------------------
# Definition-line parser (the LAS "MNEM.UNIT VALUE :DESC" edge case)
# ---------------------------------------------------------------------------
def test_parse_definition_line_unit_and_value_present():
    d = _parse_definition_line("STRT.M             490.0000            : START DEPTH")
    assert (d.mnemonic, d.unit, d.value, d.description) == ("STRT", "M", "490.0000", "START DEPTH")


def test_parse_definition_line_empty_unit_multiword_value():
    d = _parse_definition_line("WELL.              Poseidon 2          : WELL NAME")
    assert (d.mnemonic, d.unit, d.value, d.description) == ("WELL", "", "Poseidon 2", "WELL NAME")


def test_parse_definition_line_empty_mnemonic_curve_style():
    d = _parse_definition_line(".in                                    : 1 DCAV")
    assert (d.mnemonic, d.unit, d.value, d.description) == ("", "in", "", "1 DCAV")


def test_parse_definition_line_no_description():
    d = _parse_definition_line("CREA.              17-jun-2026")
    assert (d.mnemonic, d.unit, d.value, d.description) == ("CREA", "", "17-jun-2026", "")


def test_parse_definition_line_requires_dot():
    with pytest.raises(LasParsingError):
        _parse_definition_line("NOAAADOT VALUE : DESC")


def test_parse_description_ordinal_name():
    assert _parse_description_ordinal_name("1 DCAV") == (1, "DCAV")
    assert _parse_description_ordinal_name("0 Depth") == (0, "Depth")
    assert _parse_description_ordinal_name("Gamma Ray") == (None, None)
    assert _parse_description_ordinal_name("") == (None, None)


# ---------------------------------------------------------------------------
# Header parsing + scalar metadata extraction
# ---------------------------------------------------------------------------
def test_parse_las_header_standard_mnemonics():
    header = parse_las_header(str(FIXTURES / "standard_mnemonics.las"))
    assert header.source_filename == "standard_mnemonics.las"
    assert len(header.sha256) == 64
    assert header.las_version == "2.0"
    assert header.wrap == "NO"
    assert header.well_name == "Synthetic Standard 1"
    assert header.declared_null == pytest.approx(-999.25)
    assert header.declared_strt == pytest.approx(100.0)
    assert header.declared_stop == pytest.approx(100.4)
    assert header.declared_step == pytest.approx(0.1)
    assert len(header.curve_headers) == 3
    assert [c.raw_mnemonic for c in header.curve_headers] == ["DEPT", "GR", "RHOB"]
    assert [c.ordinal for c in header.curve_headers] == [0, 1, 2]


def test_parse_las_header_empty_mnemonic_with_description():
    header = parse_las_header(str(FIXTURES / "empty_mnemonic_with_description.las"))
    assert len(header.curve_headers) == 4
    dept, dtco, gr, rhob = header.curve_headers
    assert dept.raw_mnemonic == "DEPT" and dept.description_ordinal == 0 and dept.description_name == "Depth"
    assert dtco.raw_mnemonic == "" and dtco.raw_unit == "us/ft"
    assert dtco.description_ordinal == 1 and dtco.description_name == "DTCO"
    assert gr.raw_mnemonic == "" and gr.description_ordinal == 2 and gr.description_name == "GR"
    assert rhob.raw_mnemonic == "" and rhob.description_ordinal == 3 and rhob.description_name == "RHOB"


def test_parse_las_header_missing_file_raises():
    with pytest.raises(LasFileNotFoundError):
        parse_las_header(str(FIXTURES / "does_not_exist.las"))


# ---------------------------------------------------------------------------
# Valid loads: standard mnemonics, and empty-mnemonic-with-description
# ---------------------------------------------------------------------------
def test_load_standard_mnemonics_file():
    result = load_las_file(str(FIXTURES / "standard_mnemonics.las"), _standard_contract())
    assert result.contract_status == "PASSED"
    assert set(result.canonical_data) == {"MD_m", "GR_api", "RHOB_kg_m3"}
    assert result.depth.n_samples == 5
    # NULL sentinel -> NaN, non-null values preserved
    assert np.isnan(result.canonical_data["GR_api"][2])
    assert result.canonical_data["GR_api"][0] == pytest.approx(50.1)
    # RHOB row 3 is NULL -> NaN even after the exact g/cc->kg/m3 conversion
    assert np.isnan(result.canonical_data["RHOB_kg_m3"][3])
    assert result.canonical_data["RHOB_kg_m3"][0] == pytest.approx(2450.0)  # 2.45 g/cc * 1000


def test_load_empty_mnemonic_with_description_file():
    result = load_las_file(str(FIXTURES / "empty_mnemonic_with_description.las"), _empty_mnemonic_contract())
    assert result.contract_status == "PASSED"
    assert set(result.canonical_data) == {"MD_m", "VP_m_s", "GR_api", "RHOB_kg_m3"}
    # DTCO converted from us/ft to VP_m_s via the exact Increment 1.1 function
    expected_v = 304800.0 / 120.5
    assert result.canonical_data["VP_m_s"][0] == pytest.approx(expected_v, rel=1e-9)
    # raw_data must retain the untouched literal value, including NULLs
    dtco_ordinal = 1
    assert result.raw_data[2, dtco_ordinal] == pytest.approx(-999.25)


def test_raw_data_and_canonical_data_are_kept_separate():
    result = load_las_file(str(FIXTURES / "standard_mnemonics.las"), _standard_contract())
    # raw_data must never have NULL substituted (still literally -999.25)
    assert result.raw_data[2, 1] == pytest.approx(-999.25)
    # canonical_data for the same cell must be NaN
    assert np.isnan(result.canonical_data["GR_api"][2])


# ---------------------------------------------------------------------------
# Increment 2.1: explicit, unit-suffixed naming (never the bare mnemonic)
# ---------------------------------------------------------------------------
def test_canonical_velocity_names_never_use_slowness_mnemonic():
    result = load_las_file(str(FIXTURES / "empty_mnemonic_with_description.las"), _empty_mnemonic_contract())
    # "DTCO" (a slowness mnemonic, us/ft) must never label the converted
    # velocity array (m/s) - the canonical key is "VP_m_s".
    assert "DTCO" not in result.canonical_data
    assert "VP_m_s" in result.canonical_data
    dtco_entry = next(c for c in result.contract.curves if c.source_curve_name == "DTCO")
    assert dtco_entry.raw_canonical_name == "DTCO_us_per_ft"
    assert dtco_entry.canonical_name == "VP_m_s"
    assert dtco_entry.canonical_unit == "m/s"


def test_canonical_shear_velocity_naming():
    contract = _fixture_contract(
        "empty_mnemonic_with_description.las",
        "Synthetic Empty Mnemonic 1",
        4,
        [
            _depth_entry(0, "DEPT", "0 Depth", 0, "Depth"),
            _entry(1, "", "us/ft", "1 DTCO", 1, "DTCO", "DTSM", "DTSM_us_per_ft", "VS_m_s", "m/s", conversion_function="us_per_ft_to_m_per_s"),
            _entry(2, "", "API", "2 GR", 2, "GR", "GR", "GR_api", "GR_api", "API"),
            _entry(3, "", "g/cc", "3 RHOB", 3, "RHOB", "RHOB", "RHOB_g_per_cm3", "RHOB_kg_m3", "kg/m3", conversion_function="gcc_to_kgm3"),
        ],
    )
    result = load_las_file(str(FIXTURES / "empty_mnemonic_with_description.las"), contract)
    assert "DTSM" not in result.canonical_data
    assert "VS_m_s" in result.canonical_data
    entry = next(c for c in result.contract.curves if c.source_curve_name == "DTSM")
    assert entry.raw_canonical_name == "DTSM_us_per_ft"
    assert entry.canonical_name == "VS_m_s"


def test_rhob_raw_and_canonical_names_and_units_are_distinct():
    result = load_las_file(str(FIXTURES / "standard_mnemonics.las"), _standard_contract())
    entry = next(c for c in result.contract.curves if c.source_curve_name == "RHOB")
    assert entry.raw_canonical_name == "RHOB_g_per_cm3"
    assert entry.canonical_name == "RHOB_kg_m3"
    assert entry.raw_unit.lower() == "g/cc"
    assert entry.canonical_unit == "kg/m3"
    assert entry.raw_canonical_name != entry.canonical_name


# ---------------------------------------------------------------------------
# Declared NULL conversion
# ---------------------------------------------------------------------------
def test_declared_null_converted_to_nan_only_for_exact_match():
    result = load_las_file(str(FIXTURES / "standard_mnemonics.las"), _standard_contract())
    gr = result.canonical_data["GR_api"]
    assert np.sum(np.isnan(gr)) == 1
    assert np.isnan(gr[2])
    assert gr[4] == pytest.approx(54.5)


def test_raw_data_retains_literal_sentinel_canonical_replaces_only_declared_sentinel():
    result = load_las_file(str(FIXTURES / "standard_mnemonics.las"), _standard_contract())
    # Row 2, column 1 (GR) is the literal NULL sentinel in the source file.
    assert result.raw_data[2, 1] == pytest.approx(-999.25)
    assert np.isnan(result.canonical_data["GR_api"][2])
    # No other raw value is altered, and no other canonical value becomes
    # NaN just because it happens to be numerically extreme.
    assert result.raw_data[4, 1] == pytest.approx(54.5)
    assert not np.isnan(result.canonical_data["GR_api"][4])


# ---------------------------------------------------------------------------
# Duplicate / ambiguous description
# ---------------------------------------------------------------------------
def test_duplicate_raw_description_fails_contract():
    contract = _fixture_contract(
        "duplicate_description.las",
        "Synthetic Duplicate Description 1",
        3,
        [
            _depth_entry(0, "DEPT", "0 Depth", 0, "Depth"),
            _entry(1, "", "API", "GR", None, "GR", "GR_A", "GR_A_api", "GR_A_api", "API"),
            _entry(2, "", "API", "GR", None, "GR", "GR_B", "GR_B_api", "GR_B_api", "API"),
        ],
    )
    with pytest.raises(LasContractError) as excinfo:
        load_las_file(str(FIXTURES / "duplicate_description.las"), contract)
    assert any(i.code == "DUPLICATE_RAW_DESCRIPTION" for i in excinfo.value.issues)


# ---------------------------------------------------------------------------
# Duplicate canonical mapping (contract-authoring error, caught at config-load time)
# ---------------------------------------------------------------------------
_MINIMAL_FILE_HEADER_FIELDS = """
    expected_sha256: "abc123"
    expected_well_identifier: "Some Well"
    expected_las_version: "2.0"
    expected_wrap: "NO"
    expected_null_value: -999.25
    expected_data_layout: "unwrapped_whitespace_delimited"
"""


def test_duplicate_canonical_mapping_rejected_at_contract_load(tmp_path):
    bad_yaml = tmp_path / "bad_contracts.yml"
    bad_yaml.write_text(
        f"""
files:
  some_file.las:
{_MINIMAL_FILE_HEADER_FIELDS}
    expected_curve_count: 2
    curves:
      - ordinal: 0
        semantic_role: measured_depth
        source_curve_name: "DEPT"
        raw_mnemonic: "DEPT"
        raw_unit: "M"
        raw_description: "0 Depth"
        raw_canonical_name: "DEPT_m"
        canonical_name: "MD_m"
        canonical_unit: "m"
        required: true
        conversion_function: "identity"
      - ordinal: 1
        source_curve_name: "GR"
        raw_mnemonic: "GR"
        raw_unit: "API"
        raw_description: "1 GR"
        raw_canonical_name: "GR_api"
        canonical_name: "MD_m"
        canonical_unit: "API"
        required: true
        conversion_function: "identity"
"""
    )
    with pytest.raises(LasContractDefinitionError):
        load_file_contract_config(str(bad_yaml))


# ---------------------------------------------------------------------------
# Wrong unit
# ---------------------------------------------------------------------------
def test_wrong_unit_fails_contract():
    contract = _empty_mnemonic_contract()
    curves = list(contract.curves)
    # GR is declared "API" in the file; assert the contract expects "ohmm" instead.
    curves[2] = _entry(2, "", "ohmm", "2 GR", 2, "GR", "GR", "GR_api", "GR_api", "API")
    bad_contract = FileContract(
        source_filename=contract.source_filename,
        expected_sha256=contract.expected_sha256,
        expected_well_identifier=contract.expected_well_identifier,
        expected_las_version=contract.expected_las_version,
        expected_wrap=contract.expected_wrap,
        expected_null_value=contract.expected_null_value,
        expected_curve_count=contract.expected_curve_count,
        expected_data_layout=contract.expected_data_layout,
        curves=tuple(curves),
    )
    with pytest.raises(LasContractError) as excinfo:
        load_las_file(str(FIXTURES / "empty_mnemonic_with_description.las"), bad_contract)
    assert any(i.code == "CURVE_IDENTITY_MISMATCH" for i in excinfo.value.issues)


# ---------------------------------------------------------------------------
# Unexpected column order
# ---------------------------------------------------------------------------
def test_unexpected_column_order_fails_contract():
    contract = _fixture_contract(
        "empty_mnemonic_with_description.las",
        "Synthetic Empty Mnemonic 1",
        4,
        [
            _depth_entry(0, "DEPT", "0 Depth", 0, "Depth"),
            _entry(1, "", "API", "2 GR", 2, "GR", "GR", "GR_api", "GR_api", "API"),
            _entry(2, "", "us/ft", "1 DTCO", 1, "DTCO", "DTCO", "DTCO_us_per_ft", "VP_m_s", "m/s", conversion_function="us_per_ft_to_m_per_s"),
            _entry(3, "", "g/cc", "3 RHOB", 3, "RHOB", "RHOB", "RHOB_g_per_cm3", "RHOB_kg_m3", "kg/m3", conversion_function="gcc_to_kgm3"),
        ],
    )
    with pytest.raises(LasContractError) as excinfo:
        load_las_file(str(FIXTURES / "empty_mnemonic_with_description.las"), contract)
    codes = {i.code for i in excinfo.value.issues}
    assert "CURVE_IDENTITY_MISMATCH" in codes


# ---------------------------------------------------------------------------
# Mismatched data-column width
# ---------------------------------------------------------------------------
def test_mismatched_data_width_raises():
    contract = _fixture_contract(
        "mismatched_width_row.las",
        "Synthetic Mismatched Width 1",
        3,
        [
            _depth_entry(0, "DEPT", "0 Depth", 0, "Depth"),
            _entry(1, "", "API", "1 GR", 1, "GR", "GR", "GR_api", "GR_api", "API"),
            _entry(2, "", "g/cc", "2 RHOB", 2, "RHOB", "RHOB", "RHOB_g_per_cm3", "RHOB_kg_m3", "kg/m3", conversion_function="gcc_to_kgm3"),
        ],
    )
    with pytest.raises(LasParsingError, match="width mismatch"):
        load_las_file(str(FIXTURES / "mismatched_width_row.las"), contract)


# ---------------------------------------------------------------------------
# Malformed numeric row
# ---------------------------------------------------------------------------
def test_malformed_numeric_row_raises():
    contract = _fixture_contract(
        "malformed_numeric_row.las",
        "Synthetic Malformed Row 1",
        2,
        [
            _depth_entry(0, "DEPT", "0 Depth", 0, "Depth"),
            _entry(1, "", "API", "1 GR", 1, "GR", "GR", "GR_api", "GR_api", "API"),
        ],
    )
    with pytest.raises(LasParsingError, match="malformed numeric value"):
        load_las_file(str(FIXTURES / "malformed_numeric_row.las"), contract)


# ---------------------------------------------------------------------------
# Increment 2.1: literal non-finite token is rejected structurally
# ---------------------------------------------------------------------------
def test_non_finite_literal_token_is_rejected():
    with pytest.raises(LasParsingError, match="non-finite"):
        parse_las_header(str(FIXTURES / "non_finite_token.las"))  # header parses fine
        contract = _fixture_contract(
            "non_finite_token.las", "Synthetic Non Finite 1", 2,
            [_depth_entry(0, "DEPT", "0 Depth", 0, "Depth"),
             _entry(1, "", "API", "1 GR", 1, "GR", "GR", "GR_api", "GR_api", "API")],
        )
        load_las_file(str(FIXTURES / "non_finite_token.las"), contract)


def test_non_finite_depth_specifically_is_rejected():
    # The literal "nan" token in non_finite_token.las sits in the GR
    # column at row 1 (see the fixture) - build a variant contract where
    # that same ordinal is instead declared as the measured-depth curve,
    # to prove a non-finite value specifically in the depth position is
    # caught (the general structural check in _read_ascii_data already
    # rejects ANY non-finite token file-wide, so this documents that a
    # non-finite depth is never reachable past that point).
    path = FIXTURES / "non_finite_token.las"
    with pytest.raises(LasParsingError):
        contract = _fixture_contract(
            "non_finite_token.las", "Synthetic Non Finite 1", 2,
            [_depth_entry(0, "DEPT", "0 Depth", 0, "Depth"),
             _entry(1, "", "API", "1 GR", 1, "GR", "GR", "GR_api", "GR_api", "API")],
        )
        load_las_file(str(path), contract)


# ---------------------------------------------------------------------------
# Non-monotonic / duplicate measured depth (WARNING, not blocking)
# ---------------------------------------------------------------------------
def test_non_monotonic_and_duplicate_md_are_warnings_not_errors():
    contract = _fixture_contract(
        "non_monotonic_and_duplicate_md.las",
        "Synthetic Non Monotonic 1",
        2,
        [
            _depth_entry(0, "DEPT", "0 Depth", 0, "Depth"),
            _entry(1, "", "API", "1 GR", 1, "GR", "GR", "GR_api", "GR_api", "API"),
        ],
    )
    result = load_las_file(str(FIXTURES / "non_monotonic_and_duplicate_md.las"), contract)
    assert result.contract_status == "PASSED"
    assert result.depth.n_duplicate_md == 1
    assert result.depth.n_non_monotonic == 1
    codes = {i.code for i in result.issues}
    assert "DUPLICATE_MD" in codes
    assert "NON_MONOTONIC_MD" in codes
    assert all(i.severity == "WARNING" for i in result.issues if i.code in ("DUPLICATE_MD", "NON_MONOTONIC_MD"))


def test_compute_depth_diagnostics_directly():
    md = np.array([100.0, 100.1, 100.1, 100.05, 100.4])
    diag = compute_depth_diagnostics(md, declared_strt=100.0, declared_stop=100.4, declared_step=0.1)
    assert diag.n_duplicate_md == 1
    assert diag.n_non_monotonic == 1
    assert diag.data_start == pytest.approx(100.0)
    assert diag.data_stop == pytest.approx(100.4)
    assert diag.strt_matches_declared is True
    assert diag.stop_matches_declared is True


# ---------------------------------------------------------------------------
# Missing required curve vs. missing-but-optional curve
# ---------------------------------------------------------------------------
def test_missing_required_curve_fails_contract():
    contract = _fixture_contract(
        "standard_mnemonics.las",
        "Synthetic Standard 1",
        3,
        [
            _depth_entry(0, "DEPT", "Depth"),
            _entry(1, "GR", "API", "Gamma Ray", None, None, "GR", "GR_api", "GR_api", "API"),
            _entry(2, "RHOB", "G/CC", "Bulk Density", None, None, "RHOB", "RHOB_g_per_cm3", "RHOB_kg_m3", "kg/m3", conversion_function="gcc_to_kgm3"),
            _entry(3, "", "us/ft", "3 DTCO", 3, "DTCO", "DTCO", "DTCO_us_per_ft", "VP_m_s", "m/s", required=True, conversion_function="us_per_ft_to_m_per_s"),
        ],
    )
    with pytest.raises(LasContractError) as excinfo:
        load_las_file(str(FIXTURES / "standard_mnemonics.las"), contract)
    assert any(i.code == "CURVE_NOT_FOUND" for i in excinfo.value.issues)


def test_missing_optional_curve_does_not_fail_contract():
    contract = _fixture_contract(
        "standard_mnemonics.las",
        "Synthetic Standard 1",
        3,
        [
            _depth_entry(0, "DEPT", "Depth"),
            _entry(1, "GR", "API", "Gamma Ray", None, None, "GR", "GR_api", "GR_api", "API"),
            _entry(2, "RHOB", "G/CC", "Bulk Density", None, None, "RHOB", "RHOB_g_per_cm3", "RHOB_kg_m3", "kg/m3", conversion_function="gcc_to_kgm3"),
            _entry(3, "", "us/ft", "3 DTCO", 3, "DTCO", "DTCO", "DTCO_us_per_ft", "VP_m_s", "m/s", required=False, conversion_function="us_per_ft_to_m_per_s"),
        ],
    )
    result = load_las_file(str(FIXTURES / "standard_mnemonics.las"), contract)
    assert result.contract_status == "PASSED"
    assert "VP_m_s" not in result.canonical_data
    resolutions_by_name = {r.canonical_name: r for r in result.resolutions}
    assert resolutions_by_name["VP_m_s"].status == "MISSING_OPTIONAL"


# ---------------------------------------------------------------------------
# resolve_curve_contract used standalone (no data read)
# ---------------------------------------------------------------------------
def test_resolve_curve_contract_standalone_pass():
    header = parse_las_header(str(FIXTURES / "empty_mnemonic_with_description.las"))
    resolutions, issues, status = resolve_curve_contract(header, _empty_mnemonic_contract())
    assert status == "PASSED"
    assert all(r.status == "RESOLVED" for r in resolutions)
    assert issues == ()


def test_resolve_curve_contract_never_resolves_empty_mnemonic_on_ordinal_alone():
    header = parse_las_header(str(FIXTURES / "empty_mnemonic_with_description.las"))
    contract = _fixture_contract(
        "empty_mnemonic_with_description.las",
        "Synthetic Empty Mnemonic 1",
        4,
        [
            _depth_entry(0, "DEPT", "0 Depth", 0, "Depth"),
            _entry(1, "", "ft", "1 DTCO", 1, "DTCO", "DTCO", "DTCO_us_per_ft", "VP_m_s", "m/s"),  # wrong unit only
            _entry(2, "", "API", "2 GR", 2, "GR", "GR", "GR_api", "GR_api", "API"),
            _entry(3, "", "g/cc", "3 RHOB", 3, "RHOB", "RHOB", "RHOB_g_per_cm3", "RHOB_kg_m3", "kg/m3", conversion_function="gcc_to_kgm3"),
        ],
    )
    resolutions, issues, status = resolve_curve_contract(header, contract)
    assert status == "FAILED"
    by_name = {r.canonical_name: r for r in resolutions}
    assert by_name["VP_m_s"].status == "FAILED"


# ---------------------------------------------------------------------------
# Well-identifier and curve-count cross-checks
# ---------------------------------------------------------------------------
def test_well_identifier_mismatch_is_reported():
    header = parse_las_header(str(FIXTURES / "standard_mnemonics.las"))
    contract = _fixture_contract("standard_mnemonics.las", "Some Other Well", 3, _standard_contract().curves)
    _, issues, status = resolve_curve_contract(header, contract)
    assert status == "FAILED"
    assert any(i.code == "WELL_IDENTIFIER_MISMATCH" for i in issues)


def test_curve_count_mismatch_is_reported():
    header = parse_las_header(str(FIXTURES / "standard_mnemonics.las"))
    contract = FileContract(
        source_filename="standard_mnemonics.las",
        expected_sha256=_sha256(FIXTURES / "standard_mnemonics.las"),
        expected_well_identifier="Synthetic Standard 1",
        expected_las_version="2.0",
        expected_wrap="NO",
        expected_null_value=-999.25,
        expected_curve_count=99,
        expected_data_layout="unwrapped_whitespace_delimited",
        curves=_standard_contract().curves,
    )
    _, issues, status = resolve_curve_contract(header, contract)
    assert status == "FAILED"
    assert any(i.code == "CURVE_COUNT_MISMATCH" for i in issues)


# ---------------------------------------------------------------------------
# Increment 2.1: file-identity checks (filename, SHA-256, WELL, VERS, WRAP, NULL)
# ---------------------------------------------------------------------------
def test_filename_mismatch_is_blocking():
    # A contract authored for a DIFFERENT file than the one actually
    # loaded must fail, not silently apply.
    header = parse_las_header(str(FIXTURES / "standard_mnemonics.las"))
    contract = FileContract(
        source_filename="some_other_file.las",  # deliberately wrong
        expected_sha256=_sha256(FIXTURES / "standard_mnemonics.las"),
        expected_well_identifier="Synthetic Standard 1",
        expected_las_version="2.0",
        expected_wrap="NO",
        expected_null_value=-999.25,
        expected_curve_count=3,
        expected_data_layout="unwrapped_whitespace_delimited",
        curves=_standard_contract().curves,
    )
    _, issues, status = resolve_curve_contract(header, contract)
    assert status == "FAILED"
    assert any(i.code == "FILENAME_MISMATCH" for i in issues)


def test_sha256_mismatch_is_blocking():
    header = parse_las_header(str(FIXTURES / "standard_mnemonics.las"))
    contract = FileContract(
        source_filename="standard_mnemonics.las",
        expected_sha256="0" * 64,  # deliberately wrong
        expected_well_identifier="Synthetic Standard 1",
        expected_las_version="2.0",
        expected_wrap="NO",
        expected_null_value=-999.25,
        expected_curve_count=3,
        expected_data_layout="unwrapped_whitespace_delimited",
        curves=_standard_contract().curves,
    )
    _, issues, status = resolve_curve_contract(header, contract)
    assert status == "FAILED"
    assert any(i.code == "SHA256_MISMATCH" for i in issues)


def test_missing_well_is_blocking():
    header = parse_las_header(str(FIXTURES / "missing_well.las"))
    contract = _fixture_contract(
        "missing_well.las", "Anything", 2,
        [_depth_entry(0, "DEPT", "0 Depth", 0, "Depth"),
         _entry(1, "", "API", "1 GR", 1, "GR", "GR", "GR_api", "GR_api", "API")],
    )
    _, issues, status = resolve_curve_contract(header, contract)
    assert status == "FAILED"
    assert any(i.code == "WELL_MISSING" for i in issues)
    with pytest.raises(LasContractError) as excinfo:
        load_las_file(str(FIXTURES / "missing_well.las"), contract)
    assert any(i.code == "WELL_MISSING" for i in excinfo.value.issues)


def test_missing_null_is_blocking():
    header = parse_las_header(str(FIXTURES / "missing_null.las"))
    contract = _fixture_contract(
        "missing_null.las", "Synthetic Missing Null 1", 2,
        [_depth_entry(0, "DEPT", "0 Depth", 0, "Depth"),
         _entry(1, "", "API", "1 GR", 1, "GR", "GR", "GR_api", "GR_api", "API")],
    )
    _, issues, status = resolve_curve_contract(header, contract)
    assert status == "FAILED"
    assert any(i.code == "NULL_MISSING" for i in issues)


def test_vers_mismatch_is_blocking():
    header = parse_las_header(str(FIXTURES / "vers_mismatch.las"))
    contract = _fixture_contract(
        "vers_mismatch.las", "Synthetic Vers Mismatch 1", 2,
        [_depth_entry(0, "DEPT", "0 Depth", 0, "Depth"),
         _entry(1, "", "API", "1 GR", 1, "GR", "GR", "GR_api", "GR_api", "API")],
    )
    _, issues, status = resolve_curve_contract(header, contract)
    assert status == "FAILED"
    # The file declares VERS=3.0, which this parser does not implement at
    # all (regardless of what the contract expects), so this is reported
    # as unsupported.
    assert any(i.code == "VERS_UNSUPPORTED" for i in issues)


def test_wrap_yes_is_rejected_as_unsupported():
    header = parse_las_header(str(FIXTURES / "wrapped_unsupported.las"))
    contract = _fixture_contract(
        "wrapped_unsupported.las", "Synthetic Wrapped 1", 2,
        [_depth_entry(0, "DEPT", "0 Depth", 0, "Depth"),
         _entry(1, "", "API", "1 GR", 1, "GR", "GR", "GR_api", "GR_api", "API")],
    )
    _, issues, status = resolve_curve_contract(header, contract)
    assert status == "FAILED"
    assert any(i.code == "WRAP_UNSUPPORTED" for i in issues)


def test_null_mismatch_is_blocking_error_not_warning():
    header = parse_las_header(str(FIXTURES / "standard_mnemonics.las"))
    contract = FileContract(
        source_filename="standard_mnemonics.las",
        expected_sha256=_sha256(FIXTURES / "standard_mnemonics.las"),
        expected_well_identifier="Synthetic Standard 1",
        expected_las_version="2.0",
        expected_wrap="NO",
        expected_null_value=-888.0,  # file actually declares -999.25
        expected_curve_count=3,
        expected_data_layout="unwrapped_whitespace_delimited",
        curves=_standard_contract().curves,
    )
    _, issues, status = resolve_curve_contract(header, contract)
    assert status == "FAILED"
    null_issues = [i for i in issues if i.code == "NULL_MISMATCH"]
    assert len(null_issues) == 1
    assert null_issues[0].severity == "ERROR"


def test_unsupported_las_version_rejected_at_contract_load(tmp_path):
    bad = tmp_path / "bad_version.yml"
    bad.write_text(
        f"""
files:
  x.las:
    expected_sha256: "abc"
    expected_well_identifier: "W"
    expected_las_version: "3.0"
    expected_wrap: "NO"
    expected_null_value: -999.25
    expected_curve_count: 1
    expected_data_layout: "unwrapped_whitespace_delimited"
    curves:
      - ordinal: 0
        semantic_role: measured_depth
        source_curve_name: "DEPT"
        raw_mnemonic: "DEPT"
        raw_unit: "M"
        raw_description: "0 Depth"
        raw_canonical_name: "DEPT_m"
        canonical_name: "MD_m"
        canonical_unit: "m"
        required: true
        conversion_function: "identity"
"""
    )
    with pytest.raises(LasContractDefinitionError):
        load_file_contract_config(str(bad))


def test_unsupported_wrap_rejected_at_contract_load(tmp_path):
    bad = tmp_path / "bad_wrap.yml"
    bad.write_text(
        f"""
files:
  x.las:
    expected_sha256: "abc"
    expected_well_identifier: "W"
    expected_las_version: "2.0"
    expected_wrap: "YES"
    expected_null_value: -999.25
    expected_curve_count: 1
    expected_data_layout: "unwrapped_whitespace_delimited"
    curves:
      - ordinal: 0
        semantic_role: measured_depth
        source_curve_name: "DEPT"
        raw_mnemonic: "DEPT"
        raw_unit: "M"
        raw_description: "0 Depth"
        raw_canonical_name: "DEPT_m"
        canonical_name: "MD_m"
        canonical_unit: "m"
        required: true
        conversion_function: "identity"
"""
    )
    with pytest.raises(LasContractDefinitionError):
        load_file_contract_config(str(bad))


# ---------------------------------------------------------------------------
# Increment 2.1: contract-authoring validation
# ---------------------------------------------------------------------------
def test_duplicate_ordinal_rejected_at_contract_load(tmp_path):
    bad = tmp_path / "dup_ordinal.yml"
    bad.write_text(
        f"""
files:
  x.las:
{_MINIMAL_FILE_HEADER_FIELDS}
    expected_curve_count: 2
    curves:
      - ordinal: 0
        semantic_role: measured_depth
        source_curve_name: "DEPT"
        raw_mnemonic: "DEPT"
        raw_unit: "M"
        raw_description: "0 Depth"
        raw_canonical_name: "DEPT_m"
        canonical_name: "MD_m"
        canonical_unit: "m"
        required: true
        conversion_function: "identity"
      - ordinal: 0
        source_curve_name: "GR"
        raw_mnemonic: "GR"
        raw_unit: "API"
        raw_description: "1 GR"
        raw_canonical_name: "GR_api"
        canonical_name: "GR_api"
        canonical_unit: "API"
        required: true
        conversion_function: "identity"
"""
    )
    with pytest.raises(LasContractDefinitionError, match="duplicate ordinal"):
        load_file_contract_config(str(bad))


def test_negative_ordinal_rejected_at_contract_load(tmp_path):
    bad = tmp_path / "neg_ordinal.yml"
    bad.write_text(
        f"""
files:
  x.las:
{_MINIMAL_FILE_HEADER_FIELDS}
    expected_curve_count: 1
    curves:
      - ordinal: -1
        semantic_role: measured_depth
        source_curve_name: "DEPT"
        raw_mnemonic: "DEPT"
        raw_unit: "M"
        raw_description: "0 Depth"
        raw_canonical_name: "DEPT_m"
        canonical_name: "MD_m"
        canonical_unit: "m"
        required: true
        conversion_function: "identity"
"""
    )
    with pytest.raises(LasContractDefinitionError):
        load_file_contract_config(str(bad))


def test_conversion_unit_incompatibility_rejected_at_contract_load(tmp_path):
    bad = tmp_path / "bad_conv_unit.yml"
    bad.write_text(
        f"""
files:
  x.las:
{_MINIMAL_FILE_HEADER_FIELDS}
    expected_curve_count: 2
    curves:
      - ordinal: 0
        semantic_role: measured_depth
        source_curve_name: "DEPT"
        raw_mnemonic: "DEPT"
        raw_unit: "M"
        raw_description: "0 Depth"
        raw_canonical_name: "DEPT_m"
        canonical_name: "MD_m"
        canonical_unit: "m"
        required: true
        conversion_function: "identity"
      - ordinal: 1
        source_curve_name: "GR"
        raw_mnemonic: "GR"
        raw_unit: "API"
        raw_description: "1 GR"
        raw_canonical_name: "GR_api"
        canonical_name: "GR_kgm3"
        canonical_unit: "kg/m3"
        required: true
        conversion_function: "gcc_to_kgm3"
"""
    )
    # gcc_to_kgm3 requires raw_unit "g/cc"; this contract declares "API" -
    # a contract must not be able to apply a density conversion to a
    # gamma-ray curve.
    with pytest.raises(LasContractDefinitionError):
        load_file_contract_config(str(bad))


def test_zero_measured_depth_roles_rejected_at_contract_load(tmp_path):
    bad = tmp_path / "no_depth_role.yml"
    bad.write_text(
        f"""
files:
  x.las:
{_MINIMAL_FILE_HEADER_FIELDS}
    expected_curve_count: 1
    curves:
      - ordinal: 0
        source_curve_name: "GR"
        raw_mnemonic: "GR"
        raw_unit: "API"
        raw_description: "0 GR"
        raw_canonical_name: "GR_api"
        canonical_name: "GR_api"
        canonical_unit: "API"
        required: true
        conversion_function: "identity"
"""
    )
    with pytest.raises(LasContractDefinitionError, match="measured_depth"):
        load_file_contract_config(str(bad))


def test_multiple_measured_depth_roles_rejected_at_contract_load(tmp_path):
    bad = tmp_path / "two_depth_roles.yml"
    bad.write_text(
        f"""
files:
  x.las:
{_MINIMAL_FILE_HEADER_FIELDS}
    expected_curve_count: 2
    curves:
      - ordinal: 0
        semantic_role: measured_depth
        source_curve_name: "DEPT"
        raw_mnemonic: "DEPT"
        raw_unit: "M"
        raw_description: "0 Depth"
        raw_canonical_name: "DEPT_m"
        canonical_name: "MD_m"
        canonical_unit: "m"
        required: true
        conversion_function: "identity"
      - ordinal: 1
        semantic_role: measured_depth
        source_curve_name: "DEPT2"
        raw_mnemonic: "DEPT2"
        raw_unit: "M"
        raw_description: "1 Depth2"
        raw_canonical_name: "DEPT2_m"
        canonical_name: "MD2_m"
        canonical_unit: "m"
        required: true
        conversion_function: "identity"
"""
    )
    with pytest.raises(LasContractDefinitionError):
        load_file_contract_config(str(bad))


def test_measured_depth_wrong_canonical_name_rejected(tmp_path):
    bad = tmp_path / "wrong_depth_name.yml"
    bad.write_text(
        f"""
files:
  x.las:
{_MINIMAL_FILE_HEADER_FIELDS}
    expected_curve_count: 1
    curves:
      - ordinal: 0
        semantic_role: measured_depth
        source_curve_name: "DEPT"
        raw_mnemonic: "DEPT"
        raw_unit: "M"
        raw_description: "0 Depth"
        raw_canonical_name: "DEPT_m"
        canonical_name: "DEPTH_INDEX"
        canonical_unit: "m"
        required: true
        conversion_function: "identity"
"""
    )
    with pytest.raises(LasContractDefinitionError, match="MD_m"):
        load_file_contract_config(str(bad))


def test_non_boolean_required_rejected_at_contract_load(tmp_path):
    bad = tmp_path / "bad_required.yml"
    bad.write_text(
        f"""
files:
  x.las:
{_MINIMAL_FILE_HEADER_FIELDS}
    expected_curve_count: 1
    curves:
      - ordinal: 0
        semantic_role: measured_depth
        source_curve_name: "DEPT"
        raw_mnemonic: "DEPT"
        raw_unit: "M"
        raw_description: "0 Depth"
        raw_canonical_name: "DEPT_m"
        canonical_name: "MD_m"
        canonical_unit: "m"
        required: "yes"
        conversion_function: "identity"
"""
    )
    with pytest.raises(LasContractDefinitionError):
        load_file_contract_config(str(bad))


def test_nonnumeric_null_rejected_at_contract_load(tmp_path):
    bad = tmp_path / "bad_null.yml"
    bad.write_text(
        """
files:
  x.las:
    expected_sha256: "abc"
    expected_well_identifier: "W"
    expected_las_version: "2.0"
    expected_wrap: "NO"
    expected_null_value: "not-a-number"
    expected_curve_count: 1
    expected_data_layout: "unwrapped_whitespace_delimited"
    curves:
      - ordinal: 0
        semantic_role: measured_depth
        source_curve_name: "DEPT"
        raw_mnemonic: "DEPT"
        raw_unit: "M"
        raw_description: "0 Depth"
        raw_canonical_name: "DEPT_m"
        canonical_name: "MD_m"
        canonical_unit: "m"
        required: true
        conversion_function: "identity"
"""
    )
    with pytest.raises(LasContractDefinitionError):
        load_file_contract_config(str(bad))


# ---------------------------------------------------------------------------
# Contract-config (YAML) loading
# ---------------------------------------------------------------------------
def test_load_file_contract_config_valid(tmp_path):
    good_yaml = tmp_path / "good.yml"
    good_yaml.write_text(
        f"""
files:
  some_file.las:
{_MINIMAL_FILE_HEADER_FIELDS}
    expected_curve_count: 2
    curves:
      - ordinal: 0
        semantic_role: measured_depth
        source_curve_name: "DEPT"
        raw_mnemonic: "DEPT"
        raw_unit: "M"
        raw_description: "0 Depth"
        raw_canonical_name: "DEPT_m"
        canonical_name: "MD_m"
        canonical_unit: "m"
        required: true
        conversion_function: "identity"
      - ordinal: 1
        source_curve_name: "DTCO"
        raw_mnemonic: ""
        raw_unit: "us/ft"
        raw_description: "1 DTCO"
        description_ordinal: 1
        description_name: "DTCO"
        raw_canonical_name: "DTCO_us_per_ft"
        canonical_name: "VP_m_s"
        canonical_unit: "m/s"
        required: true
        conversion_function: "us_per_ft_to_m_per_s"
"""
    )
    contracts = load_file_contract_config(str(good_yaml))
    assert "some_file.las" in contracts
    assert len(contracts["some_file.las"].curves) == 2


def test_load_file_contract_config_missing_file():
    with pytest.raises(LasContractDefinitionError):
        load_file_contract_config("/nonexistent/path/contracts.yml")


def test_load_file_contract_config_malformed_yaml(tmp_path):
    bad = tmp_path / "bad.yml"
    bad.write_text("files: [this is not, a mapping :::")
    with pytest.raises(LasContractDefinitionError):
        load_file_contract_config(str(bad))


def test_load_file_contract_config_unknown_conversion_function(tmp_path):
    bad = tmp_path / "bad_conv.yml"
    bad.write_text(
        f"""
files:
  x.las:
{_MINIMAL_FILE_HEADER_FIELDS}
    expected_curve_count: 1
    curves:
      - ordinal: 0
        semantic_role: measured_depth
        source_curve_name: "DEPT"
        raw_mnemonic: "DEPT"
        raw_unit: "M"
        raw_description: "0 Depth"
        raw_canonical_name: "DEPT_m"
        canonical_name: "MD_m"
        canonical_unit: "m"
        required: true
        conversion_function: "not_a_real_function"
"""
    )
    with pytest.raises(LasContractDefinitionError, match="unknown"):
        load_file_contract_config(str(bad))


# ---------------------------------------------------------------------------
# Real project contract file (config/las_curve_contracts.yml) at least loads
# and validates internally, even though the four raw LAS files themselves
# are not required by this portable test suite.
# ---------------------------------------------------------------------------
def test_real_project_contract_config_loads_and_validates():
    config_path = Path(__file__).parent.parent / "config" / "las_curve_contracts.yml"
    if not config_path.exists():
        pytest.skip("config/las_curve_contracts.yml not present in this checkout")
    contracts = load_file_contract_config(str(config_path))
    expected_files = {
        "Poseidon_2_logs.las",
        "Boreas_1_logs.las",
        "Poseidon_North_1_logs.las",
        "Proteus_1ST2_logs.las",
    }
    assert expected_files.issubset(set(contracts))
    for filename in expected_files:
        contract = contracts[filename]
        assert contract.expected_curve_count == 9
        canonical_names = [c.canonical_name for c in contract.curves]
        assert len(canonical_names) == len(set(canonical_names)), f"{filename}: duplicate canonical name in contract"
        depth_curves = [c for c in contract.curves if c.semantic_role == "measured_depth"]
        assert len(depth_curves) == 1
        assert depth_curves[0].canonical_name == "MD_m"
        assert depth_curves[0].required


def test_real_project_contract_gr_ecgr_grd_are_distinct_canonical_names():
    config_path = Path(__file__).parent.parent / "config" / "las_curve_contracts.yml"
    if not config_path.exists():
        pytest.skip("config/las_curve_contracts.yml not present in this checkout")
    contracts = load_file_contract_config(str(config_path))
    gr_names = set()
    for contract in contracts.values():
        for c in contract.curves:
            if c.source_curve_name in ("GR", "ECGR", "GRD"):
                gr_names.add(c.canonical_name)
    # GR_api, ECGR_api, GRD_api must all appear as distinct canonical
    # names - never merged into one geological interpretation.
    assert {"GR_api", "ECGR_api", "GRD_api"}.issubset(gr_names)


# ---------------------------------------------------------------------------
# load_wells: batch loading, typed IngestionFailure isolation
# ---------------------------------------------------------------------------
def test_load_wells_isolates_parsing_failure_from_a_successful_well():
    good_path = str(FIXTURES / "standard_mnemonics.las")
    bad_path = str(FIXTURES / "malformed_numeric_row.las")

    good_contract = _standard_contract()
    bad_contract = _fixture_contract(
        "malformed_numeric_row.las", "Synthetic Malformed Row 1", 2,
        [_depth_entry(0, "DEPT", "0 Depth", 0, "Depth"),
         _entry(1, "", "API", "1 GR", 1, "GR", "GR", "GR_api", "GR_api", "API")],
    )
    contracts = {
        "standard_mnemonics.las": good_contract,
        "malformed_numeric_row.las": bad_contract,
    }
    file_paths = {"GOOD": good_path, "BAD": bad_path}

    results, errors = load_wells(file_paths, contracts)
    assert "GOOD" in results
    assert "BAD" in errors
    failure = errors["BAD"]
    assert isinstance(failure, IngestionFailure)
    assert failure.error_type == "parsing_failure"
    assert "malformed" in failure.message.lower()


def test_load_wells_isolates_contract_failure_from_a_successful_well():
    good_path = str(FIXTURES / "standard_mnemonics.las")
    wrong_well_contract = _fixture_contract(
        "standard_mnemonics.las", "Wrong Well Name", 3, _standard_contract().curves
    )
    contracts = {"standard_mnemonics.las": wrong_well_contract}
    file_paths = {"WRONG_WELL": good_path}
    results, errors = load_wells(file_paths, contracts)
    assert "WRONG_WELL" in errors
    assert "WRONG_WELL" not in results
    failure = errors["WRONG_WELL"]
    assert failure.error_type == "contract_failure"
    assert any(i.code == "WELL_IDENTIFIER_MISMATCH" for i in failure.exception.issues)


def test_load_wells_isolates_file_not_found():
    # A contract entry exists for this basename, but no file with that
    # name exists on disk - this must surface as "file_not_found", not as
    # a LasContractDefinitionError (which is reserved for a basename with
    # NO matching contract at all - a configuration problem, not a
    # missing-file problem).
    missing_path = str(FIXTURES / "does_not_exist.las")
    contracts = {"does_not_exist.las": _standard_contract()}
    file_paths = {"MISSING": missing_path}
    results, errors = load_wells(file_paths, contracts)
    assert "MISSING" in errors
    assert errors["MISSING"].error_type == "file_not_found"


def test_load_wells_unknown_file_raises_definition_error():
    with pytest.raises(LasContractDefinitionError):
        load_wells({"X": str(FIXTURES / "standard_mnemonics.las")}, contracts={})


def test_load_wells_programming_error_still_propagates(monkeypatch):
    # A non-ingestion exception (e.g. a bug) must NOT be silently absorbed
    # into the typed-failure dict - only the four documented ingestion-
    # failure categories are isolated per well.
    import p2mem.io.las as las_module

    def _boom(path, contract):
        raise KeyError("programming error, not an ingestion failure")

    monkeypatch.setattr(las_module, "load_las_file", _boom)
    with pytest.raises(KeyError):
        load_wells({"X": str(FIXTURES / "standard_mnemonics.las")}, {"standard_mnemonics.las": _standard_contract()})


# ---------------------------------------------------------------------------
# Deterministic output ordering
# ---------------------------------------------------------------------------
def test_deterministic_ordering_across_repeated_loads():
    contract = _empty_mnemonic_contract()
    path = str(FIXTURES / "empty_mnemonic_with_description.las")
    result_a = load_las_file(path, contract)
    result_b = load_las_file(path, contract)

    assert list(result_a.canonical_data.keys()) == list(result_b.canonical_data.keys())
    assert [r.canonical_name for r in result_a.resolutions] == [r.canonical_name for r in result_b.resolutions]
    assert [c.ordinal for c in result_a.header.curve_headers] == [c.ordinal for c in result_b.header.curve_headers]
    # Canonical data key order must follow contract declaration order.
    assert list(result_a.canonical_data.keys()) == ["MD_m", "VP_m_s", "GR_api", "RHOB_kg_m3"]
    np.testing.assert_array_equal(result_a.raw_data, result_b.raw_data)


def test_curve_stats_report_raw_and_canonical_units_explicitly():
    result = load_las_file(str(FIXTURES / "standard_mnemonics.las"), _standard_contract())
    stats_by_name = {s.canonical_name: s for s in result.curve_stats}

    gr_stats = stats_by_name["GR_api"]
    assert gr_stats.n_samples == 5
    assert gr_stats.valid_count == 4
    assert gr_stats.null_count == 1
    assert gr_stats.valid_fraction == pytest.approx(0.8)
    assert gr_stats.raw_unit == "API"
    assert gr_stats.canonical_unit == "API"
    assert gr_stats.raw_min == pytest.approx(50.1)
    assert gr_stats.raw_max == pytest.approx(54.5)
    assert gr_stats.canonical_min == pytest.approx(50.1)
    assert gr_stats.canonical_max == pytest.approx(54.5)
    assert "NULL" in gr_stats.statistics_basis

    rhob_stats = stats_by_name["RHOB_kg_m3"]
    assert rhob_stats.raw_unit.upper() == "G/CC"
    assert rhob_stats.canonical_unit == "kg/m3"
    # Raw stats are in g/cc; canonical stats are the same values * 1000.
    assert rhob_stats.canonical_min == pytest.approx(rhob_stats.raw_min * 1000, rel=1e-9)
    assert rhob_stats.canonical_max == pytest.approx(rhob_stats.raw_max * 1000, rel=1e-9)


def test_curve_coverage_row_shows_dtco_raw_and_vp_canonical_units():
    from p2mem.io.inventory import build_curve_coverage_rows

    result = load_las_file(
        str(FIXTURES / "empty_mnemonic_with_description.las"), _empty_mnemonic_contract()
    )
    rows = build_curve_coverage_rows({"TEST": result})
    dtco_row = next(r for r in rows if r["source_curve_name"] == "DTCO")
    assert dtco_row["canonical_name"] == "VP_m_s"
    assert dtco_row["raw_unit"] == "us/ft"
    assert dtco_row["canonical_unit"] == "m/s"
    assert dtco_row["raw_min"] is not None and dtco_row["raw_max"] is not None
    assert dtco_row["canonical_min"] is not None and dtco_row["canonical_max"] is not None
    # us/ft -> m/s is an inverse relationship: the largest raw slowness
    # corresponds to the smallest canonical velocity, and vice versa.
    assert dtco_row["canonical_min"] < dtco_row["canonical_max"]


### Step 8 — Install the package in editable mode

**Technical objective:** re-install `p2mem` (now at version 0.2.1, with the corrected `p2mem.io` subpackage and `p2mem.models` module) in editable mode, so the corrected modules are importable without any `sys.path` manipulation.

**Expected result:** installs cleanly; `import p2mem; p2mem.__version__` reports `"0.2.1"`.

In [ ]:
%cd /content/drive/MyDrive/Poseidon_1D_MEM
!pip install -e . -q
import importlib
import p2mem
importlib.reload(p2mem)
print("p2mem version:", p2mem.__version__)
print("Assurance tier:", p2mem.ASSURANCE_TIER)

### Step 9 — Run the complete unit-test suite (Increment 1.1 + Increment 2 together)

**Technical objective:** run every test in `tests/` — `test_units.py` (Increment 1.1, unchanged) and `test_las.py` (new in Increment 2) — in one `pytest` invocation, confirming that adding LAS ingestion did not remove or weaken any prior test.

**Validation logic:** `test_las.py` uses ONLY the small synthetic fixtures written in Step 7b; it does not require and does not read the four real project LAS files. The real four-well integration run is a separate, later step (Steps 11-16) - this separation is intentional, per the increment specification, so the portable unit-test suite runs identically for anyone who clones this repository without the private raw data.

**Expected result:** this notebook does not assert a fixed pass count in advance. Read the actual number from the `pytest` output below and report exactly that number - not an assumed one.

In [ ]:
!pytest -v

### Step 10 — Inspect the actual LAS headers of all four wells, and verify their SHA-256

**Technical objective:** parse ONLY the header sections (Version, Well, Curve, Parameter) of all four real files - no numeric data yet - and print exactly what each file declares, so the contract written in Step 7 can be visually cross-checked against the real files it claims to describe, one more time, in this notebook's own output (not just asserted in prose). This step also independently verifies each file's SHA-256 against the four hashes already recorded in `INCREMENT_02_v2.1_MANIFEST.md` — the same values every contract entry's `expected_sha256` field declares, so a mismatch here would mean the uploaded file is not the one this contract was authored against.

**LAS theory:** `parse_las_header` (see `p2mem/io/las.py`) walks the file's `~`-prefixed section markers, applies the LAS 2.0 `MNEM.UNIT VALUE :DESCRIPTION` definition-line convention (including the empty-unit-field edge case needed for lines like the WELL name), and stops at the `~A` marker without reading any data row.

**Inputs:** the four files under `data/raw/logs/`.
**Outputs:** printed header summaries only; no files written.

**Failure behavior:** `LasFileNotFoundError` if a file is missing (already checked in Step 4); `LasParsingError` if a header line cannot be parsed or no data-section marker is found.

**Expected result:** four wells, 9 curves each, each SHA-256 reporting MATCH against the manifest's recorded value.

In [ ]:
from p2mem.io.las import parse_las_header

file_paths = {
    "Poseidon_2": os.path.join(LOGS_DIR, "Poseidon_2_logs.las"),
    "Boreas_1": os.path.join(LOGS_DIR, "Boreas_1_logs.las"),
    "Poseidon_North_1": os.path.join(LOGS_DIR, "Poseidon_North_1_logs.las"),
    "Proteus_1ST2": os.path.join(LOGS_DIR, "Proteus_1ST2_logs.las"),
}

# Independently recorded in INCREMENT_02_v2.1_MANIFEST.md (and declared as
# expected_sha256 in each contract entry) - compared here, not assumed.
EXPECTED_SHA256 = {
    "Poseidon_2": "4684864b2d4b37be1f132cc264d4672055865aa283b4fc7d7ef702f5d9a86828",
    "Boreas_1": "b2fdf60cd9dfeb7843ffa211232d909e7e76b6122cb2fd84f212edc2371dd61f",
    "Poseidon_North_1": "ca3fe7b6547a72559127fab8c9540643600f4207efd067b84a08ad219354d78f",
    "Proteus_1ST2": "ffb100de5fdd5e0639eee71a59a2602aad19a386116d48b8110912e58939e92f",
}

headers = {}
for key, path in file_paths.items():
    h = parse_las_header(path)
    headers[key] = h
    sha_match = h.sha256 == EXPECTED_SHA256[key]
    print(f"=== {key} ({h.source_filename}) ===")
    print(f"  SHA-256:        {h.sha256}  -> {'MATCH' if sha_match else 'MISMATCH'}")
    print(f"  WELL:           {h.well_name!r}")
    print(f"  VERS / WRAP:    {h.las_version} / {h.wrap}")
    print(f"  NULL:           {h.declared_null}")
    print(f"  STRT/STOP/STEP: {h.declared_strt} / {h.declared_stop} / {h.declared_step}")
    print(f"  Curves ({len(h.curve_headers)}):")
    for c in h.curve_headers:
        mnem = c.raw_mnemonic or "(empty)"
        print(f"    [{c.ordinal}] mnemonic={mnem!r:10s} unit={c.raw_unit!r:8s} description={c.raw_description!r}")
    print()
    if not sha_match:
        raise RuntimeError(f"{key}: SHA-256 mismatch - this file is not the one the contract was authored against.")

### Step 11 — Validate each per-file contract against the actual headers (no data read yet)

**Technical objective:** run `resolve_curve_contract` for each well - comparing the contract written in Step 7 against the headers parsed in Step 10 - WITHOUT reading any numeric data, so a contract-authoring mistake is caught before the (much larger) data section is ever parsed.

**Validation logic:** every curve is resolved only when its ordinal, unit, and description-embedded ordinal/name all agree with the contract (see `p2mem/io/las.py:resolve_curve_contract` docstring) - ordinal position alone is never sufficient. File-level checks (curve count, WELL identifier, declared NULL) are also cross-checked here.

**Failure behavior:** this cell does not raise on a FAILED contract - it prints the failure clearly and continues to the next well, so all four wells' contract status is visible in one run. `load_las_file` (Step 12) is what actually raises, per well, if a contract fails.

**Expected result:** all four wells report `PASSED` with every curve `RESOLVED` (this was independently verified in the manifest before this notebook was written; a `FAILED` result here would mean the real files changed since the contract was authored, and must be investigated before proceeding - not ignored).

In [ ]:
from p2mem.io.las import load_file_contract_config, resolve_curve_contract

contracts = load_file_contract_config("config/las_curve_contracts.yml")
print(f"Loaded contracts for: {sorted(contracts)}\n")

contract_filenames = {
    "Poseidon_2": "Poseidon_2_logs.las",
    "Boreas_1": "Boreas_1_logs.las",
    "Poseidon_North_1": "Poseidon_North_1_logs.las",
    "Proteus_1ST2": "Proteus_1ST2_logs.las",
}

for key, filename in contract_filenames.items():
    resolutions, issues, status = resolve_curve_contract(headers[key], contracts[filename])
    resolved = sum(1 for r in resolutions if r.status == "RESOLVED")
    print(f"{key}: contract status = {status}  ({resolved}/{len(resolutions)} curves RESOLVED)")
    for issue in issues:
        print(f"    [{issue.severity}] {issue.code}: {issue.message}")
print()

### Step 12 — Load the four files (full ingestion: header + contract + data)

**Technical objective:** run the complete `load_las_file` pipeline for all four wells via `load_wells`: parse header, resolve contract (including the Increment 2.1 file-identity checks), parse the `~Ascii` data section, cross-check the data-column count, locate the measured-depth curve by its declared `semantic_role` and compute depth diagnostics from it, and - for every RESOLVED curve - substitute the declared NULL sentinel with NaN and apply the contract's exact conversion function (if any).

**Inputs:** the four real LAS files plus `contracts`.
**Outputs:** one `LasFileResult` per successfully-loaded well (raw data matrix + canonical, unit-suffixed, unit-converted curves + depth diagnostics + curve statistics + issues), or a typed `IngestionFailure` per well that failed (`error_type` one of "file_not_found", "parsing_failure", "contract_failure", "conversion_failure").

**Validation logic / failure behavior:** `load_wells` isolates all four of those expected per-well ingestion-failure categories from each other - one well's failure does not stop the remaining wells from loading. Any OTHER exception (a programming error, not an expected data-quality problem) is NOT caught here and would stop the whole run - that is intentional.

**Expected result:** all four wells load successfully (`results` has 4 entries, `errors` is empty), consistent with Step 11's all-PASSED contract validation.

In [ ]:
from p2mem.io.las import load_wells

results, errors = load_wells(file_paths, contracts)

print(f"Loaded successfully: {sorted(results)}")
print(f"Failed to load:      {sorted(errors)}")
for key, failure in errors.items():
    print(f"  {key}: [{failure.error_type}] {failure.message}")

### Step 13 — Generate the deterministic inventory outputs

**Technical objective:** write the five required files under `outputs/02_las_inventory/`: `las_file_inventory.csv`, `las_curve_catalog.csv`, `las_curve_coverage.csv`, `las_ingestion_issues.csv`, and `las_ingestion_manifest.json`. Every builder function lives in `p2mem/io/inventory.py` and emits metadata/statistics only - no raw LAS sample data is ever written to these files.

**Validation logic:** row order follows the deterministic order of `results`/`errors` as given (well iteration order, then contract-declaration order within a well) - re-running this cell against unchanged inputs reproduces identical output.

**Expected result:** five files appear under `outputs/02_las_inventory/`; `las_ingestion_issues.csv` may legitimately contain only a header row if zero issues were found (a factual, auditable statement in itself, not a broken file).

In [ ]:
import csv
import json

from p2mem.io.inventory import (
    build_curve_catalog_rows,
    build_curve_coverage_rows,
    build_file_inventory_rows,
    build_ingestion_manifest,
    build_issues_rows,
)

OUT_DIR = os.path.join(PROJECT_ROOT, "outputs", "02_las_inventory")
os.makedirs(OUT_DIR, exist_ok=True)

ISSUES_FIELDNAMES = ["well_key", "source_filename", "severity", "code", "message", "context"]

def write_csv(path, rows, fieldnames=None):
    fieldnames = fieldnames or (list(rows[0].keys()) if rows else [])
    with open(path, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)

write_csv(os.path.join(OUT_DIR, "las_file_inventory.csv"), build_file_inventory_rows(results, errors))
write_csv(os.path.join(OUT_DIR, "las_curve_catalog.csv"), build_curve_catalog_rows(results))
write_csv(os.path.join(OUT_DIR, "las_curve_coverage.csv"), build_curve_coverage_rows(results))
write_csv(os.path.join(OUT_DIR, "las_ingestion_issues.csv"), build_issues_rows(results, errors), fieldnames=ISSUES_FIELDNAMES)

manifest = build_ingestion_manifest(results, errors)
with open(os.path.join(OUT_DIR, "las_ingestion_manifest.json"), "w") as f:
    json.dump(manifest, f, indent=2)

print("Inventory outputs written to:", OUT_DIR)
for f in sorted(os.listdir(OUT_DIR)):
    print(" -", f, os.path.getsize(os.path.join(OUT_DIR, f)), "bytes")

### Step 14 — Display concise inventory tables

**Technical objective:** show a compact, readable summary of the inventory outputs directly in this notebook, using `pandas` purely as a display convenience (NOT a `p2mem` package dependency - see Step 5's note). These are factual summaries only; no interpretation, no lithology or fluid inference, no flagging of curves as "good"/"bad" - min/max/valid-fraction are shown exactly as `p2mem.io.las.compute_curve_stats` computed them.

**Expected result:** one row per well (file inventory) and a curve-coverage table with one row per resolved curve across all four wells.

In [ ]:
import pandas as pd

file_inventory_df = pd.DataFrame(build_file_inventory_rows(results, errors))
display_cols = ["well_key", "well_name", "n_samples", "data_start_m", "data_stop_m",
                 "n_duplicate_md", "n_non_monotonic_md", "contract_status", "n_warnings", "n_errors"]
print("=== File inventory ===")
display(file_inventory_df[display_cols])

coverage_df = pd.DataFrame(build_curve_coverage_rows(results))
print("\n=== Curve coverage (raw AND canonical statistics, each with its own explicit unit) ===")
display(coverage_df[[
    "well_key", "source_curve_name", "canonical_name",
    "raw_unit", "raw_min", "raw_max",
    "canonical_unit", "canonical_min", "canonical_max",
    "n_samples", "valid_count", "valid_fraction",
]])

### Step 15 — Integration regression checks

**Technical objective:** independently VERIFY three known numbers from prior review, rather than hard-coding or assuming them: Poseidon 2 should contain 31,897 log rows spanning approximately 490.0-5350.9507 m MD, and the four approved wells together should contain 127,287 log rows. These numbers are compared AFTER the fact against what this run's loader actually computed - they are never returned BY the loader itself, and a mismatch is reported clearly rather than silently reconciled.

**This step also reconfirms, as a plain fact and without acting on it, the Boreas 1 ECGR anomaly** (median/range far outside the other three wells' GR-family curves) already flagged in the Rev 1 design review: this notebook does not rescale it, does not reinterpret its lithology meaning, does not compute an NCT from it, and does not correct it automatically.

**Expected result:** all three regression checks report MATCH, and the ECGR statistics are printed as a factual carry-forward, not a new finding requiring action in this increment.

In [ ]:
EXPECTED_POSEIDON_2_ROWS = 31897
EXPECTED_POSEIDON_2_MD_START = 490.0
EXPECTED_POSEIDON_2_MD_STOP = 5350.9507
EXPECTED_TOTAL_ROWS_FOUR_WELLS = 127287

print("=== Integration regression checks (independently verified, not hard-coded into the loader) ===")

p2 = results["Poseidon_2"]
match_rows = p2.depth.n_samples == EXPECTED_POSEIDON_2_ROWS
print(f"Poseidon 2 row count: {p2.depth.n_samples} (expected {EXPECTED_POSEIDON_2_ROWS}) -> {'MATCH' if match_rows else 'MISMATCH'}")

md_start_close = abs(p2.depth.data_start - EXPECTED_POSEIDON_2_MD_START) < 1e-3
md_stop_close = abs(p2.depth.data_stop - EXPECTED_POSEIDON_2_MD_STOP) < 1e-3
print(f"Poseidon 2 MD range: {p2.depth.data_start}-{p2.depth.data_stop} "
      f"(expected {EXPECTED_POSEIDON_2_MD_START}-{EXPECTED_POSEIDON_2_MD_STOP}) -> "
      f"{'MATCH' if (md_start_close and md_stop_close) else 'MISMATCH'}")

total_rows = sum(r.depth.n_samples for r in results.values())
match_total = total_rows == EXPECTED_TOTAL_ROWS_FOUR_WELLS
print(f"Total rows, four wells: {total_rows} (expected {EXPECTED_TOTAL_ROWS_FOUR_WELLS}) -> "
      f"{'MATCH' if match_total else 'MISMATCH'}")

if not (match_rows and md_start_close and md_stop_close and match_total):
    print("\n*** At least one regression check MISMATCHED - investigate before proceeding to Increment 3. ***")

print("\n=== Boreas 1 ECGR anomaly - reconfirmed as fact, not acted on ===")
ecgr_stats = next(s for s in results["Boreas_1"].curve_stats if s.canonical_name == "ECGR_api")
print(f"ECGR_api: valid_fraction={ecgr_stats.valid_fraction:.4f}, raw_min={ecgr_stats.raw_min}, raw_max={ecgr_stats.raw_max} ({ecgr_stats.raw_unit})")
for key in ("Poseidon_2", "Poseidon_North_1", "Proteus_1ST2"):
    for s in results[key].curve_stats:
        if s.canonical_name in ("GR_api", "GRD_api"):
            print(f"{key} {s.canonical_name}: valid_fraction={s.valid_fraction:.4f}, raw_min={s.raw_min}, raw_max={s.raw_max} ({s.raw_unit})")
print("No rescaling, reinterpretation, NCT computation, or automatic correction applied to ECGR in this increment.")

### Step 16 — Completion gate and limitations

**This increment ends here.** The next step (LAS-derived curves feeding into deviation-survey depth harmonization) is Increment 3 and is explicitly OUT OF SCOPE for this notebook.

In [ ]:
print("=" * 78)
print("INCREMENT 2.1 COMPLETION GATE")
print("=" * 78)
print()
print(f"Wells loaded successfully:  {len(results)} / 4")
print(f"Wells failed to load:       {len(errors)} / 4")
print(f"Total log rows (4 wells):   {sum(r.depth.n_samples for r in results.values())}")
print(f"Total ERROR-severity issues across all wells: {sum(sum(1 for i in r.issues if i.severity == 'ERROR') for r in results.values())}")
print(f"Total WARNING-severity issues across all wells: {sum(sum(1 for i in r.issues if i.severity == 'WARNING') for r in results.values())}")
print()
print("Known, carried-forward limitations (not resolved by this increment):")
print(" - Boreas 1 ECGR scale anomaly: reported as fact, not rescaled/reinterpreted/corrected.")
print(" - Proteus 1ST2 NPHI column-order difference: handled by its own per-file contract, not unified across wells.")
print(" - Poseidon North 1 formation tops: still not obtained (open Rev 1 gating item, unrelated to LAS ingestion).")
print(" - RHOB coverage in Poseidon 2 ends near 5296.85 m MD even though DEPT extends further - carried forward, not re-derived here.")
print(" - No petrophysical interpretation, lithology classification, or unit conversion beyond exact SI factors was performed.")
print()
print("Increment 2.1 is complete. Stopping here per instruction.")
print("Do NOT proceed to deviation-survey processing or Increment 3 without separate review and approval.")